# FLASHCROWD ML pipeline: PC3 to PC1

Every line of PC3's pipeline in one notebook: pre-processing, features and labels, the leak gate, systematic training,
refinement, post-processing, the forecast job, the shared engine and the handoff to PC1 and PC2.

**How to use it in VS Code**
1. Open this file, pick a Python 3.10+ kernel (top right).
2. Run **Part A** once: it writes every source file into `flashcrowd-model/` next to this notebook.
3. Put PC1's key at `flashcrowd-model/secrets/fc_pc3_rsa_key.p8` (never commit it).
4. Run **Part B** top to bottom: install, connect, check, train, refine, backtest, export, first live forecast.

Set `MODE = "local"` in B.1 to run everything on synthetic data first (no Snowflake needed, about 5 minutes).

# Part A: the code (each cell writes one file)

In [ ]:
import os
for d in ['flashcrowd-model/src/flashcrowd_model/sql', 'flashcrowd-model/src/flashcrowd_engine', 'flashcrowd-model/configs', 'flashcrowd-model/secrets']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

## 1. Configuration and contracts

Shared constants (wikis, regions, surge rule, feature lists, paths), the systematic-training configs, the roadmap's Snowflake DDL (verbatim) and the `.env` template with PC3's Snowflake settings.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/__init__.py
"""FLASHCROWD model pipeline (PC3)."""

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/config.py
"""Shared constants for the PC3 model pipeline. Anything marked CONTRACT must match CONTRACTS.md."""
from __future__ import annotations

import math
import os
from pathlib import Path

SEED = 7
CONTRACTS_VERSION = "1.0"

# ---- CONTRACT §3.1: wikis, regions, region weights
WIKIS = ("enwiki", "dewiki", "frwiki", "itwiki", "nlwiki", "eswiki", "ptwiki", "jawiki", "kowiki", "idwiki")
REGIONS = ("LDN", "NJ", "TYO")
WIKI_REGION = {"dewiki": "LDN", "frwiki": "LDN", "itwiki": "LDN", "nlwiki": "LDN",
               "eswiki": "NJ", "ptwiki": "NJ", "jawiki": "TYO", "kowiki": "TYO", "idwiki": "TYO"}
EN_DEFAULT_WEIGHTS = {"LDN": 0.35, "NJ": 0.45, "TYO": 0.20}
REGION_TZ = {"LDN": "Europe/London", "NJ": "America/New_York", "TYO": "Asia/Tokyo"}
WIKI_MAIN_REGION = {**WIKI_REGION, "enwiki": "NJ"}


def hour_curve(local_hour: float) -> float:
    """Relative reading activity by local hour (stated assumption): trough ~04:00, peak ~20:00."""
    return 0.85 + 0.45 * math.cos(2 * math.pi * (local_hour - 20) / 24)


# ---- CONTRACT §7: surge definition (shared with PC1's scorecard via the SURGE_HOURS view)
SURGE_MULT = 5.0
SURGE_FLOOR = {"enwiki": 3000}
SURGE_FLOOR_DEFAULT = 800
HORIZONS = (1, 2, 3)

# ---- decision grid (CONTRACT §7)
CAND_EDITS_3H = 3          # >= 3 edits in (t-3h, t]
CAND_TOP_N = 500           # top 500 of its wiki in the latest published hour
CAND_MAIN_PAGE_H = 6       # on the Main Page in the last 6 h

# ---- Gemini (CONTRACT §3.4 enum)
CAUSES = ("breaking_news", "death", "sports", "scheduled_event", "entertainment", "vandalism", "edit_war",
          "bot_maintenance", "other")
GEMINI_BURST_EDITS_60M = 8
GEMINI_BURST_HUMANS_60M = 3

# ---- features (all strictly as of t). Labels start with y_ / size_ and never appear here.
EDIT_FEATURES = ["edits_15m", "edits_30m", "edits_60m", "edits_180m", "humans_60m", "humans_180m",
                 "bot_share_60m", "revert_share_60m", "bytes_added_60m", "mins_since_last_edit", "accel_30m"]
PAGE_FEATURES = ["is_new_page", "page_age_h"]
VIEW_FEATURES = ["last_pub_views", "log_last_pub_views", "ratio_same_hour_med7", "slope_3h",
                 "views_same_hour_yesterday", "med7_next_hour", "hours_since_pub", "surge_last_pub"]
MAIN_PAGE_FEATURES = ["is_tfa_today", "mins_since_itn", "in_itn_6h"]
TIME_FEATURES = ["hour_utc", "dow", "local_hour", "wiki_id"]
XLANG_FEATURES = ["n_other_wikis_3h"]
CAUSE_FEATURES = [f"cause_{c}" for c in CAUSES] + ["cause_confidence"]
BASE_FEATURES = EDIT_FEATURES + PAGE_FEATURES + VIEW_FEATURES + MAIN_PAGE_FEATURES + TIME_FEATURES + XLANG_FEATURES
CATEGORICAL = ["wiki_id"]

# ---- training
VAL_DAYS = 1
TEST_DAYS = 2
EMBARGO_H = max(HORIZONS)
REGION_Q90_P_MIN = 0.3      # CONTRACT §7 region forecast: q90 adds articles with p_surge >= 0.3
FORECAST_WRITE_TOP = 300    # titles per issue time written to FORECASTS (all horizons each)

# ---- paths
ROOT = Path(os.environ.get("FLASHCROWD_HOME", Path(__file__).resolve().parents[2]))
ARTIFACTS = ROOT / "artifacts"
MODEL_PATH = ARTIFACTS / "model.pkl"
CHART_DIR = ARTIFACTS / "charts"
LIVE_DIR = ARTIFACTS / "live"
CACHE_DIR = ARTIFACTS / "cache"
SQL_DIR = Path(__file__).resolve().parent / "sql"
LOCAL_DB = ROOT / "data" / "flashcrowd_local.duckdb"


def surge_floor(wiki: str) -> int:
    return SURGE_FLOOR.get(wiki, SURGE_FLOOR_DEFAULT)

In [ ]:
%%writefile flashcrowd-model/configs/train.json
{
  "_comment": "Systematic training config. Every run copies this file into artifacts/runs/<run_id>/config.json.",
  "window": {"start": null, "end": null},
  "splits": {"test_days": 2, "cv_folds": 3, "min_train_days": 4, "embargo_hours": 3},
  "search": {
    "n_trials": 10,
    "seed": 7,
    "early_stopping_rounds": 100,
    "max_estimators": 1000,
    "space": {
      "is_unbalance": [true, false],
      "learning_rate": [0.02, 0.05],
      "num_leaves": [7, 15, 31],
      "min_child_samples": [10, 20, 40],
      "colsample_bytree": [0.6, 0.8],
      "reg_lambda": [0.0, 1.0, 5.0]
    },
    "roadmap_default": {"is_unbalance": true, "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 40,
                        "colsample_bytree": 0.8, "reg_lambda": 1.0}
  },
  "size_model": {"learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 40},
  "calibration": {"min_oof_positives": 10},
  "causes": "auto",
  "validation": {"min_days": 7, "min_test_onsets": 15, "max_missing_view_hours": 6, "min_publish_lag_min": 30,
                 "max_publish_lag_min": 360},
  "gate": {"min_gain_over_persistence": 0.05, "max_drop_vs_champion": 0.01, "metric": "pr_auc_h2"}
}

In [ ]:
%%writefile flashcrowd-model/configs/train_short.json
{
  "_comment": "For a short backfill (6-8 days of edits). Same pipeline, smaller splits: 1 test day, 2 CV days.",
  "window": {"start": null, "end": null},
  "splits": {"test_days": 1, "cv_folds": 2, "min_train_days": 3, "embargo_hours": 3},
  "search": {
    "n_trials": 8, "seed": 7, "early_stopping_rounds": 100, "max_estimators": 1000,
    "space": {"is_unbalance": [true, false], "learning_rate": [0.02, 0.05], "num_leaves": [7, 15, 31],
              "min_child_samples": [10, 20, 40], "colsample_bytree": [0.6, 0.8], "reg_lambda": [0.0, 1.0, 5.0]},
    "roadmap_default": {"is_unbalance": true, "learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 40,
                        "colsample_bytree": 0.8, "reg_lambda": 1.0}
  },
  "size_model": {"learning_rate": 0.05, "num_leaves": 31, "min_child_samples": 40},
  "calibration": {"min_oof_positives": 8},
  "causes": "auto",
  "validation": {"min_days": 6, "min_test_onsets": 8, "max_missing_view_hours": 6, "min_publish_lag_min": 30,
                 "max_publish_lag_min": 360},
  "gate": {"min_gain_over_persistence": 0.05, "max_drop_vs_champion": 0.01, "metric": "pr_auc_h2"}
}

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/sql/ddl_snowflake.sql
CREATE DATABASE IF NOT EXISTS FLASHCROWD;
CREATE SCHEMA IF NOT EXISTS FLASHCROWD.CORE;
USE SCHEMA FLASHCROWD.CORE;
CREATE WAREHOUSE IF NOT EXISTS FC_WH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

CREATE TABLE IF NOT EXISTS EDITS (
  ts           TIMESTAMP_NTZ,   -- event time (UTC)
  wiki         STRING,
  title        STRING,          -- underscores
  namespace    INT,
  type         STRING,          -- edit | new
  user_name    STRING,
  is_bot       BOOLEAN,
  is_revert    BOOLEAN,         -- comment matches revert patterns
  delta_bytes  INT,
  comment      STRING
);

CREATE TABLE IF NOT EXISTS PAGEVIEWS_HOURLY (
  hour_utc      TIMESTAMP_NTZ,  -- start of the hour the views happened in
  wiki          STRING,
  title         STRING,
  views         INT,            -- desktop + mobile web summed
  published_at  TIMESTAMP_NTZ   -- Last-Modified of the dump file (≈ hour + 2h). THE no-look-ahead key.
);

CREATE TABLE IF NOT EXISTS MAIN_PAGE_EVENTS (
  ts     TIMESTAMP_NTZ,   -- when it became visible (TFA: 00:00 UTC of its day; ITN: revision timestamp)
  wiki   STRING,          -- enwiki only is fine
  title  STRING,
  slot   STRING           -- tfa | itn
);

CREATE TABLE IF NOT EXISTS COUNTRY_SHARE (
  day     DATE,
  wiki    STRING,
  title   STRING,
  region  STRING,         -- LDN | NJ | TYO (mapped from country)
  share   FLOAT
);

CREATE TABLE IF NOT EXISTS CAUSE_LABELS (   -- written by PC3's Gemini labeller
  ts        TIMESTAMP_NTZ,  -- decision time the label was produced for
  wiki      STRING,
  title     STRING,
  cause     STRING,         -- see enum in 3.4
  confidence FLOAT,
  reason    STRING
);

CREATE TABLE IF NOT EXISTS FORECASTS (        -- written by PC3 forecast job
  issued_at  TIMESTAMP_NTZ, wiki STRING, title STRING, region STRING,
  horizon_h  INT, p_surge FLOAT, size_q50 FLOAT, size_q90 FLOAT,
  cause STRING, top_reasons VARIANT
);

CREATE TABLE IF NOT EXISTS REGION_FORECASTS ( -- written by PC3, consumed by decision engine + API
  issued_at TIMESTAMP_NTZ, region STRING, target_hour TIMESTAMP_NTZ,
  q50_views FLOAT, q90_views FLOAT
);

CREATE TABLE IF NOT EXISTS INFRA_MEASUREMENTS (  -- written by PC1
  ts TIMESTAMP_NTZ, kind STRING,  -- boot | rtt | capacity
  region STRING, peer STRING, value FLOAT, unit STRING, note STRING
);

-- PC3 addition (optional, not part of CONTRACTS 1.0): Wikidata item per article for the cross-language feature.
CREATE TABLE IF NOT EXISTS WIKIDATA_ITEMS (
  wiki   STRING,
  title  STRING,
  qid    STRING
);

In [ ]:
%%writefile flashcrowd-model/.env.example
# Copy to .env (never committed). Values below match what PC1 issued for PC3.
FLASHCROWD_STORE=snowflake
SNOWFLAKE_ACCOUNT=ZOUHELY-XW50213
SNOWFLAKE_USER=FC_PC3
# Key-pair login: put PC1's private key file here. secrets/ is git-ignored.
SNOWFLAKE_PRIVATE_KEY_PATH=secrets/fc_pc3_rsa_key.p8
# Only if the key was generated with a passphrase:
# SNOWFLAKE_PRIVATE_KEY_PASSPHRASE=
SNOWFLAKE_ROLE=FC_ROLE
SNOWFLAKE_WAREHOUSE=FC_WH
SNOWFLAKE_DATABASE=FLASHCROWD
SNOWFLAKE_SCHEMA=CORE
# Gemini (optional until cause labels are switched on)
GEMINI_API_KEY=
GEMINI_MODEL=gemini-2.5-flash

## 2. Connection: PC1's `sf.py` and the data store

Key-pair login exactly as PC1's helper (with `__file__` fixed), and one store interface over Snowflake or a local DuckDB file.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/sf.py
"""Snowflake connection helper shared with PC1 (key-pair auth). Usage: with connect() as conn: ...

Same code as PC1's flashcrowd-server/pipeline/sf.py, with two fixes:
  * `_file_` -> `__file__` (the original raises NameError on import)
  * ROOT comes from flashcrowd_model.config so `.env` and `secrets/` resolve from this repo's root
  * an optional SNOWFLAKE_PRIVATE_KEY_PASSPHRASE is honoured (None if unset, as in PC1's version)
"""
import os
import pathlib

from cryptography.hazmat.primitives import serialization

from . import config as C

ROOT = pathlib.Path(C.ROOT)

# Load .env from the repo root (python-dotenv is optional: plain environment variables also work).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass


# Read the .p8 key (relative paths from the repo root) and return DER bytes, exactly as PC1 does.
def private_key_der() -> bytes:
    kp = pathlib.Path(os.environ["SNOWFLAKE_PRIVATE_KEY_PATH"]).expanduser()
    kp = kp if kp.is_absolute() else ROOT / kp
    if not kp.exists():
        raise FileNotFoundError(f"private key not found at {kp}; put PC1's .p8 file there (never commit it)")
    pw = os.environ.get("SNOWFLAKE_PRIVATE_KEY_PASSPHRASE")
    key = serialization.load_pem_private_key(kp.read_bytes(), password=pw.encode() if pw else None)
    return key.private_bytes(serialization.Encoding.DER, serialization.PrivateFormat.PKCS8,
                             serialization.NoEncryption())


# Same connection parameters as PC1's helper; keep-alive so long training pulls do not time out.
def connect():
    import snowflake.connector
    return snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"], user=os.environ["SNOWFLAKE_USER"], private_key=private_key_der(),
        role=os.environ["SNOWFLAKE_ROLE"], warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
        database=os.environ["SNOWFLAKE_DATABASE"], schema=os.environ["SNOWFLAKE_SCHEMA"],
        client_session_keep_alive=True, application="flashcrowd-model")

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/store.py
"""Data access: one interface over Snowflake (PC1's FLASHCROWD.CORE) and a local DuckDB file (offline / tests).

Choose with FLASHCROWD_STORE:
    snowflake                      -> SNOWFLAKE_* env vars (see .env.example)
    duckdb:/path/to/file.duckdb    -> local file with the same tables (created from the contract DDL)
All timestamps are naive UTC (Snowflake TIMESTAMP_NTZ holding UTC). Column names come back lower-case.
"""
from __future__ import annotations

import os
import re
from pathlib import Path

import pandas as pd

from . import config as C

# The contract tables (plus the optional WIKIDATA_ITEMS).
TABLES = ("EDITS", "PAGEVIEWS_HOURLY", "MAIN_PAGE_EVENTS", "COUNTRY_SHARE", "CAUSE_LABELS", "FORECASTS",
          "REGION_FORECASTS", "INFRA_MEASUREMENTS", "WIKIDATA_ITEMS")


# A timestamp literal both Snowflake and DuckDB accept.
def ts_literal(t) -> str:
    """Portable timestamp literal for both dialects."""
    return pd.Timestamp(t).strftime("'%Y-%m-%d %H:%M:%S'")


# Reuse the roadmap's DDL verbatim; for DuckDB only the type names are translated.
def contract_ddl(dialect: str) -> list[str]:
    """CREATE TABLE statements from sql/ddl_snowflake.sql, translated for DuckDB when needed."""
    text = (C.SQL_DIR / "ddl_snowflake.sql").read_text()
    text = "\n".join(l.split("--", 1)[0] for l in text.splitlines())
    stmts = [s.strip() for s in text.split(";") if s.strip().upper().startswith("CREATE TABLE")]
    if dialect == "duckdb":
        subs = [(r"\bTIMESTAMP_NTZ\b", "TIMESTAMP"), (r"\bSTRING\b", "VARCHAR"), (r"\bVARIANT\b", "VARCHAR"),
                (r"\bFLOAT\b", "DOUBLE")]
        for a, b in subs:
            stmts = [re.sub(a, b, s) for s in stmts]
    return stmts


# Minimal interface the pipeline needs; both backends implement it.
class Store:
    dialect = "?"

    def query(self, sql: str) -> pd.DataFrame:
        raise NotImplementedError

    def execute(self, sql: str) -> None:
        raise NotImplementedError

    def register_temp(self, df: pd.DataFrame, name: str) -> None:
        raise NotImplementedError

    def insert(self, df: pd.DataFrame, table: str, variant_cols=()) -> None:
        raise NotImplementedError

    def has_table(self, name: str) -> bool:
        try:
            self.query(f"SELECT * FROM {name} LIMIT 0")
            return True
        except Exception:
            return False

    def replace_rows(self, df: pd.DataFrame, table: str, where: str, variant_cols=()) -> int:
        """Idempotent write: delete rows matching ``where`` then insert ``df`` (cron re-runs never duplicate)."""
        self.execute(f"DELETE FROM {table} WHERE {where}")
        if len(df):
            self.insert(df, table, variant_cols)
        return len(df)


# Local file backend: tests, synthetic data, and a laptop fallback if Snowflake is down.
class DuckDBStore(Store):
    dialect = "duckdb"

    def __init__(self, path: str | Path = C.LOCAL_DB, create: bool = True):
        import duckdb
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        self.con = duckdb.connect(str(path))
        if create:
            for s in contract_ddl("duckdb"):
                self.con.execute(s)

    def query(self, sql):
        df = self.con.execute(sql).df()
        df.columns = [c.lower() for c in df.columns]
        return df

    def execute(self, sql):
        self.con.execute(sql)

    def register_temp(self, df, name):
        self.con.register(f"__{name}_df", df)
        self.con.execute(f"CREATE OR REPLACE TEMP TABLE {name} AS SELECT * FROM __{name}_df")
        self.con.unregister(f"__{name}_df")

    # VARIANT columns (top_reasons) go through a temp stage + PARSE_JSON, which write_pandas cannot do directly.
    def insert(self, df, table, variant_cols=()):
        cols = [c.lower() for c in self.query(f"SELECT * FROM {table} LIMIT 0").columns]
        df = df.copy()
        for c in variant_cols:
            df[c] = df[c].map(lambda v: v if isinstance(v, str) or v is None else __import__("json").dumps(v, ensure_ascii=False))
        for c in cols:
            if c not in df.columns:
                df[c] = None
        self.con.register("__ins_df", df[cols])
        self.con.execute(f"INSERT INTO {table} SELECT * FROM __ins_df")
        self.con.unregister("__ins_df")


# PC1's FLASHCROWD.CORE schema, credentials from .env (never committed).
class SnowflakeStore(Store):
    dialect = "snowflake"

    def __init__(self):
        # Key-pair login through the same helper PC1 uses (sf.py); a password still works as a fallback.
        if os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH") or not os.environ.get("SNOWFLAKE_PASSWORD"):
            from .sf import connect
            self.con = connect()
        else:
            import snowflake.connector
            e = os.environ
            self.con = snowflake.connector.connect(
                account=e["SNOWFLAKE_ACCOUNT"], user=e["SNOWFLAKE_USER"], password=e["SNOWFLAKE_PASSWORD"],
                role=e.get("SNOWFLAKE_ROLE"), warehouse=e.get("SNOWFLAKE_WAREHOUSE", "FC_WH"),
                database=e.get("SNOWFLAKE_DATABASE", "FLASHCROWD"), schema=e.get("SNOWFLAKE_SCHEMA", "CORE"))

    def query(self, sql):
        cur = self.con.cursor()
        try:
            cur.execute(sql)
            df = cur.fetch_pandas_all()
        finally:
            cur.close()
        df.columns = [c.lower() for c in df.columns]
        return df

    def execute(self, sql):
        self.con.cursor().execute(sql).close()

    def _write(self, df, name, **kw):
        from snowflake.connector.pandas_tools import write_pandas
        df = df.copy()
        df.columns = [c.upper() for c in df.columns]
        try:
            write_pandas(self.con, df, name, quote_identifiers=False, use_logical_type=True, **kw)
        except TypeError:                              # older connectors have no use_logical_type
            write_pandas(self.con, df, name, quote_identifiers=False, **kw)

    def register_temp(self, df, name):
        self._write(df, name.upper(), auto_create_table=True, table_type="temporary", overwrite=True)

    def insert(self, df, table, variant_cols=()):
        import json
        if not variant_cols:
            self._write(df, table.upper())
            return
        df = df.copy()
        for c in variant_cols:
            df[c] = df[c].map(lambda v: v if isinstance(v, str) else json.dumps(v, ensure_ascii=False))
        stage = f"{table.upper()}_STAGE_TMP"
        self.register_temp(df, stage)
        cols = list(df.columns)
        sel = ", ".join(f"PARSE_JSON({c})" if c in variant_cols else c for c in cols)
        self.execute(f"INSERT INTO {table} ({', '.join(cols)}) SELECT {sel} FROM {stage}")


# Read the .p8 private key PC1 generated and return it in the DER form the Snowflake connector expects.
# A relative path is resolved from the repo root, so `secrets/fc_pc3_rsa_key.p8` works from any folder.
def load_private_key(path: str, passphrase: str | None = None) -> bytes:
    from cryptography.hazmat.primitives import serialization
    p = Path(path).expanduser()
    if not p.is_absolute():
        p = C.ROOT / p
    if not p.exists():
        raise FileNotFoundError(f"private key not found at {p}; put PC1's .p8 file there (never commit it)")
    key = serialization.load_pem_private_key(p.read_bytes(), password=passphrase.encode() if passphrase else None)
    return key.private_bytes(encoding=serialization.Encoding.DER, format=serialization.PrivateFormat.PKCS8,
                             encryption_algorithm=serialization.NoEncryption())


# Pick the backend from FLASHCROWD_STORE.
def open_store(spec: str | None = None) -> Store:
    spec = spec or os.environ.get("FLASHCROWD_STORE", f"duckdb:{C.LOCAL_DB}")
    if spec == "snowflake":
        return SnowflakeStore()
    if spec.startswith("duckdb:"):
        return DuckDBStore(spec.split(":", 1)[1])
    raise ValueError(f"unknown store spec {spec!r}")

## 3. Pre-processing: extraction, cleaning, validation

Pull one window with portable SQL, clean edits and page views, build the publication-time map, and validate the data before any feature is built (FAIL stops training).

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/data.py
"""Extraction (Snowflake/DuckDB -> pandas) and preprocessing for one time window.

``load_window(store, start, end)`` returns a ``RawData`` holding everything the features and labels need for
decision times t in [start, end]. Pulls are kept small: edits for the window, page views only for candidate
titles (CONTRACT §7 decision grid), regional totals aggregated in the warehouse.

Preprocessing applied here, once, for training and live alike:
  * titles: spaces -> underscores; wikis outside CONTRACT §3.1 dropped; namespace 0 only
  * edits: deduplicated on (ts, wiki, title, user_name); is_revert recomputed from the comment when NULL
  * page views: summed per (hour, wiki, title) in case desktop/mobile rows were loaded separately
  * hour_published: when each hourly dump became available = max(published_at) per hour (the no-look-ahead key)
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd

from . import config as C
from .store import Store, ts_literal

# Same revert pattern as PC1's recorder, used when is_revert arrives NULL.
REVERT = re.compile(r"(?:revert|undid|undo|rv\b|rvv|restored revision)", re.I)
HOUR = pd.Timedelta(hours=1)


# Everything one window needs, already cleaned. Features and labels are pure functions of this object.
@dataclass
class RawData:
    start: pd.Timestamp
    end: pd.Timestamp
    edits: pd.DataFrame                 # ts, wiki, title, type, user_name, is_bot, is_revert, delta_bytes, comment
    pageviews: pd.DataFrame             # hour_utc, wiki, title, views, published_at   (candidate titles only)
    hour_published: pd.Series           # index hour_utc -> when that hour's dump was published
    top: pd.DataFrame                   # hour_utc, wiki, title (top-N per wiki per hour)
    main_page: pd.DataFrame             # ts, wiki, title, slot
    country_share: pd.DataFrame         # day, wiki, title, region, share
    causes: pd.DataFrame                # ts, wiki, title, cause, confidence, reason
    wikidata: pd.DataFrame              # wiki, title, qid
    wiki_hour_views: pd.DataFrame       # hour_utc, wiki, views (all tracked titles, for regional demand)
    en_shared: pd.DataFrame             # hour_utc, region, views_weighted, views_raw (enwiki titles with shares)
    meta: dict = field(default_factory=dict)

    # Cut the window to what existed at time t; the live job and the leak test both rely on this.
    def available(self, t) -> "RawData":
        """Everything as it existed at time t (used by the live job and the perturbation leak test)."""
        t = pd.Timestamp(t)
        hp = self.hour_published[self.hour_published <= t]
        return replace(
            self,
            edits=self.edits[self.edits.ts <= t],
            pageviews=self.pageviews[self.pageviews.published_at <= t],
            hour_published=hp,
            top=self.top[self.top.hour_utc.isin(hp.index)],
            main_page=self.main_page[self.main_page.ts <= t],
            causes=self.causes[self.causes.ts <= t],
            wiki_hour_views=self.wiki_hour_views[self.wiki_hour_views.hour_utc.isin(hp.index)],
            en_shared=self.en_shared[self.en_shared.hour_utc.isin(hp.index)],
        )


# ---------------------------------------------------------------- preprocessing
def clean_edits(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["ts"] = pd.to_datetime(df["ts"])
    df["title"] = df["title"].astype(str).str.replace(" ", "_", regex=False)
    df = df[df["wiki"].isin(C.WIKIS)]
    if "namespace" in df.columns:
        df = df[df["namespace"].fillna(0).astype(int) == 0]
    df["is_bot"] = df["is_bot"].fillna(False).astype(bool)
    comment = df["comment"].fillna("") if "comment" in df.columns else pd.Series("", index=df.index)
    rev = df["is_revert"] if "is_revert" in df.columns else pd.Series(np.nan, index=df.index)
    df["is_revert"] = rev.where(rev.notna(), comment.str.contains(REVERT)).astype(bool)
    df["delta_bytes"] = pd.to_numeric(df.get("delta_bytes", 0), errors="coerce").fillna(0).astype(int)
    df["type"] = df.get("type", "edit")
    df["comment"] = comment
    df = df.drop_duplicates(["ts", "wiki", "title", "user_name"])
    return df.sort_values("ts").reset_index(drop=True)[
        ["ts", "wiki", "title", "type", "user_name", "is_bot", "is_revert", "delta_bytes", "comment"]]


# Normalise page views and sum desktop + mobile rows per hour/title.
def clean_pageviews(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour_utc"] = pd.to_datetime(df["hour_utc"])
    df["published_at"] = pd.to_datetime(df["published_at"])
    df["title"] = df["title"].astype(str).str.replace(" ", "_", regex=False)
    df = df[df["wiki"].isin(C.WIKIS)]
    return (df.groupby(["hour_utc", "wiki", "title"], as_index=False)
              .agg(views=("views", "sum"), published_at=("published_at", "max")))


def _ts(df, *cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c])
    return df


# ---------------------------------------------------------------- candidates (CONTRACT §7 decision grid)
def edit_candidates(edits: pd.DataFrame, start, end) -> pd.DataFrame:
    """(wiki, title, t) with >= CAND_EDITS_3H edits in (t-3h, t], for whole hours t in [start, end]."""
    if edits.empty:
        return pd.DataFrame(columns=["wiki", "title", "t"])
    e = edits.assign(t=edits.ts.dt.ceil("h"))                         # first decision hour that includes the edit
    c = e.groupby(["wiki", "title", "t"]).size().rename("n").reset_index()
    parts = [c.assign(t=c.t + k * HOUR) for k in range(3)]            # edit at bucket b counts for t=b,b+1h,b+2h
    s = pd.concat(parts).groupby(["wiki", "title", "t"]).n.sum().reset_index()
    s = s[(s.n >= C.CAND_EDITS_3H) & (s.t >= start) & (s.t <= end)]
    return s[["wiki", "title", "t"]]


# ---------------------------------------------------------------- extraction
def load_window(store: Store, start, end, history_days: int = 8, label_hours: int = 4,
                with_comments: bool = True) -> RawData:
    """Pull everything needed for decision times t in [start, end] (labels need up to end + 3 h)."""
    start, end = pd.Timestamp(start).floor("h"), pd.Timestamp(end).floor("h")
    pv_from = start - pd.Timedelta(days=history_days)
    pv_to = end + pd.Timedelta(hours=label_hours)
    wikis = ", ".join(f"'{w}'" for w in C.WIKIS)
    cols = "ts, wiki, title, namespace, type, user_name, is_bot, is_revert, delta_bytes" + (", comment" if with_comments else "")
    # Edits from 26 h before the window (for 3 h windows, 24 h recency and page age).
    edits = store.query(f"""SELECT {cols} FROM EDITS
                            WHERE ts > {ts_literal(start - pd.Timedelta(hours=26))} AND ts <= {ts_literal(end)}
                              AND wiki IN ({wikis})""")
    if "comment" not in edits.columns:
        edits["comment"] = ""
    edits = clean_edits(edits)
    # page creations further back (page age feature)
    news = store.query(f"""SELECT ts, wiki, title FROM EDITS WHERE type = 'new' AND wiki IN ({wikis})
                           AND ts > {ts_literal(start - pd.Timedelta(days=30))} AND ts <= {ts_literal(end)}""")

    # Per hour and wiki: total views (regional demand) and when that hour's dump was published (availability).
    hp = store.query(f"""SELECT hour_utc, wiki, SUM(views) AS views, MAX(published_at) AS published_at
                         FROM PAGEVIEWS_HOURLY
                         WHERE hour_utc >= {ts_literal(pv_from)} AND hour_utc <= {ts_literal(pv_to)}
                         GROUP BY hour_utc, wiki""")
    hp = _ts(hp, "hour_utc", "published_at")
    hour_published = hp.groupby("hour_utc").published_at.max().sort_index()
    wiki_hour_views = hp[hp.wiki.isin(C.WIKIS)][["hour_utc", "wiki", "views"]]

    # Grid source 2: top 500 per wiki per hour, computed in the warehouse with QUALIFY.
    top = store.query(f"""SELECT hour_utc, wiki, title FROM PAGEVIEWS_HOURLY
                          WHERE hour_utc >= {ts_literal(start - pd.Timedelta(hours=6))} AND hour_utc <= {ts_literal(end)}
                            AND wiki IN ({wikis})
                          QUALIFY ROW_NUMBER() OVER (PARTITION BY wiki, hour_utc ORDER BY views DESC) <= {C.CAND_TOP_N}""")
    top = _ts(top, "hour_utc")
    top["title"] = top["title"].str.replace(" ", "_", regex=False)

    # Grid source 3 and Main Page features: TFA and ITN events from the last 2 days.
    mp = _ts(store.query(f"""SELECT ts, wiki, title, slot FROM MAIN_PAGE_EVENTS
                              WHERE ts > {ts_literal(start - pd.Timedelta(days=2))} AND ts <= {ts_literal(end)}"""), "ts")
    mp["title"] = mp["title"].astype(str).str.replace(" ", "_", regex=False)

    # Country shares (enwiki region split), Gemini cause labels and the optional Wikidata map.
    cs = store.query(f"""SELECT day, wiki, title, region, share FROM COUNTRY_SHARE
                         WHERE day >= CAST({ts_literal(pv_from)} AS DATE) AND day <= CAST({ts_literal(pv_to)} AS DATE)""")
    cs = _ts(cs, "day")
    causes = _ts(store.query(f"""SELECT ts, wiki, title, cause, confidence, reason FROM CAUSE_LABELS
                                  WHERE ts > {ts_literal(start - pd.Timedelta(hours=4))} AND ts <= {ts_literal(end)}"""), "ts")
    wikidata = (store.query("SELECT wiki, title, qid FROM WIKIDATA_ITEMS") if store.has_table("WIKIDATA_ITEMS")
                else pd.DataFrame(columns=["wiki", "title", "qid"]))

    # candidate titles -> page views for those titles only (pushed down as a temp table)
    cand = pd.concat([edit_candidates(edits, start, end)[["wiki", "title"]], top[["wiki", "title"]],
                      mp[["wiki", "title"]]]).drop_duplicates()
    store.register_temp(cand, "FC_CAND")
    pv = store.query(f"""SELECT p.hour_utc, p.wiki, p.title, p.views, p.published_at
                         FROM PAGEVIEWS_HOURLY p JOIN FC_CAND c ON p.wiki = c.wiki AND p.title = c.title
                         WHERE p.hour_utc >= {ts_literal(pv_from)} AND p.hour_utc <= {ts_literal(pv_to)}""")
    pv = clean_pageviews(pv) if len(pv) else pd.DataFrame(columns=["hour_utc", "wiki", "title", "views", "published_at"])

    # enwiki views split by COUNTRY_SHARE, and the raw total of those titles (to avoid double counting).
    en = store.query(f"""SELECT p.hour_utc, s.region, SUM(p.views * s.share) AS views_weighted
                         FROM PAGEVIEWS_HOURLY p JOIN COUNTRY_SHARE s
                           ON p.wiki = s.wiki AND p.title = s.title AND s.day = CAST(p.hour_utc AS DATE)
                         WHERE p.wiki = 'enwiki' AND p.hour_utc >= {ts_literal(pv_from)} AND p.hour_utc <= {ts_literal(pv_to)}
                         GROUP BY p.hour_utc, s.region""")
    en_raw = store.query(f"""SELECT p.hour_utc, SUM(p.views) AS views_raw
                             FROM PAGEVIEWS_HOURLY p JOIN (SELECT DISTINCT day, wiki, title FROM COUNTRY_SHARE) s
                               ON p.wiki = s.wiki AND p.title = s.title AND s.day = CAST(p.hour_utc AS DATE)
                             WHERE p.wiki = 'enwiki' AND p.hour_utc >= {ts_literal(pv_from)} AND p.hour_utc <= {ts_literal(pv_to)}
                             GROUP BY p.hour_utc""")
    en = _ts(en, "hour_utc").merge(_ts(en_raw, "hour_utc"), on="hour_utc", how="left")

    news = _ts(news, "ts")
    news["title"] = news["title"].astype(str).str.replace(" ", "_", regex=False)
    return RawData(start=start, end=end, edits=edits, pageviews=pv, hour_published=hour_published, top=top,
                   main_page=mp, country_share=cs, causes=causes, wikidata=wikidata,
                   wiki_hour_views=wiki_hour_views, en_shared=en,
                   meta={"page_created": news.groupby(["wiki", "title"]).ts.min()})

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/validate.py
"""Data validation: run before any feature is built. Each check returns PASS / WARN / FAIL.

A FAIL stops training (bad data makes a confident wrong model). A WARN is written to the run report and shown
in the console so a human can decide (e.g. "only 11 onsets in the test days").
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import config as C
from .data import HOUR, RawData


# Build one check result in a uniform shape so reports and the console can print them the same way.
def _r(name: str, status: str, detail: str, **extra) -> dict:
    return {"check": name, "status": status, "detail": detail, **extra}


def validate(raw: RawData, cfg: dict) -> dict:
    v = cfg["validation"]
    out = []

    # Enough calendar days of decision times for train + CV folds + test.
    days = (raw.end.normalize() - raw.start.normalize()).days + 1
    need = cfg["splits"]["test_days"] + cfg["splits"]["cv_folds"] + cfg["splits"]["min_train_days"]
    out.append(_r("days_of_data", "PASS" if days >= max(need, v["min_days"]) else "FAIL",
                  f"{days} days of decision times (need >= {max(need, v['min_days'])})", days=days))

    # Every tracked wiki should have edits; a silent wiki usually means the recorder filter is wrong.
    per_wiki = raw.edits.groupby("wiki").size().reindex(C.WIKIS, fill_value=0)
    missing = per_wiki[per_wiki == 0].index.tolist()
    out.append(_r("edits_per_wiki", "WARN" if missing else "PASS",
                  f"no edits for {missing}" if missing else "all 10 wikis have edits",
                  counts={k: int(x) for k, x in per_wiki.items()}))

    # Gaps in the edit stream (recorder outage) show up as hours with zero edits across all wikis.
    hrs = raw.edits.ts.dt.floor("h").value_counts()
    rng = pd.date_range(raw.start.floor("h"), raw.end.floor("h"), freq="h")
    empty = [str(h) for h in rng if hrs.get(h, 0) == 0]
    out.append(_r("edit_stream_gaps", "WARN" if empty else "PASS",
                  f"{len(empty)} hours with no edits at all" if empty else "no gaps", hours=empty[:20]))

    # Page-view dumps: every hour in the window should have been loaded.
    hp = raw.hour_published
    full = pd.date_range(raw.start - pd.Timedelta(days=7), raw.end, freq="h")
    miss = [str(h) for h in full if h not in hp.index]
    out.append(_r("pageview_hours", "FAIL" if len(miss) > v["max_missing_view_hours"] else ("WARN" if miss else "PASS"),
                  f"{len(miss)} hourly dumps missing", hours=miss[:20]))

    # published_at must look like a real Last-Modified time (about 2 h after the hour). If it equals the hour
    # itself, the loader wrote the wrong field and every feature would silently look ahead.
    lag = ((hp.to_numpy() - hp.index.to_numpy()) / np.timedelta64(1, "m")) if len(hp) else np.array([])
    med = float(np.median(lag)) if len(lag) else float("nan")
    ok = v["min_publish_lag_min"] <= med <= v["max_publish_lag_min"] if len(lag) else False
    out.append(_r("published_at_lag", "PASS" if ok else "FAIL",
                  f"median publish lag {med:.0f} min (expect ~120)", median_min=med,
                  min_min=float(lag.min()) if len(lag) else None))

    # Bot flag sanity: real Wikipedia has a large bot share; 0% means is_bot was not populated.
    share = float(raw.edits.is_bot.mean()) if len(raw.edits) else 0.0
    out.append(_r("bot_flag", "WARN" if share == 0 else "PASS", f"bot share {share:.1%}", share=share))

    # Main Page events let us predict scheduled surges; none at all means the loaders did not run.
    n_tfa = int((raw.main_page.slot == "tfa").sum())
    n_itn = int((raw.main_page.slot == "itn").sum())
    out.append(_r("main_page_events", "WARN" if n_tfa == 0 or n_itn == 0 else "PASS",
                  f"{n_tfa} TFA and {n_itn} ITN events"))

    # Duplicate edits inflate edit counts (e.g. the 5-minute loader re-reading the same bytes).
    dup = int(raw.edits.duplicated(["ts", "wiki", "title", "user_name"]).sum())
    out.append(_r("duplicate_edits", "WARN" if dup else "PASS", f"{dup} duplicate edits (already dropped)"))

    status = "FAIL" if any(c["status"] == "FAIL" for c in out) else ("WARN" if any(c["status"] == "WARN" for c in out) else "PASS")
    return {"status": status, "checks": out}


def onset_check(n_test_onsets: int, cfg: dict) -> dict:
    # The roadmap's risk table: fewer than 15 onsets in the test days makes every metric noise.
    m = cfg["validation"]["min_test_onsets"]
    return _r("test_onsets", "PASS" if n_test_onsets >= m else "WARN",
              f"{n_test_onsets} onsets in the test days (want >= {m}; else lower SURGE_MULT to 4 or add a day)")

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/warehouse.py
"""First contact with PC1's Snowflake: readiness check and one-time setup.

    flashcrowd check     what is loaded, is it enough to train, can this role write what PC3 writes
    flashcrowd setup     create the SURGE_HOURS view PC1's scorecard reads (and WIKIDATA_ITEMS if missing)

Write permission is tested with INSERT inside BEGIN ... ROLLBACK, so the check never leaves a row behind.
"""
from __future__ import annotations

import pandas as pd

from . import config as C
from .labels import surge_hours_sql
from .store import Store, ts_literal

READ_TABLES = ("EDITS", "PAGEVIEWS_HOURLY", "MAIN_PAGE_EVENTS", "COUNTRY_SHARE", "INFRA_MEASUREMENTS")
WRITE_TABLES = {
    "FORECASTS": "(issued_at, wiki, title, region, horizon_h, p_surge) VALUES ({t}, 'enwiki', 'FC_PROBE', 'LDN', 1, 0)",
    "REGION_FORECASTS": "(issued_at, region, target_hour, q50_views, q90_views) VALUES ({t}, 'LDN', {t}, 0, 0)",
    "CAUSE_LABELS": "(ts, wiki, title, cause, confidence, reason) VALUES ({t}, 'enwiki', 'FC_PROBE', 'other', 0, 'probe')",
}


# One result line: status (ok / warn / fail), what was checked, and the detail a person needs to act on it.
def _line(status: str, what: str, detail: str = "") -> dict:
    return {"status": status, "check": what, "detail": detail}


def readiness(store: Store, cfg: dict) -> list[dict]:
    out = []
    # Every table PC3 reads must exist; row counts show whether PC1's loaders have run.
    for t in READ_TABLES:
        if not store.has_table(t):
            out.append(_line("fail", t, "table missing: ask PC1 to run sql/ddl.sql"))
            continue
        n = int(store.query(f"SELECT COUNT(*) AS n FROM {t}").n.iloc[0])
        optional = t in ("COUNTRY_SHARE", "INFRA_MEASUREMENTS")
        out.append(_line("ok" if n else ("warn" if optional else "fail"), t, f"{n:,} rows" + (
            "" if n else (" (optional: defaults are used)" if optional else " (empty: PC1's loader has not run)"))))

    # Time coverage: decisions need 3 h of edits and 7 days of page views behind them.
    e = store.query("SELECT MIN(ts) AS a, MAX(ts) AS b, COUNT(DISTINCT wiki) AS w FROM EDITS").iloc[0]
    p = store.query("SELECT MIN(hour_utc) AS a, MAX(hour_utc) AS b, COUNT(DISTINCT hour_utc) AS n FROM PAGEVIEWS_HOURLY").iloc[0]
    if pd.notna(e.a) and pd.notna(p.a):
        out.append(_line("ok", "edits span", f"{pd.Timestamp(e.a):%d %b %H:%M} -> {pd.Timestamp(e.b):%d %b %H:%M} UTC, {int(e.w)} wikis"))
        out.append(_line("ok", "page views span", f"{pd.Timestamp(p.a):%d %b %H:%M} -> {pd.Timestamp(p.b):%d %b %H:%M} UTC, {int(p.n)} hours loaded"))
        start = max(pd.Timestamp(e.a).ceil("h") + pd.Timedelta(hours=3), pd.Timestamp(p.a) + pd.Timedelta(days=7, hours=3))
        end = min(pd.Timestamp(e.b).floor("h"), pd.Timestamp(p.b) - pd.Timedelta(hours=3))
        days = max(0, (end.normalize() - start.normalize()).days + 1) if end > start else 0
        s = cfg["splits"]
        need = s["test_days"] + s["cv_folds"] + s["min_train_days"]
        status = "ok" if days >= need else ("warn" if days >= 6 else "fail")
        hint = "" if days >= need else (" -> use configs/train_short.json" if days >= 6 else
                                        " -> need more backfill (edits, plus page views 7 days before them)")
        out.append(_line(status, "trainable days", f"{days} days of decision times ({start:%d %b %H:%M} -> {end:%d %b %H:%M}), "
                                                    f"config needs {need}{hint}"))

    # published_at must be the dump's Last-Modified (about +2 h). If it equals the hour, every feature looks ahead.
    lag = store.query(f"""SELECT hour_utc, MAX(published_at) AS published_at FROM PAGEVIEWS_HOURLY
                          WHERE hour_utc >= {ts_literal(pd.Timestamp(p.b) - pd.Timedelta(days=2))} GROUP BY hour_utc""") \
        if pd.notna(p.b) else pd.DataFrame()
    if len(lag):
        mins = ((pd.to_datetime(lag.published_at) - pd.to_datetime(lag.hour_utc)) / pd.Timedelta(minutes=1)).median()
        v = cfg["validation"]
        ok = v["min_publish_lag_min"] <= mins <= v["max_publish_lag_min"]
        out.append(_line("ok" if ok else "fail", "published_at", f"median {mins:.0f} min after the hour" + (
            "" if ok else " (expected ~120: the loader must store the dump's Last-Modified time)")))

    # Main Page and measurements: optional, but they change results (scheduled surges, real boot times).
    if store.has_table("MAIN_PAGE_EVENTS"):
        mp = store.query("SELECT slot, COUNT(*) AS n FROM MAIN_PAGE_EVENTS GROUP BY slot")
        d = dict(zip(mp.slot, mp.n))
        out.append(_line("ok" if d.get("tfa") and d.get("itn") else "warn", "Main Page events",
                         f"{d.get('tfa', 0)} TFA, {d.get('itn', 0)} ITN"))
    if store.has_table("INFRA_MEASUREMENTS"):
        im = store.query("SELECT kind, region, COUNT(*) AS n FROM INFRA_MEASUREMENTS GROUP BY kind, region")
        boots = sorted(im[im.kind == "boot"].region.dropna().unique())
        full = len(boots) == 3 and (im.kind == "capacity").any()
        out.append(_line("ok" if full else "warn", "infra measurements",
                         f"boot times for {boots or 'no regions'}, capacity {'yes' if (im.kind == 'capacity').any() else 'missing'}"
                         + ("" if full else " (engine uses defaults until PC1 writes them)")))
    return out


def write_probe(store: Store) -> list[dict]:
    """Can this role write what PC3 writes? Each INSERT runs inside a transaction that is rolled back."""
    out = []
    t = ts_literal("2000-01-01 00:00:00")
    for table, values in WRITE_TABLES.items():
        try:
            store.execute("BEGIN")
            store.execute(f"INSERT INTO {table} " + values.format(t=t))
            store.execute("ROLLBACK")
            out.append(_line("ok", f"write {table}", "insert allowed (rolled back)"))
        except Exception as ex:                                      # noqa: BLE001
            try:
                store.execute("ROLLBACK")
            except Exception:                                        # noqa: BLE001
                pass
            out.append(_line("fail", f"write {table}", f"{str(ex).splitlines()[0][:140]} -> ask PC1 to GRANT INSERT, DELETE"))
    # Temporary tables carry the candidate list and the SQL features; they need CREATE TABLE on the schema.
    try:
        store.execute("CREATE OR REPLACE TEMPORARY TABLE FC_PROBE (x INT)")
        store.execute("DROP TABLE IF EXISTS FC_PROBE")
        out.append(_line("ok", "temporary tables", "allowed"))
    except Exception as ex:                                          # noqa: BLE001
        out.append(_line("fail", "temporary tables", f"{str(ex).splitlines()[0][:140]} -> ask PC1 to GRANT CREATE TABLE ON SCHEMA CORE"))
    return out


def setup(store: Store) -> list[dict]:
    """One-time objects PC3 owns in the shared schema."""
    out = []
    try:
        store.execute(surge_hours_sql(store.dialect))
        n = int(store.query("SELECT COUNT(*) AS n FROM SURGE_HOURS").n.iloc[0])
        out.append(_line("ok", "SURGE_HOURS view", f"created, {n:,} surge hours so far (PC1's scorecard reads this)"))
    except Exception as ex:                                          # noqa: BLE001
        out.append(_line("fail", "SURGE_HOURS view", f"{str(ex).splitlines()[0][:140]} -> ask PC1 to GRANT CREATE VIEW"))
    try:
        store.execute("CREATE TABLE IF NOT EXISTS WIKIDATA_ITEMS (wiki STRING, title STRING, qid STRING)"
                      if store.dialect == "snowflake" else
                      "CREATE TABLE IF NOT EXISTS WIKIDATA_ITEMS (wiki VARCHAR, title VARCHAR, qid VARCHAR)")
        out.append(_line("ok", "WIKIDATA_ITEMS", "present (optional cross-language feature)"))
    except Exception as ex:                                          # noqa: BLE001
        out.append(_line("warn", "WIKIDATA_ITEMS", f"not created ({str(ex).splitlines()[0][:100]}); feature stays 0"))
    return out

## 4. Labels and features

The shared surge rule (also as the `SURGE_HOURS` view), the decision grid and every feature as of *t*, plus the Snowflake SQL version of the features (tested equal to the Python one).

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/labels.py
"""Surge definition (CONTRACT §7), shared with PC1's scorecard.

surge hour : views(h) >= max(SURGE_MULT x median(views at the same hour on the previous 7 days, missing = 0), FLOOR[wiki])
onset      : first surge hour after a non-surge hour
y_h        : 1 if an onset falls in (t, t+h]  -> onset hour buckets t .. t+h-1 (bucket h covers views in [h, h+1))
size target: log1p(peak views in (t, t+3])   -> max over buckets t .. t+2
"""
from __future__ import annotations

import numpy as np

from . import config as C


# Stack the 7 previous same-hour values (24 h apart) and take the median; missing history counts as 0.
def med7_same_hour(v: np.ndarray) -> np.ndarray:
    """Median of v[h-24k], k=1..7 (missing = 0). v is a dense hourly array; first 168 h get partial history."""
    n = len(v)
    stack = np.zeros((7, n))
    for k in range(1, 8):
        stack[k - 1, 24 * k:] = v[: n - 24 * k] if n > 24 * k else 0
    return np.median(stack, axis=0)


# Surge hour = at least 5x that median AND at least the wiki's floor.
def surge_flags(v: np.ndarray, wiki: str) -> np.ndarray:
    med = med7_same_hour(v)
    return v >= np.maximum(C.SURGE_MULT * med, C.surge_floor(wiki))


# Onset = surge hour whose previous hour was not a surge.
def onset_flags(surge: np.ndarray) -> np.ndarray:
    prev = np.r_[False, surge[:-1]]
    return surge & ~prev


# The same rule as a SQL view for PC1's scorecard (parity with surge_flags is a test).
def surge_hours_sql(dialect: str = "snowflake", table: str = "PAGEVIEWS_HOURLY") -> str:
    """SQL for the SURGE_HOURS view PC1's scorecard uses (same rule as ``surge_flags``; parity is tested).

    Rows with < 20 views are absent from the table, so missing lags count as 0 via LEFT JOIN + COALESCE.
    Only rows that exist need evaluating: a surge hour always has views >= the lowest FLOOR.
    """
    lag = "DATEADD(day, -ks.k, p.hour_utc)" if dialect == "snowflake" else "p.hour_utc - ks.k * INTERVAL 1 DAY"
    floor = ("CASE p.wiki " + " ".join(f"WHEN '{w}' THEN {f}" for w, f in C.SURGE_FLOOR.items())
             + f" ELSE {C.SURGE_FLOOR_DEFAULT} END")
    ks = " UNION ALL ".join(f"SELECT {k} AS k" for k in range(1, 8))
    lowest = min([C.SURGE_FLOOR_DEFAULT, *C.SURGE_FLOOR.values()])
    return f"""CREATE OR REPLACE VIEW SURGE_HOURS AS
WITH ks AS ({ks}),
lagged AS (
  SELECT p.hour_utc, p.wiki, p.title, p.views, p.published_at, {floor} AS floor_views, COALESCE(l.views, 0) AS v
  FROM {table} p
  CROSS JOIN ks
  LEFT JOIN {table} l ON l.wiki = p.wiki AND l.title = p.title AND l.hour_utc = {lag}
  WHERE p.views >= {lowest}
),
med AS (
  SELECT hour_utc, wiki, title, views, published_at, floor_views, MEDIAN(v) AS med7
  FROM lagged
  GROUP BY hour_utc, wiki, title, views, published_at, floor_views
)
SELECT hour_utc, wiki, title, views, published_at, med7, floor_views
FROM med
WHERE views >= GREATEST({C.SURGE_MULT} * med7, floor_views)"""

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/features.py
"""Decision grid, as-of-t features and labels (CONTRACT §7). Pure functions of ``RawData``.

Availability rules (the no-look-ahead contract):
  edits          ts <= t
  page views     hour h only if hour_published[h] <= t       (published_at, not hour + 2 h guesswork)
  Main Page      ts <= t
  cause labels   ts <= t (and from the last 3 h)
  page creation  first type='new' edit with ts <= t
Each row carries provenance columns (max_*_used) so the leak check can assert all of them are <= t.
"""
from __future__ import annotations

import warnings
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from . import config as C
from .data import HOUR, RawData, edit_candidates
from .labels import med7_same_hour, onset_flags, surge_flags

# Work in int64 nanoseconds so every window is a pair of binary searches.
NS_MIN = 60 * 10**9
NS_H = 60 * NS_MIN
WIKI_ID = {w: i for i, w in enumerate(C.WIKIS)}


def _ns(x) -> np.ndarray:
    return pd.to_datetime(pd.Series(x)).to_numpy("datetime64[ns]").astype(np.int64)


# ---------------------------------------------------------------- grid
def last_published_hour(raw: RawData, ts: pd.Series) -> pd.Series:
    """For each t: the newest hour whose dump was published at or before t."""
    hp = raw.hour_published.sort_values()
    hours = hp.index.to_numpy("datetime64[ns]").astype(np.int64)
    cmax = np.maximum.accumulate(hours)
    pub = hp.to_numpy("datetime64[ns]").astype(np.int64)
    idx = np.searchsorted(pub, _ns(ts), side="right") - 1
    out = np.where(idx >= 0, cmax[np.clip(idx, 0, None)], np.iinfo(np.int64).min)
    return pd.Series(pd.to_datetime(np.where(idx >= 0, out, 0)).where(idx >= 0), index=ts.index)


# Decision grid = union of the three candidate sources for every issue time in the window.
def make_grid(raw: RawData, start=None, end=None, every: str = "h") -> pd.DataFrame:
    """(wiki, title, t) candidates for every t in [start, end] (CONTRACT §7 decision grid)."""
    start = pd.Timestamp(start or raw.start)
    end = pd.Timestamp(end or raw.end)
    ts = pd.Series(pd.date_range(start, end, freq=every))
    parts = [edit_candidates(raw.edits, start, end)]
    if every != "h":                                       # off-hour issue times (live every 15 min)
        # Off-hour issue times (live job every 15 min): count edits in (t-3h, t] directly for each t.
        e = raw.edits
        rows = []
        for t in ts:
            w = e[(e.ts > t - 3 * HOUR) & (e.ts <= t)].groupby(["wiki", "title"]).size()
            w = w[w >= C.CAND_EDITS_3H].reset_index()[["wiki", "title"]]
            rows.append(w.assign(t=t))
        parts = [pd.concat(rows) if rows else parts[0]]
    # Top-500 rows are taken from the latest PUBLISHED hour at each t, never from an hour not yet out.
    lh = pd.DataFrame({"t": ts, "lh": last_published_hour(raw, ts)})
    top = raw.top.merge(lh, left_on="hour_utc", right_on="lh")[["wiki", "title", "t"]]
    parts.append(top)
    mp = raw.main_page
    for t in ts:                                           # Main Page in the last 6 h
        m = mp[(mp.ts > t - C.CAND_MAIN_PAGE_H * HOUR) & (mp.ts <= t)]
        if len(m):
            parts.append(m[["wiki", "title"]].assign(t=t))
    g = pd.concat(parts, ignore_index=True).drop_duplicates()
    g = g[g.wiki.isin(C.WIKIS)]
    return g.sort_values(["t", "wiki", "title"]).reset_index(drop=True)


# ---------------------------------------------------------------- features + labels
def build(raw: RawData, grid: pd.DataFrame, with_labels: bool = True, with_causes: bool = True) -> pd.DataFrame:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)          # all-NaN medians before history exists
        return _build(raw, grid, with_labels, with_causes)


def _build(raw: RawData, grid: pd.DataFrame, with_labels: bool, with_causes: bool) -> pd.DataFrame:
    if grid.empty:
        return grid.assign(**{c: [] for c in C.BASE_FEATURES})
    grid = grid.reset_index(drop=True)
    t_ns_all = _ns(grid.t)

    # A dense hourly axis covering history and labels; pub_ns[h] = when hour h became available.
    pv = raw.pageviews
    hp = raw.hour_published
    h0 = min([x for x in (pv.hour_utc.min() if len(pv) else None, hp.index.min() if len(hp) else None) if x is not None])
    h1 = max([x for x in (pv.hour_utc.max() if len(pv) else None, hp.index.max() if len(hp) else None,
                          grid.t.max() + 3 * HOUR) if x is not None])
    n_h = int((h1 - h0) / HOUR) + 1
    h0_ns = pd.Timestamp(h0).value
    pub_ns = np.full(n_h, np.iinfo(np.int64).max, dtype=np.int64)
    hi = ((hp.index - h0) / HOUR).astype(int)
    keep = (hi >= 0) & (hi < n_h)
    pub_ns[hi[keep]] = hp.to_numpy("datetime64[ns]").astype(np.int64)[keep]
    last_obs_h = int((pd.Timestamp(hp.index.max()) - h0) / HOUR) if len(hp) else -1   # truth exists up to here
    lph = last_published_hour(raw, grid.t)
    lh_all = np.where(lph.notna(), ((lph - h0) / HOUR).fillna(-1).astype(int), -1)

    # Index edits once (per title, sorted) so each title's windows are slices, not scans.
    e = raw.edits
    e_idx = e.groupby(["wiki", "title"]).indices if len(e) else {}
    e_ts = _ns(e.ts) if len(e) else np.array([], dtype=np.int64)
    e_bot = e.is_bot.to_numpy(bool) if len(e) else np.array([], bool)
    e_rev = e.is_revert.to_numpy(bool) if len(e) else np.array([], bool)
    e_bytes = np.maximum(e.delta_bytes.to_numpy(float), 0) if len(e) else np.array([])
    e_user = pd.Categorical(e.user_name).codes if len(e) else np.array([], int)
    e_wiki = pd.Categorical(e.wiki, categories=C.WIKIS).codes if len(e) else np.array([], int)

    # Same for page views, Main Page events, page creation, Wikidata items and cause labels.
    pv_idx = pv.groupby(["wiki", "title"]).indices if len(pv) else {}
    pv_h = ((pv.hour_utc - h0) / HOUR).astype(int).to_numpy() if len(pv) else np.array([], int)
    pv_v = pv.views.to_numpy(float) if len(pv) else np.array([])

    mp = raw.main_page
    mp_idx = mp.groupby(["wiki", "title"]).indices if len(mp) else {}
    mp_ts = _ns(mp.ts) if len(mp) else np.array([], dtype=np.int64)
    mp_slot = mp.slot.to_numpy() if len(mp) else np.array([])

    created = raw.meta.get("page_created", pd.Series(dtype="datetime64[ns]"))
    qid_of = {(w, t): q for w, t, q in raw.wikidata[["wiki", "title", "qid"]].itertuples(index=False)} if len(raw.wikidata) else {}
    q_rows: dict[str, list[int]] = {}
    for key, ix in e_idx.items():
        q = qid_of.get(key)
        if q is not None:
            q_rows.setdefault(q, []).extend(ix.tolist())
    q_arr = {q: np.array(sorted(ix, key=lambda i: e_ts[i])) for q, ix in q_rows.items()}

    cz = raw.causes
    cz_idx = cz.groupby(["wiki", "title"]).indices if (with_causes and len(cz)) else {}
    cz_ts = _ns(cz.ts) if len(cz) else np.array([], dtype=np.int64)

    # Output columns start as NaN and are filled title by title.
    out = {c: np.full(len(grid), np.nan) for c in C.BASE_FEATURES + C.CAUSE_FEATURES +
           ["max_edit_ts_used", "max_pub_used", "max_mainpage_used", "max_cause_used", "y_1", "y_2", "y_3",
            "size_target", "region_main_w"]}
    cause_label = np.full(len(grid), None, dtype=object)

    for (wiki, title), rows in grid.groupby(["wiki", "title"]).indices.items():
        t = t_ns_all[rows]
        # ---------------- edits
        ix = e_idx.get((wiki, title), np.array([], int))
        ts_all = e_ts[ix]
        hum = ix[~e_bot[ix]]
        th = e_ts[hum]
        tb = e_ts[ix[e_bot[ix]]]

        def cnt(arr, w_min):
            return np.searchsorted(arr, t, "right") - np.searchsorted(arr, t - w_min * NS_MIN, "right")

        # Human edit counts in each window: (# edits <= t) - (# edits <= t - window).
        for w in (15, 30, 60, 180):
            out[f"edits_{w}m"][rows] = cnt(th, w)
        b60 = cnt(tb, 60)
        e60 = out["edits_60m"][rows]
        out["bot_share_60m"][rows] = np.where(b60 + e60 > 0, b60 / np.maximum(b60 + e60, 1), 0.0)
        out["accel_30m"][rows] = out["edits_30m"][rows] - (e60 - out["edits_30m"][rows])
        # Cumulative sums turn revert share and bytes added into O(1) window differences.
        cum_rev = np.r_[0, np.cumsum(e_rev[hum])]
        cum_bytes = np.r_[0, np.cumsum(e_bytes[hum])]
        r_hi = np.searchsorted(th, t, "right")
        r_lo60 = np.searchsorted(th, t - 60 * NS_MIN, "right")
        r_lo180 = np.searchsorted(th, t - 180 * NS_MIN, "right")
        out["revert_share_60m"][rows] = np.where(e60 > 0, (cum_rev[r_hi] - cum_rev[r_lo60]) / np.maximum(e60, 1), 0.0)
        out["bytes_added_60m"][rows] = cum_bytes[r_hi] - cum_bytes[r_lo60]
        last = np.where(r_hi > 0, th[np.clip(r_hi - 1, 0, None)] if len(th) else 0, np.iinfo(np.int64).min)
        out["mins_since_last_edit"][rows] = np.where(r_hi > 0, np.minimum((t - last) / NS_MIN, 1440), 1440)
        # Distinct human editors need a unique() per row, only where there were edits.
        users = e_user[hum]
        h60 = np.zeros(len(rows))
        h180 = np.zeros(len(rows))
        for j in np.flatnonzero(r_hi > r_lo180):
            h180[j] = np.unique(users[r_lo180[j]: r_hi[j]]).size
            h60[j] = np.unique(users[r_lo60[j]: r_hi[j]]).size if r_hi[j] > r_lo60[j] else 0
        out["humans_60m"][rows], out["humans_180m"][rows] = h60, h180
        a_hi = np.searchsorted(ts_all, t, "right")
        out["max_edit_ts_used"][rows] = np.where(a_hi > 0, ts_all[np.clip(a_hi - 1, 0, None)] if len(ts_all) else 0, np.nan)

        # ---------------- page lifecycle
        c = created.get((wiki, title)) if len(created) else None
        if c is not None and not pd.isna(c):
            age = (t - pd.Timestamp(c).value) / NS_H
            out["page_age_h"][rows] = np.where(age >= 0, age, np.nan)
            out["is_new_page"][rows] = ((age >= 0) & (age <= 24)).astype(float)
        else:
            out["is_new_page"][rows] = 0.0

        # ---------------- views (published only)
        v = np.zeros(n_h)
        pix = pv_idx.get((wiki, title))
        if pix is not None:
            v[pv_h[pix]] = pv_v[pix]
        th_idx = ((t - h0_ns) // NS_H).astype(int)                        # hour index of t
        pub_ok = lambda hidx, tt: (hidx >= 0) & (hidx < n_h) & (pub_ns[np.clip(hidx, 0, n_h - 1)] <= tt)
        lh = lh_all[rows]                                                  # newest published hour <= t
        has = lh >= 0
        val = lambda hidx, tt: np.where(pub_ok(hidx, tt), v[np.clip(hidx, 0, n_h - 1)], np.nan)
        lv = val(lh, t)
        out["last_pub_views"][rows] = np.where(has, lv, np.nan)
        out["log_last_pub_views"][rows] = np.log1p(out["last_pub_views"][rows])
        # Same-hour 7-day median around the latest published hour, unpublished hours ignored.
        med_lh = np.nanmedian(np.stack([val(lh - 24 * k, t) for k in range(1, 8)]), axis=0) if len(rows) else []
        med_lh = np.nan_to_num(med_lh, nan=0.0)
        out["ratio_same_hour_med7"][rows] = (lv + 1) / (med_lh + 1)
        out["slope_3h"][rows] = (lv - val(lh - 2, t)) / 2.0
        out["views_same_hour_yesterday"][rows] = val(th_idx - 24, t)
        med_next = np.nanmedian(np.stack([val(th_idx - 24 * k, t) for k in range(1, 8)]), axis=0)
        out["med7_next_hour"][rows] = med_next
        out["hours_since_pub"][rows] = np.where(has, (t - (h0_ns + lh * NS_H)) / NS_H, np.nan)
        out["surge_last_pub"][rows] = np.where(has, lv >= np.maximum(C.SURGE_MULT * med_lh, C.surge_floor(wiki)), 0).astype(float)
        # Provenance: the newest publication time among all hours this row read.
        used = np.stack([lh, lh - 2] + [lh - 24 * k for k in range(1, 8)] + [th_idx - 24 * k for k in range(1, 8)])
        pub_used = np.where(pub_ok(used, t), pub_ns[np.clip(used, 0, n_h - 1)], np.iinfo(np.int64).min)
        out["max_pub_used"][rows] = pub_used.max(axis=0)

        # ---------------- Main Page
        mix = mp_idx.get((wiki, title), np.array([], int))
        tfa_ts = mp_ts[mix][mp_slot[mix] == "tfa"]
        itn_ts = np.sort(mp_ts[mix][mp_slot[mix] == "itn"])
        day0 = (t // (24 * NS_H)) * 24 * NS_H
        out["is_tfa_today"][rows] = np.array([float(((tfa_ts >= d) & (tfa_ts <= tt)).any()) for d, tt in zip(day0, t)])
        k_itn = np.searchsorted(itn_ts, t, "right")
        since = np.where(k_itn > 0, (t - itn_ts[np.clip(k_itn - 1, 0, None)]) / NS_MIN if len(itn_ts) else np.nan, np.nan)
        since = np.where(since <= 48 * 60, since, np.nan)
        out["mins_since_itn"][rows] = since
        out["in_itn_6h"][rows] = (np.nan_to_num(since, nan=1e9) <= 360).astype(float)
        mp_all = np.sort(mp_ts[mix])
        k_mp = np.searchsorted(mp_all, t, "right")
        out["max_mainpage_used"][rows] = np.where(k_mp > 0, mp_all[np.clip(k_mp - 1, 0, None)] if len(mp_all) else 0, np.nan)

        # ---------------- time
        out["hour_utc"][rows] = (t // NS_H) % 24
        out["dow"][rows] = ((t // (24 * NS_H)) + 3) % 7                    # 1970-01-01 was a Thursday -> Mon=0
        tz = ZoneInfo(C.REGION_TZ[C.WIKI_MAIN_REGION[wiki]])
        out["local_hour"][rows] = pd.to_datetime(t).tz_localize("UTC").tz_convert(tz).hour
        out["wiki_id"][rows] = WIKI_ID[wiki]

        # ---------------- cross-language (same Wikidata item, other tracked wikis)
        q = qid_of.get((wiki, title))
        if q is not None and q in q_arr:
            qa = q_arr[q]
            qa = qa[~e_bot[qa]]
            qts, qw = e_ts[qa], e_wiki[qa]
            hi_ = np.searchsorted(qts, t, "right")
            lo_ = np.searchsorted(qts, t - 3 * NS_H, "right")
            me = WIKI_ID[wiki]
            out["n_other_wikis_3h"][rows] = [np.setdiff1d(np.unique(qw[a:b]), [me]).size for a, b in zip(lo_, hi_)]
        else:
            out["n_other_wikis_3h"][rows] = 0.0

        # ---------------- causes (Gemini labels written before t, last 3 h)
        for cc in C.CAUSE_FEATURES:
            out[cc][rows] = 0.0
        cix = cz_idx.get((wiki, title))
        if cix is not None:
            order = cix[np.argsort(cz_ts[cix])]
            cts = cz_ts[order]
            k_c = np.searchsorted(cts, t, "right") - 1
            for j, kc in enumerate(k_c):
                if kc >= 0 and t[j] - cts[kc] <= 3 * NS_H:
                    r_ = cz.iloc[order[kc]]
                    if r_.cause in C.CAUSES:
                        out[f"cause_{r_.cause}"][rows[j]] = 1.0
                        out["cause_confidence"][rows[j]] = float(r_.confidence or 0)
                        cause_label[rows[j]] = r_.cause
                        out["max_cause_used"][rows[j]] = cts[kc]

        # ---------------- labels (ground truth; never features)
        if with_labels:
            s = surge_flags(v, wiki)
            o = onset_flags(s)
            co = np.r_[0, np.cumsum(o)]
            for hzn in C.HORIZONS:
                end_h = th_idx + hzn - 1
                ok = end_h <= last_obs_h
                lo_i = np.clip(th_idx, 0, n_h)
                hi_i = np.clip(end_h + 1, 0, n_h)
                out[f"y_{hzn}"][rows] = np.where(ok, (co[hi_i] - co[lo_i]) > 0, np.nan)
            ok3 = th_idx + 2 <= last_obs_h
            peak = np.array([v[max(a, 0): a + 3].max() if a + 3 <= n_h else np.nan for a in th_idx])
            out["size_target"][rows] = np.where(ok3, np.log1p(peak), np.nan)

    # Assemble the frame; provenance columns become timestamps for the leak check.
    f = grid.copy()
    for k, arr in out.items():
        f[k] = arr
    for c in ("max_edit_ts_used", "max_pub_used", "max_mainpage_used", "max_cause_used"):
        f[c] = pd.to_datetime(f[c], unit="ns")
    f["cause"] = cause_label
    f["region"] = f["wiki"].map(C.WIKI_MAIN_REGION)
    f = f.drop(columns=["region_main_w"])
    return f


# Ground-truth onset hours for a set of titles (lead time, boss levels, the surge chart).
def onsets(raw: RawData, titles: pd.DataFrame) -> pd.DataFrame:
    """Ground-truth onset hours for the given (wiki, title) pairs (for lead time and boss levels)."""
    pv = raw.pageviews
    rows = []
    if pv.empty:
        return pd.DataFrame(columns=["wiki", "title", "hour_utc", "views"])
    h0, h1 = pv.hour_utc.min(), pv.hour_utc.max()
    n_h = int((h1 - h0) / HOUR) + 1
    want = set(map(tuple, titles[["wiki", "title"]].drop_duplicates().to_numpy()))
    for (w, t), g in pv.groupby(["wiki", "title"]):
        if (w, t) not in want:
            continue
        v = np.zeros(n_h)
        v[((g.hour_utc - h0) / HOUR).astype(int).to_numpy()] = g.views.to_numpy()
        o = onset_flags(surge_flags(v, w))
        for i in np.flatnonzero(o):
            rows.append((w, t, h0 + i * HOUR, v[i]))
    return pd.DataFrame(rows, columns=["wiki", "title", "hour_utc", "views"])


LABEL_COLS = ["y_1", "y_2", "y_3", "size_target"]
PROVENANCE = ["max_edit_ts_used", "max_pub_used", "max_mainpage_used", "max_cause_used"]

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/sql_features.py
"""Features in Snowflake SQL (CONTRACT §7 "features (Snowflake SQL)"), rendered for Snowflake or DuckDB.

The same SQL text runs in both dialects except for four small time helpers, so the DuckDB run in the tests proves
the Snowflake version computes the same numbers as the Python pipeline (tests/test_sql_parity.py).

    python -m flashcrowd_model.sql_features --dialect snowflake --start 2026-10-17 --end 2026-10-24 > features.sql

Tables it creates (temporary):
  FC_HOUR_PUBLISHED  hour -> when its dump was published, and the newest hour published by then (the ASOF key)
  FC_GRID            decision grid: >= 3 edits in (t-3h, t], top 500 of the latest published hour, Main Page in 6 h
  FC_FEATURES_SQL    edit, view and Main Page features as of t, plus provenance columns for the leak check
Python-only features (page age, cross-language, causes, time of day) are added in features.py.
"""
from __future__ import annotations

import argparse

import pandas as pd

from . import config as C
from .labels import surge_hours_sql
from .store import ts_literal


# The only dialect differences: time arithmetic, hour series, truncation and the ASOF join spelling.
class Dialect:
    def __init__(self, name: str):
        self.name = name

    def add(self, unit: str, n, col: str) -> str:
        if self.name == "snowflake":
            return f"DATEADD({unit}, {n}, {col})"
        return f"({col} + ({n}) * INTERVAL 1 {unit.upper()})"

    def hours(self, start, end) -> str:
        n = int((pd.Timestamp(end) - pd.Timestamp(start)) / pd.Timedelta(hours=1)) + 1
        if self.name == "snowflake":
            return (f"SELECT DATEADD(hour, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, {ts_literal(start)}::TIMESTAMP_NTZ) AS t "
                    f"FROM TABLE(GENERATOR(ROWCOUNT => {n}))")
        return f"SELECT CAST(range AS TIMESTAMP) AS t FROM range(TIMESTAMP {ts_literal(start)}, TIMESTAMP {ts_literal(end)} + INTERVAL 1 HOUR, INTERVAL 1 HOUR)"

    def minutes_between(self, a: str, b: str) -> str:
        if self.name == "snowflake":
            return f"DATEDIFF(second, {a}, {b}) / 60.0"
        return f"(epoch({b}) - epoch({a})) / 60.0"

    def asof(self, left: str, right: str, cond: str) -> str:
        if self.name == "snowflake":
            return f"{left} ASOF JOIN {right} MATCH_CONDITION ({cond})"
        return f"{left} ASOF JOIN {right} ON {cond}"


def render(dialect: str, start, end, top_n: int = C.CAND_TOP_N) -> list[str]:
    d = Dialect(dialect)
    s, e = f"CAST({ts_literal(start)} AS TIMESTAMP)", f"CAST({ts_literal(end)} AS TIMESTAMP)"
    wikis = ", ".join(f"'{w}'" for w in C.WIKIS)
    tmp = "CREATE OR REPLACE TEMPORARY TABLE"

    stmts = []
    # Hour -> published time, and the newest hour whose dump existed by that moment. A late file for an old hour
    # must not make "latest published hour" go backwards, hence the running MAX over publication order.
    stmts.append(f"""{tmp} FC_HOUR_PUBLISHED AS
SELECT hour_utc, published_at,
       MAX(hour_utc) OVER (ORDER BY published_at, hour_utc ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS latest_hour
FROM (SELECT hour_utc, MAX(published_at) AS published_at FROM PAGEVIEWS_HOURLY
      WHERE hour_utc >= {d.add('day', -9, s)} AND hour_utc <= {e} GROUP BY hour_utc) x""")

    # Decision times t (whole hours) and, for each, the latest published hour via the ASOF JOIN.
    stmts.append(f"""{tmp} FC_TIMES AS
SELECT h.t, p.latest_hour AS last_pub_hour
FROM {d.asof(f"({d.hours(start, end)}) h", "FC_HOUR_PUBLISHED p", "h.t >= p.published_at")}""")

    # The decision grid (CONTRACT §7): three candidate sources, unioned.
    stmts.append(f"""{tmp} FC_GRID AS
SELECT DISTINCT wiki, title, t FROM (
  SELECT e.wiki, e.title, g.t
  FROM FC_TIMES g JOIN EDITS e ON e.ts > {d.add('hour', -3, 'g.t')} AND e.ts <= g.t
  WHERE e.wiki IN ({wikis}) AND e.namespace = 0
  GROUP BY e.wiki, e.title, g.t HAVING COUNT(*) >= {C.CAND_EDITS_3H}
  UNION ALL
  SELECT v.wiki, v.title, g.t
  FROM FC_TIMES g JOIN (SELECT hour_utc, wiki, title FROM PAGEVIEWS_HOURLY
                        WHERE wiki IN ({wikis}) AND hour_utc >= {d.add('hour', -6, s)} AND hour_utc <= {e}
                        QUALIFY ROW_NUMBER() OVER (PARTITION BY wiki, hour_utc ORDER BY views DESC) <= {top_n}) v
    ON v.hour_utc = g.last_pub_hour
  UNION ALL
  SELECT m.wiki, m.title, g.t
  FROM FC_TIMES g JOIN MAIN_PAGE_EVENTS m ON m.ts > {d.add('hour', -C.CAND_MAIN_PAGE_H, 'g.t')} AND m.ts <= g.t
  WHERE m.wiki IN ({wikis})
) c""")

    # Edit features as of t: human edits in four windows, distinct human editors, bot share, reverts, bytes.
    w = lambda m: d.add("minute", -m, "g.t")
    stmts.append(f"""{tmp} FC_EDIT_FEATS AS
SELECT g.wiki, g.title, g.t,
  COALESCE(COUNT_IF(NOT e.is_bot AND e.ts > {w(15)}), 0)  AS edits_15m,
  COALESCE(COUNT_IF(NOT e.is_bot AND e.ts > {w(30)}), 0)  AS edits_30m,
  COALESCE(COUNT_IF(NOT e.is_bot AND e.ts > {w(60)}), 0)  AS edits_60m,
  COALESCE(COUNT_IF(NOT e.is_bot AND e.ts > {w(180)}), 0) AS edits_180m,
  COUNT(DISTINCT CASE WHEN NOT e.is_bot AND e.ts > {w(60)} THEN e.user_name END)  AS humans_60m,
  COUNT(DISTINCT CASE WHEN NOT e.is_bot AND e.ts > {w(180)} THEN e.user_name END) AS humans_180m,
  COALESCE(COUNT_IF(e.is_bot AND e.ts > {w(60)}), 0) AS bots_60m,
  COALESCE(COUNT_IF(NOT e.is_bot AND e.is_revert AND e.ts > {w(60)}), 0) AS reverts_60m,
  SUM(CASE WHEN NOT e.is_bot AND e.ts > {w(60)} THEN GREATEST(e.delta_bytes, 0) ELSE 0 END) AS bytes_added_60m,
  MAX(CASE WHEN NOT e.is_bot THEN e.ts END) AS last_human_ts,
  MAX(e.ts) AS max_edit_ts_used
FROM FC_GRID g
LEFT JOIN EDITS e ON e.wiki = g.wiki AND e.title = g.title AND e.ts <= g.t AND e.ts > {w(1440)} AND e.namespace = 0
GROUP BY g.wiki, g.title, g.t""")

    # View features: only hours whose dump was PUBLISHED by t. Missing rows of a published hour count as 0 views
    # (the loader drops < 20); unpublished hours are NULL and ignored by MEDIAN.
    ks = " UNION ALL ".join(f"SELECT {k} AS k" for k in range(1, 8))
    stmts.append(f"""{tmp} FC_VIEW_FEATS AS
WITH base AS (
  SELECT g.wiki, g.title, g.t, ft.last_pub_hour FROM FC_GRID g JOIN FC_TIMES ft ON ft.t = g.t
),
med AS (
  SELECT b.wiki, b.title, b.t,
         MEDIAN(CASE WHEN hp.published_at <= b.t THEN COALESCE(v.views, 0) END) AS med7_last
  FROM base b CROSS JOIN ({ks}) ks
  LEFT JOIN FC_HOUR_PUBLISHED hp ON hp.hour_utc = {d.add('day', '-ks.k', 'b.last_pub_hour')}
  LEFT JOIN PAGEVIEWS_HOURLY v ON v.wiki = b.wiki AND v.title = b.title AND v.hour_utc = hp.hour_utc
  GROUP BY b.wiki, b.title, b.t
)
SELECT b.wiki, b.title, b.t, b.last_pub_hour,
       CASE WHEN b.last_pub_hour IS NOT NULL THEN COALESCE(lv.views, 0) END AS last_pub_views,
       {d.minutes_between('b.last_pub_hour', 'b.t')} / 60.0 AS hours_since_pub,
       (COALESCE(lv.views, 0) + 1.0) / (COALESCE(m.med7_last, 0) + 1.0) AS ratio_same_hour_med7,
       CASE WHEN hy.published_at <= b.t THEN COALESCE(yv.views, 0) END AS views_same_hour_yesterday,
       hl.published_at AS max_pub_used
FROM base b
JOIN med m ON m.wiki = b.wiki AND m.title = b.title AND m.t = b.t
LEFT JOIN FC_HOUR_PUBLISHED hl ON hl.hour_utc = b.last_pub_hour
LEFT JOIN PAGEVIEWS_HOURLY lv ON lv.wiki = b.wiki AND lv.title = b.title AND lv.hour_utc = b.last_pub_hour
LEFT JOIN FC_HOUR_PUBLISHED hy ON hy.hour_utc = {d.add('hour', -24, 'b.t')}
LEFT JOIN PAGEVIEWS_HOURLY yv ON yv.wiki = b.wiki AND yv.title = b.title AND yv.hour_utc = {d.add('hour', -24, 'b.t')}""")

    # Main Page features as of t: TFA today (scheduled at 00:00, so no look-ahead), minutes since added to ITN.
    stmts.append(f"""{tmp} FC_MP_FEATS AS
SELECT g.wiki, g.title, g.t,
  MAX(CASE WHEN m.slot = 'tfa' AND CAST(m.ts AS DATE) = CAST(g.t AS DATE) THEN 1 ELSE 0 END) AS is_tfa_today,
  MAX(CASE WHEN m.slot = 'itn' THEN m.ts END) AS last_itn_ts,
  MAX(m.ts) AS max_mainpage_used
FROM FC_GRID g
LEFT JOIN MAIN_PAGE_EVENTS m ON m.wiki = g.wiki AND m.title = g.title AND m.ts <= g.t AND m.ts > {d.add('hour', -48, 'g.t')}
GROUP BY g.wiki, g.title, g.t""")

    # Final table: derived ratios and the provenance columns the leak check reads.
    stmts.append(f"""{tmp} FC_FEATURES_SQL AS
SELECT g.wiki, g.title, g.t,
  ef.edits_15m, ef.edits_30m, ef.edits_60m, ef.edits_180m, ef.humans_60m, ef.humans_180m,
  CASE WHEN ef.bots_60m + ef.edits_60m > 0 THEN ef.bots_60m * 1.0 / (ef.bots_60m + ef.edits_60m) ELSE 0 END AS bot_share_60m,
  CASE WHEN ef.edits_60m > 0 THEN ef.reverts_60m * 1.0 / ef.edits_60m ELSE 0 END AS revert_share_60m,
  ef.bytes_added_60m,
  CASE WHEN ef.last_human_ts IS NULL THEN 1440 ELSE LEAST({d.minutes_between('ef.last_human_ts', 'g.t')}, 1440) END AS mins_since_last_edit,
  ef.edits_30m - (ef.edits_60m - ef.edits_30m) AS accel_30m,
  vf.last_pub_views, vf.hours_since_pub, vf.ratio_same_hour_med7, vf.views_same_hour_yesterday,
  mp.is_tfa_today,
  CASE WHEN mp.last_itn_ts IS NOT NULL THEN {d.minutes_between('mp.last_itn_ts', 'g.t')} END AS mins_since_itn,
  CASE WHEN mp.last_itn_ts IS NOT NULL AND {d.minutes_between('mp.last_itn_ts', 'g.t')} <= 360 THEN 1 ELSE 0 END AS in_itn_6h,
  ef.max_edit_ts_used, vf.max_pub_used, mp.max_mainpage_used
FROM FC_GRID g
JOIN FC_EDIT_FEATS ef ON ef.wiki = g.wiki AND ef.title = g.title AND ef.t = g.t
JOIN FC_VIEW_FEATS vf ON vf.wiki = g.wiki AND vf.title = g.title AND vf.t = g.t
JOIN FC_MP_FEATS mp ON mp.wiki = g.wiki AND mp.title = g.title AND mp.t = g.t""")
    return stmts


# The roadmap's leak test as SQL: must return 0 rows.
LEAK_CHECK_SQL = """SELECT wiki, title, t, max_edit_ts_used, max_pub_used, max_mainpage_used
FROM FC_FEATURES_SQL
WHERE max_edit_ts_used > t OR max_pub_used > t OR max_mainpage_used > t"""

SQL_FEATURE_COLS = ["edits_15m", "edits_30m", "edits_60m", "edits_180m", "humans_60m", "humans_180m", "bot_share_60m",
                    "revert_share_60m", "bytes_added_60m", "mins_since_last_edit", "accel_30m", "last_pub_views",
                    "hours_since_pub", "ratio_same_hour_med7", "views_same_hour_yesterday", "is_tfa_today",
                    "mins_since_itn", "in_itn_6h"]


def run(store, start, end) -> pd.DataFrame:
    for s in render(store.dialect, start, end):
        store.execute(s)
    bad = store.query(LEAK_CHECK_SQL)
    if len(bad):
        raise RuntimeError(f"SQL leak check returned {len(bad)} rows")
    return store.query("SELECT * FROM FC_FEATURES_SQL")


def main(argv=None):
    ap = argparse.ArgumentParser(description="Print the feature SQL for a window")
    ap.add_argument("--dialect", choices=["snowflake", "duckdb"], default="snowflake")
    ap.add_argument("--start", required=True)
    ap.add_argument("--end", required=True)
    a = ap.parse_args(argv)
    print(surge_hours_sql(a.dialect) + ";\n")
    for s in render(a.dialect, a.start, a.end):
        print(s + ";\n")
    print("-- leak check: must return 0 rows\n" + LEAK_CHECK_SQL + ";")


if __name__ == "__main__":
    main()

## 5. Leak gate

Provenance check, perturbation test, schema and split checks. Training refuses to run if any fails.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/leakcheck.py
"""No-look-ahead checks (CONTRACT §7 "Leak test"). Training refuses to run unless all of them pass.

1. provenance   for every row: max edit ts used, max published_at used, max Main Page ts used and max cause-label
                ts used are all <= t (the roadmap's "SQL check that returns 0 rows", done on the feature frame)
2. perturbation rewrite everything NOT available at a cut time (later edits, later-published page views, later Main
                Page events and cause labels) with junk, rebuild the grid and features, and require identical results
                for every t <= cut. A power check confirms the junk does change rows after the cut.
3. schema       no label or provenance column is a model feature
4. split        train/validation/test separated by the 3 h label embargo
"""
from __future__ import annotations

import json

import numpy as np
import pandas as pd

from . import config as C
from .data import HOUR, RawData
from .features import LABEL_COLS, PROVENANCE, build, make_grid


# Check 1: no provenance column may be later than its row's decision time.
def provenance_check(frame: pd.DataFrame) -> dict:
    bad = {}
    for c in PROVENANCE:
        if c in frame:
            n = int((frame[c].notna() & (frame[c] > frame["t"])).sum())
            if n:
                bad[c] = n
    return {"check": "provenance", "passed": not bad, "rows_after_t": bad, "rows": int(len(frame))}


# Junk everything that was NOT available at `cut`: later edits, later-published views, later events and labels.
def _corrupt(raw: RawData, cut: pd.Timestamp, rng: np.random.Generator) -> RawData:
    from dataclasses import replace
    e = raw.edits.copy()
    late = e.ts > cut
    e.loc[late, "user_name"] = [f"Ghost{i}" for i in rng.integers(0, 30, late.sum())]
    e.loc[late, "is_bot"] = rng.random(late.sum()) < 0.5
    e.loc[late, "is_revert"] = rng.random(late.sum()) < 0.5
    e.loc[late, "delta_bytes"] = rng.integers(-5000, 5000, late.sum())
    keys = e[["wiki", "title"]].drop_duplicates()
    flood = keys.sample(n=min(len(keys), 300), replace=True, random_state=int(rng.integers(1 << 30)))
    flood = flood.assign(ts=cut + pd.to_timedelta(rng.uniform(1, 180, len(flood)), unit="min"), type="edit",
                         user_name=[f"Flood{i}" for i in range(len(flood))], is_bot=False, is_revert=False,
                         delta_bytes=500, comment="flood")
    e = pd.concat([e, flood[e.columns]], ignore_index=True).sort_values("ts").reset_index(drop=True)
    p = raw.pageviews.copy()
    p["views"] = p["views"].astype("int64")
    hid = p.published_at > cut
    p.loc[hid, "views"] = rng.integers(0, 10**6, hid.sum())
    later_hours = raw.hour_published[raw.hour_published > cut].index
    top = raw.top.copy()
    if len(later_hours) and len(top):
        m = top.hour_utc.isin(later_hours)
        top.loc[m, "title"] = top.loc[m, "title"].sample(frac=1, random_state=1).to_numpy()
    mp = pd.concat([raw.main_page, keys.head(20).assign(ts=cut + pd.Timedelta(minutes=30), slot="itn")[raw.main_page.columns]],
                   ignore_index=True)
    cz = pd.concat([raw.causes, keys.head(20).assign(ts=cut + pd.Timedelta(minutes=20), cause="death", confidence=0.99,
                                                     reason="junk")[["ts", "wiki", "title", "cause", "confidence", "reason"]]],
                   ignore_index=True)
    return replace(raw, edits=e, pageviews=p, top=top, main_page=mp, causes=cz)


# Check 2: build the grid + features twice (clean and junked) and compare every row with t <= cut.
def perturbation_check(raw: RawData, cut, seed: int = 0, hours_before: int = 12) -> dict:
    cut = pd.Timestamp(cut).floor("h")
    rng = np.random.default_rng(seed)
    lo, hi = cut - hours_before * HOUR, cut + 3 * HOUR
    g1 = make_grid(raw, lo, hi)
    bad_raw = _corrupt(raw, cut, rng)
    g2 = make_grid(bad_raw, lo, hi)
    key = ["wiki", "title", "t"]
    a_set = set(map(tuple, g1[g1.t <= cut][key].to_numpy()))
    b_set = set(map(tuple, g2[g2.t <= cut][key].to_numpy()))
    grid_ok = a_set == b_set
    f1 = build(raw, g1, with_labels=False)
    f2 = build(bad_raw, g1, with_labels=False)
    cols = C.BASE_FEATURES + C.CAUSE_FEATURES
    before = (f1.t <= cut).to_numpy()
    diffs = {}
    for c in cols:
        a, b = f1[c].to_numpy(float)[before], f2[c].to_numpy(float)[before]
        n = int((~((a == b) | (np.isnan(a) & np.isnan(b)))).sum())
        if n:
            diffs[c] = n
    # Power check: rows after the cut must change, otherwise the test compared nothing.
    after = ~before
    power = int(sum((f1[c].fillna(-1).to_numpy()[after] != f2[c].fillna(-1).to_numpy()[after]).sum() for c in cols))
    return {"check": f"perturbation@{cut:%Y-%m-%d %H:%M}", "passed": grid_ok and not diffs and power > 0,
            "grid_identical": grid_ok, "leaking_columns": diffs, "rows_compared": int(before.sum()),
            "changed_after_cut": power}


# Check 3: no label or provenance column is allowed to be a model input.
def schema_check(features: list[str]) -> dict:
    bad = [c for c in features if c in LABEL_COLS or c in PROVENANCE or c.startswith(("y_", "size_target"))]
    return {"check": "schema", "passed": not bad, "label_or_provenance_features": bad}


# Check 4: each segment's labels (3 h ahead) end before the next segment starts.
def split_check(splits: dict) -> dict:
    probs = []
    order = [k for k in ("train", "val", "test") if k in splits]
    for a, b in zip(order, order[1:]):
        if True:
            last_label = pd.Timestamp(splits[a][1]) + (max(C.HORIZONS)) * HOUR
            if last_label > pd.Timestamp(splits[b][0]):
                probs.append(f"{a} labels reach {last_label} > {b} starts {splits[b][0]}")
    return {"check": "split_embargo", "passed": not probs, "problems": probs}


# Run every check; two perturbation cuts at seeded random times inside the window.
def run_all(raw: RawData, frame: pd.DataFrame, features: list[str], splits: dict, n_cuts: int = 2,
            seed: int = C.SEED) -> dict:
    rng = np.random.default_rng(seed)
    ts = frame.t.drop_duplicates().sort_values()
    lo, hi = ts.iloc[min(len(ts) - 1, 24)], ts.iloc[max(0, len(ts) - 6)]
    cuts = sorted(lo + (hi - lo) * float(x) for x in rng.random(n_cuts))
    checks = [provenance_check(frame), schema_check(features), split_check(splits)]
    checks += [perturbation_check(raw, c, seed + i) for i, c in enumerate(cuts)]
    return {"passed": all(c["passed"] for c in checks), "checks": checks}


def write_report(rep: dict, path=None):
    path = path or C.ARTIFACTS / "leakcheck.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(rep, indent=2, default=str))
    return path

## 6. Model, evaluation and explanations

The trained bundle (3 calibrated classifiers + 2 size quantile models), the metrics next to persistence, and the `top_reasons` templates.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/model.py
"""Trained bundle: 3 calibrated surge classifiers (h = 1, 2, 3) + 2 size quantile regressors (0.5, 0.9)."""
from __future__ import annotations

import pickle
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from . import config as C
from .explain import top_reasons


# Stand-in calibrator when there are too few positives for isotonic regression.
class Identity:
    def predict(self, x):
        return np.asarray(x)


# Everything forecast_job needs in one pickle: models, calibrators, feature list, threshold, provenance.
@dataclass
class Bundle:
    version: str
    features: list[str]
    classifiers: dict           # h -> LGBMClassifier
    calibrators: dict           # h -> IsotonicRegression (or Identity)
    size_models: dict           # 0.5 / 0.9 -> LGBMRegressor on log1p(peak views in (t, t+3])
    threshold: float            # alert threshold chosen on validation (max F1, h = 2)
    uses_causes: bool
    meta: dict = field(default_factory=dict)

    def predict(self, frame: pd.DataFrame, reasons: bool = False) -> pd.DataFrame:
        """Adds p_1, p_2, p_3 (calibrated), size_q50 / size_q90 (views) and optionally top_reasons."""
        X = frame[self.features].astype(float)
        out = frame.copy()
        for h in C.HORIZONS:
            raw = self.classifiers[h].predict_proba(X)[:, 1]
            out[f"p_{h}"] = np.clip(self.calibrators[h].predict(raw), 0, 1)
        # An onset within 3 h is at least as likely as one within 2 h: enforce that ordering.
        for h in C.HORIZONS[1:]:                                   # P(onset within h) can only grow with h
            out[f"p_{h}"] = np.maximum(out[f"p_{h}"], out[f"p_{h - 1}"])
        # Size models predict log1p(peak views); convert back and keep q90 >= q50.
        q50 = np.expm1(self.size_models[0.5].predict(X))
        q90 = np.expm1(self.size_models[0.9].predict(X))
        out["size_q50"] = np.maximum(q50, 0)
        out["size_q90"] = np.maximum(q90, out["size_q50"])
        if reasons:
            out["top_reasons"] = top_reasons(self.classifiers[2].booster_, X, frame)
        return out

    def save(self, path=None):
        path = path or C.MODEL_PATH
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as fh:
            pickle.dump(self, fh)
        return path

    @staticmethod
    def load(path=None) -> "Bundle":
        path = path or C.MODEL_PATH
        with open(path, "rb") as fh:
            return pickle.load(fh)

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/evaluate.py
"""Forecast metrics (CONTRACT §7): PR-AUC, precision@20 per hour, onset recall, median lead time, Brier,
reliability, always next to the persistence baseline."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss

from . import config as C


# PR-AUC (average precision): the right score for rare events; NaN if a segment has no positives.
def pr_auc(y, p) -> float:
    y = np.asarray(y).astype(int)
    return float(average_precision_score(y, p)) if 0 < y.sum() < len(y) else float("nan")


# Of the 20 articles the radar would show each hour, how many really surged.
def precision_at_k_per_hour(frame: pd.DataFrame, pcol: str, ycol: str, k: int = 20) -> float:
    vals = []
    for _, g in frame.groupby("t"):
        top = g.nlargest(k, pcol)
        vals.append(top[ycol].mean())
    return float(np.mean(vals)) if vals else float("nan")


# Calibration bins for the reliability chart.
def reliability(y, p, bins: int = 10) -> list[dict]:
    y, p = np.asarray(y, float), np.asarray(p, float)
    edges = np.linspace(0, 1, bins + 1)
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (p >= a) & ((p < b) if b < 1 else (p <= b))
        if m.any():
            out.append({"bin": f"{a:.1f}-{b:.1f}", "mean_p": float(p[m].mean()), "frac_pos": float(y[m].mean()),
                        "n": int(m.sum())})
    return out


# Alert threshold = best F1 on out-of-fold predictions (never on test).
def best_threshold(y, p) -> float:
    y, p = np.asarray(y).astype(int), np.asarray(p, float)
    best, thr = -1.0, 0.5
    for t in np.linspace(0.05, 0.95, 19):
        pred = p >= t
        tp = (pred & (y == 1)).sum()
        f1 = 2 * tp / max(1, pred.sum() + y.sum())
        if f1 > best:
            best, thr = f1, float(t)
    return thr


# Hours of warning per true onset at the alert threshold; missed onsets lower recall.
def lead_times(frame: pd.DataFrame, onsets: pd.DataFrame, threshold: float, pcols=("p_1", "p_2", "p_3")) -> dict:
    """For each true onset (wiki, title, hour o): the earliest issue time t with p_h >= threshold where
    o falls in (t, t+h]  (bucket o in [t, t+h-1]). lead = o - t in hours; undetected onsets count as misses."""
    if onsets.empty:
        return {"onsets": 0, "recall": float("nan"), "median_lead_h": float("nan"), "leads": []}
    f = frame[["wiki", "title", "t", *pcols]]
    m = onsets.merge(f, on=["wiki", "title"])
    m["gap"] = (m.hour_utc - m.t) / pd.Timedelta(hours=1)
    hit = np.zeros(len(m), bool)
    for h, pc in zip(C.HORIZONS, pcols):
        hit |= (m.gap >= 0) & (m.gap <= h - 1) & (m[pc] >= threshold)
    m = m[hit]
    lead = m.groupby(["wiki", "title", "hour_utc"]).gap.max()
    n = len(onsets)
    return {"onsets": int(n), "recall": float(len(lead) / n), "median_lead_h": float(lead.median()) if len(lead) else float("nan"),
            "leads": lead.tolist()}


# All test metrics, each next to the persistence baseline (surge in the latest published hour).
def evaluate(frame: pd.DataFrame, onsets: pd.DataFrame, threshold: float) -> dict:
    """frame has p_1..p_3, y_1..y_3 and surge_last_pub (persistence baseline)."""
    out = {}
    for h in C.HORIZONS:
        f = frame[frame[f"y_{h}"].notna()]
        y = f[f"y_{h}"].astype(int)
        out[f"h{h}"] = {
            "n": int(len(f)), "positives": int(y.sum()),
            "pr_auc": pr_auc(y, f[f"p_{h}"]), "pr_auc_persistence": pr_auc(y, f["surge_last_pub"]),
            "brier": float(brier_score_loss(y, f[f"p_{h}"])) if len(f) else float("nan"),
            "brier_persistence": float(brier_score_loss(y, f["surge_last_pub"].clip(0, 1))) if len(f) else float("nan"),
            "precision_at_20": precision_at_k_per_hour(f, f"p_{h}", f"y_{h}"),
            "precision_at_20_persistence": precision_at_k_per_hour(f, "surge_last_pub", f"y_{h}"),
        }
    out["reliability_h2"] = reliability(frame.loc[frame.y_2.notna(), "y_2"], frame.loc[frame.y_2.notna(), "p_2"])
    lt = lead_times(frame, onsets, threshold)
    pers = frame.assign(q1=frame.surge_last_pub, q2=frame.surge_last_pub, q3=frame.surge_last_pub)
    lt_p = lead_times(pers, onsets, 0.5, ("q1", "q2", "q3"))
    out["onsets"] = {"threshold": threshold, **{k: v for k, v in lt.items() if k != "leads"},
                     "persistence_recall": lt_p["recall"], "persistence_median_lead_h": lt_p["median_lead_h"]}
    out["_leads"] = lt["leads"]
    return out

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/explain.py
"""top_reasons: LightGBM pred_contrib -> top 3 positive contributions -> templated English (CONTRACT §3.4)."""
from __future__ import annotations

import math

import numpy as np
import pandas as pd


def _hhmm(row, mins) -> str:
    t = pd.Timestamp(row["t"]) - pd.Timedelta(minutes=float(mins))
    return t.strftime("%H:%M")


def _n(x, word, plural=None):
    x = int(round(x))
    return f"{x} {word if x == 1 else (plural or word + 's')}"


# English templates per feature; None means 'never use this as a reason'.
TEMPLATES = {
    "edits_15m": lambda r, v: f"{_n(v, 'edit')} in the last 15 minutes",
    "edits_30m": lambda r, v: f"{_n(v, 'edit')} in the last 30 minutes",
    "edits_60m": lambda r, v: f"{_n(v, 'edit')} in the last hour",
    "edits_180m": lambda r, v: f"{_n(v, 'edit')} in the last 3 hours",
    "humans_60m": lambda r, v: f"{_n(v, 'human editor')} in the last hour",
    "humans_180m": lambda r, v: f"{_n(v, 'human editor')} in the last 3 hours",
    "accel_30m": lambda r, v: f"Editing speeding up (+{int(v)} edits vs the previous half hour)" if v > 0 else None,
    "bytes_added_60m": lambda r, v: f"{int(v):,} bytes added in the last hour",
    "mins_since_last_edit": lambda r, v: f"Last edit {int(v)} min ago" if v < 60 else None,
    "is_new_page": lambda r, v: f"Page created {r['page_age_h']:.0f} h ago" if v >= 1 and not math.isnan(r["page_age_h"]) else None,
    "n_other_wikis_3h": lambda r, v: f"Edited in {int(v) + 1} languages" if v >= 1 else None,
    "is_tfa_today": lambda r, v: "Today's Featured Article" if v >= 1 else None,
    "mins_since_itn": lambda r, v: f"In 'In the news' since {_hhmm(r, v)}" if not math.isnan(v) else None,
    "in_itn_6h": lambda r, v: "In 'In the news'" if v >= 1 else None,
    "ratio_same_hour_med7": lambda r, v: f"Views {v:.1f}x the usual for this hour" if v >= 1.5 else None,
    "slope_3h": lambda r, v: f"Views rising by {int(v):,}/hour" if v > 0 else None,
    "last_pub_views": lambda r, v: f"{int(v):,} views in the latest published hour",
    "surge_last_pub": lambda r, v: "Already surging in the latest published hour" if v >= 1 else None,
    "revert_share_60m": lambda r, v: None,           # high reverts push p DOWN; never a reason to scale up
    "cause_breaking_news": lambda r, v: "Gemini: looks like breaking news" if v >= 1 else None,
    "cause_death": lambda r, v: "Gemini: a reported death" if v >= 1 else None,
    "cause_sports": lambda r, v: "Gemini: sports result" if v >= 1 else None,
    "cause_scheduled_event": lambda r, v: "Gemini: scheduled event" if v >= 1 else None,
    "cause_entertainment": lambda r, v: "Gemini: entertainment news" if v >= 1 else None,
}


# SHAP-style contributions from LightGBM; the top 3 positive ones become the radar's 'Why?' lines.
def top_reasons(booster, X: pd.DataFrame, ctx: pd.DataFrame, k: int = 3) -> list[list[str]]:
    """X = model inputs; ctx = the same rows with context columns (t, page_age_h) for the templates."""
    contrib = booster.predict(X, pred_contrib=True)[:, :-1]          # last column is the bias term
    cols = list(X.columns)
    ctx = ctx.reset_index(drop=True)
    out = []
    for i in range(len(X)):
        row = {**ctx.iloc[i].to_dict(), **X.iloc[i].to_dict()}
        reasons = []
        for j in np.argsort(-contrib[i]):
            if contrib[i, j] <= 0 or len(reasons) == k:
                break
            f = TEMPLATES.get(cols[j])
            if f is None:
                continue
            try:
                txt = f(row, float(row[cols[j]]))
            except (TypeError, ValueError):
                txt = None
            if txt and txt not in reasons:
                reasons.append(txt)
        out.append(reasons)
    return out

## 7. Systematic training, registry and promotion gate

Rolling-origin CV search, isotonic calibration on pooled out-of-fold predictions, final fit, one look at the test days, Gemini ablation, and the gate that decides the champion. Charts for the slides.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/registry.py
"""Run registry + promotion gate.

Every training run gets its own folder (artifacts/runs/<run_id>/) holding its config, data report, leak report,
CV results, metrics, model and charts. artifacts/registry.json lists every run and names the champion.
Only the champion is copied to artifacts/model.pkl, which is what forecast_job and the backtest load.
"""
from __future__ import annotations

import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

from . import config as C

REGISTRY = C.ARTIFACTS / "registry.json"


# Load the registry, or start an empty one on the very first run.
def load() -> dict:
    if REGISTRY.exists():
        return json.loads(REGISTRY.read_text())
    return {"champion": None, "runs": []}


def save(reg: dict) -> None:
    REGISTRY.parent.mkdir(parents=True, exist_ok=True)
    REGISTRY.write_text(json.dumps(reg, indent=2, default=str))


# A run id sorts by time and is unique enough for a hackathon team (UTC timestamp + data fingerprint).
def new_run_dir(fingerprint: str) -> tuple[str, Path]:
    run_id = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S}-{fingerprint[:6]}"
    d = C.ARTIFACTS / "runs" / run_id
    (d / "charts").mkdir(parents=True, exist_ok=True)
    return run_id, d


def write_json(run_dir: Path, name: str, obj) -> Path:
    p = run_dir / name
    p.write_text(json.dumps(obj, indent=2, default=str))
    return p


def champion_path() -> Path | None:
    reg = load()
    if not reg["champion"]:
        return None
    p = C.ARTIFACTS / "runs" / reg["champion"] / "model.pkl"
    return p if p.exists() else None


# The gate: a new model replaces the champion only if it is leak-free, its data passed validation, it clearly
# beats persistence, and it is not worse than the current champion scored on the SAME new test days.
def gate(metrics: dict, data_status: str, leak_passed: bool, champion_score: float | None, cfg: dict) -> dict:
    g = cfg["gate"]
    score = metrics["test"]["h2"]["pr_auc"]
    pers = metrics["test"]["h2"]["pr_auc_persistence"] or 0.0
    reasons = []
    if not leak_passed:
        reasons.append("leak checks failed")
    if data_status == "FAIL":
        reasons.append("data validation failed")
    if score is None or score - pers < g["min_gain_over_persistence"]:
        reasons.append(f"PR-AUC h2 {score} beats persistence {pers:.3f} by less than {g['min_gain_over_persistence']}")
    if champion_score is not None and score is not None and score < champion_score - g["max_drop_vs_champion"]:
        reasons.append(f"PR-AUC h2 {score:.3f} is below the champion's {champion_score:.3f} on the same test days")
    return {"promote": not reasons, "reasons": reasons, "score": score, "persistence": pers,
            "champion_score": champion_score}


# Record the run; if it passed the gate (or --force), make it the champion and copy its model into place.
def register(run_id: str, run_dir: Path, summary: dict, decision: dict, force: bool = False) -> bool:
    reg = load()
    promoted = decision["promote"] or force
    reg["runs"].append({"run_id": run_id, "created": datetime.now(timezone.utc).isoformat(), **summary,
                        "gate": decision, "promoted": promoted, "forced": force and not decision["promote"]})
    if promoted:
        reg["champion"] = run_id
        shutil.copy(run_dir / "model.pkl", C.MODEL_PATH)
        for png in (run_dir / "charts").glob("*.png"):
            C.CHART_DIR.mkdir(parents=True, exist_ok=True)
            shutil.copy(png, C.CHART_DIR / png.name)
    save(reg)
    return promoted

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/train.py
"""Systematic training on whatever PC1 has loaded into Snowflake (or the local DuckDB).

    flashcrowd-train                         # auto window, configs/train.json, Snowflake if FLASHCROWD_STORE=snowflake
    flashcrowd-train --config configs/train.json --start 2026-10-15 --end 2026-10-24T09:00 --force

Stages (each writes into artifacts/runs/<run_id>/ so every run can be audited and repeated):
  1 config      copy configs/train.json into the run
  2 extract     pull the window from the store (data.load_window)
  3 validate    data checks; FAIL stops the run                                  -> data_report.json
  4 features    grid + as-of features + labels (cached by data fingerprint)
  5 leak gate   provenance + perturbation + schema + split checks; any failure stops the run -> leakcheck.json
  6 search      hyperparameters chosen by rolling-origin CV on the days before the test days -> cv.json
  7 calibrate   isotonic regression on the pooled out-of-fold predictions
  8 fit         final classifiers (h = 1, 2, 3) + size quantiles (0.5, 0.9) on all non-test days
  9 evaluate    test days touched exactly once; persistence baseline next to every number -> metrics.json
 10 ablation    Gemini cause features on vs off (only if CAUSE_LABELS has rows)
 11 gate        promote to champion only if it beats persistence and the current champion -> registry.json
"""
from __future__ import annotations

import argparse
import hashlib
import itertools
import json
import pickle
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression

from . import config as C
from . import registry
from .data import HOUR, load_window
from .evaluate import best_threshold, evaluate, pr_auc
from .features import build, make_grid, onsets
from .leakcheck import run_all
from .model import Bundle, Identity
from .store import Store, open_store
from .validate import onset_check, validate

DEFAULT_CONFIG = C.ROOT / "configs" / "train.json"


# ------------------------------------------------------------------ stage 1-2: config + window
# Read the JSON config; anything not given falls back to the defaults in configs/train.json.
def load_config(path: str | Path | None = None) -> dict:
    return json.loads(Path(path or DEFAULT_CONFIG).read_text())


# The usable window: decisions need 3 h of edits and 7 days of views behind them, and 3 h of views after them
# for the labels. Anything outside is dropped rather than half-computed.
def data_window(store: Store) -> tuple[pd.Timestamp, pd.Timestamp]:
    e = store.query("SELECT MIN(ts) AS a, MAX(ts) AS b FROM EDITS").iloc[0]
    p = store.query("SELECT MIN(hour_utc) AS a, MAX(hour_utc) AS b FROM PAGEVIEWS_HOURLY").iloc[0]
    start = max(pd.Timestamp(e.a).ceil("h") + 3 * HOUR, pd.Timestamp(p.a) + pd.Timedelta(days=7) + 3 * HOUR)
    end = min(pd.Timestamp(e.b).floor("h"), pd.Timestamp(p.b) - 3 * HOUR)
    return start, end


# ------------------------------------------------------------------ stage 6: splits
# Test = the last `test_days` days. The CV folds walk backwards one day at a time before that: fold k validates
# on day N-test-k and trains on every earlier day. A 3 h embargo before each validation/test day means no
# training label (which looks 3 h ahead) can see the held-out day.
def make_splits(ts: pd.Series, cfg: dict) -> dict:
    s = cfg["splits"]
    days = sorted(pd.to_datetime(ts).dt.normalize().unique())
    n_test, k = s["test_days"], s["cv_folds"]
    if len(days) < n_test + k + s["min_train_days"]:
        raise ValueError(f"need {n_test + k + s['min_train_days']} days, have {len(days)}")
    emb = s["embargo_hours"] * HOUR
    test0 = pd.Timestamp(days[-n_test])
    folds = []
    for i in range(1, k + 1):
        v0 = pd.Timestamp(days[-n_test - i])
        folds.append({"train": (pd.Timestamp(ts.min()), v0 - emb), "val": (v0, v0 + pd.Timedelta(days=1) - emb)})
    return {"folds": folds[::-1], "dev": (pd.Timestamp(ts.min()), test0 - emb), "test": (test0, pd.Timestamp(ts.max()))}


def seg(frame: pd.DataFrame, rng) -> pd.DataFrame:
    return frame[(frame.t >= rng[0]) & (frame.t <= rng[1])]


# ------------------------------------------------------------------ model factories
# Classifier with a given hyperparameter set; `metric` drives early stopping on the validation day.
def make_clf(params: dict, n_estimators: int) -> lgb.LGBMClassifier:
    return lgb.LGBMClassifier(objective="binary", n_estimators=n_estimators, subsample=0.8, subsample_freq=1,
                              metric="average_precision", random_state=C.SEED, verbose=-1, **params)


def make_qreg(alpha: float, cfg: dict, n_estimators: int = 1000) -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(objective="quantile", alpha=alpha, metric="quantile", n_estimators=n_estimators,
                             random_state=C.SEED, verbose=-1, **cfg["size_model"])


# ------------------------------------------------------------------ stage 6: hyperparameter search
# Candidate set = the roadmap's default + a seeded random sample of the grid in the config (so reruns pick the
# same trials). Each trial is scored by mean PR-AUC (h = 2) over the CV folds.
def candidates(cfg: dict) -> list[dict]:
    sp = cfg["search"]["space"]
    grid = [dict(zip(sp, vals)) for vals in itertools.product(*sp.values())]
    rng = np.random.default_rng(cfg["search"]["seed"])
    pick = rng.choice(len(grid), size=min(cfg["search"]["n_trials"], len(grid)), replace=False)
    out = [cfg["search"]["roadmap_default"]] + [grid[i] for i in pick]
    uniq = []
    for p in out:
        if p not in uniq:
            uniq.append(p)
    return uniq


# Fit one fold with early stopping and return (validation raw scores, best iteration).
def _fold_fit(tr, va, ycol, feats, params, cfg):
    X = lambda f: f[feats].astype(float)
    ya, yb = tr[ycol].astype(int), va[ycol].astype(int)
    m = make_clf(params, cfg["search"]["max_estimators"])
    if yb.sum() > 0:
        m.fit(X(tr), ya, eval_set=[(X(va), yb)], categorical_feature=C.CATEGORICAL,
              callbacks=[lgb.early_stopping(cfg["search"]["early_stopping_rounds"], verbose=False)])
        best = int(m.best_iteration_ or cfg["search"]["max_estimators"])
    else:
        m.set_params(n_estimators=300).fit(X(tr), ya, categorical_feature=C.CATEGORICAL)
        best = 300
    return m.predict_proba(X(va), num_iteration=best)[:, 1], best


def search(frame: pd.DataFrame, splits: dict, feats: list[str], cfg: dict) -> dict:
    trials = []
    for params in candidates(cfg):
        scores, iters = [], []
        for f in splits["folds"]:
            tr, va = seg(frame, f["train"]), seg(frame, f["val"])
            p, best = _fold_fit(tr, va, "y_2", feats, params, cfg)
            scores.append(pr_auc(va.y_2, p))
            iters.append(best)
        trials.append({"params": params, "fold_pr_auc_h2": scores, "mean_pr_auc_h2": float(np.nanmean(scores)),
                       "best_iterations": iters})
    trials.sort(key=lambda r: -np.nan_to_num(r["mean_pr_auc_h2"], nan=-1))
    best = trials[0]
    # Final tree count: median early-stopping point, scaled up a little because the final fit sees more days.
    n_final = int(np.median(best["best_iterations"]) * 1.15) + 1
    return {"best_params": best["params"], "n_estimators": n_final, "trials": trials}


# ------------------------------------------------------------------ stage 7: calibration on pooled OOF predictions
# Every fold model predicts its own validation day; pooling the folds gives 3x the positives a single
# validation day would, which is what isotonic regression needs to be stable.
def oof_calibrators(frame, splits, feats, params, cfg) -> tuple[dict, dict, float]:
    cals, oof_info = {}, {}
    oof_p2, oof_y2 = [], []
    for h in C.HORIZONS:
        ps, ys = [], []
        for f in splits["folds"]:
            tr, va = seg(frame, f["train"]), seg(frame, f["val"])
            p, _ = _fold_fit(tr, va, f"y_{h}", feats, params, cfg)
            ps.append(p)
            ys.append(va[f"y_{h}"].astype(int).to_numpy())
        p, y = np.concatenate(ps), np.concatenate(ys)
        if y.sum() >= cfg["calibration"]["min_oof_positives"]:
            cals[h] = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y)
        else:
            cals[h] = Identity()
        oof_info[h] = {"oof_rows": int(len(y)), "oof_positives": int(y.sum()), "oof_pr_auc": pr_auc(y, p),
                       "calibrated": not isinstance(cals[h], Identity)}
        if h == 2:
            oof_p2, oof_y2 = cals[h].predict(p), y
    # Alert threshold for the scorecard / radar: best F1 on the calibrated out-of-fold h=2 predictions.
    thr = best_threshold(oof_y2, oof_p2)
    return cals, oof_info, thr


# ------------------------------------------------------------------ stage 8: final fit
# Final models train on ALL non-test days with the chosen settings; the test days are still untouched.
def final_fit(frame, splits, feats, search_res, cals, thr, cfg, version) -> Bundle:
    dev = seg(frame, splits["dev"])
    X = dev[feats].astype(float)
    clfs = {h: make_clf(search_res["best_params"], search_res["n_estimators"])
            .fit(X, dev[f"y_{h}"].astype(int), categorical_feature=C.CATEGORICAL) for h in C.HORIZONS}
    # Size models: early-stop on the last CV validation day, then refit on all dev days with that tree count.
    last = splits["folds"][-1]
    tr, va = seg(frame, last["train"]), seg(frame, last["val"])
    sizes = {}
    for q in (0.5, 0.9):
        m = make_qreg(q, cfg).fit(tr[feats].astype(float), tr.size_target, eval_set=[(va[feats].astype(float), va.size_target)],
                                  categorical_feature=C.CATEGORICAL, callbacks=[lgb.early_stopping(50, verbose=False)])
        n = int((m.best_iteration_ or 300) * 1.15) + 1
        sizes[q] = make_qreg(q, cfg, n).fit(X, dev.size_target, categorical_feature=C.CATEGORICAL)
    return Bundle(version=version, features=feats, classifiers=clfs, calibrators=cals, size_models=sizes,
                  threshold=thr, uses_causes=any(f.startswith("cause_") for f in feats))


# One complete model (search -> calibrate -> fit) for a given feature list.
def build_model(frame, splits, feats, cfg, version):
    sres = search(frame, splits, feats, cfg)
    cals, oof, thr = oof_calibrators(frame, splits, feats, sres["best_params"], cfg)
    bundle = final_fit(frame, splits, feats, sres, cals, thr, cfg, version)
    return bundle, sres, oof


def _test_onsets(raw, test, splits):
    ons = onsets(raw, test[["wiki", "title"]])
    return ons[(ons.hour_utc >= splits["test"][0]) & (ons.hour_utc <= splits["test"][1] + 2 * HOUR)]


def _jsonable(x):
    return json.loads(json.dumps(x, default=lambda v: None if isinstance(v, float) and np.isnan(v) else str(v)))


# ------------------------------------------------------------------ shared by train and refine
# Fingerprint of the extracted window: same data -> same cache file and same run id suffix.
def fingerprint(raw, start, end) -> str:
    return hashlib.sha256(f"{start}{end}{len(raw.edits)}{len(raw.pageviews)}{int(raw.pageviews.views.sum())}"
                          f"{len(raw.causes)}{len(raw.main_page)}".encode()).hexdigest()[:12]


# Grid + features + labels for the window, cached by fingerprint; rows without a full 3 h label window dropped.
def features_cached(raw, start, end, fp: str, use_cache: bool = True) -> pd.DataFrame:
    cache = C.CACHE_DIR / f"frame_{fp}.parquet"
    if use_cache and cache.exists():
        frame = pd.read_parquet(cache)
    else:
        frame = build(raw, make_grid(raw, start, end))
        cache.parent.mkdir(parents=True, exist_ok=True)
        frame.to_parquet(cache, index=False)
    return frame[frame.y_3.notna()].reset_index(drop=True)


# Model inputs: every base feature except any the config drops (refine writes `drop_features`).
def model_features(cfg: dict) -> list[str]:
    drop = set(cfg.get("drop_features", []))
    return [f for f in C.BASE_FEATURES if f not in drop]


# ------------------------------------------------------------------ the whole pipeline
def run(store: Store, cfg: dict, start=None, end=None, force: bool = False, use_cache: bool = True,
        charts: bool = True, log=print) -> dict:
    # Stage 2: window + extraction.
    start = pd.Timestamp(start or cfg["window"]["start"]) if (start or cfg["window"]["start"]) else None
    end = pd.Timestamp(end or cfg["window"]["end"]) if (end or cfg["window"]["end"]) else None
    if start is None or end is None:
        a, b = data_window(store)
        start, end = start or a, end or b
    log(f"[1/11] window {start} -> {end}")
    raw = load_window(store, start, end)
    fp = fingerprint(raw, start, end)
    run_id, run_dir = registry.new_run_dir(fp)
    registry.write_json(run_dir, "config.json", cfg)
    log(f"[2/11] extracted {len(raw.edits):,} edits, {len(raw.pageviews):,} page-view rows -> run {run_id}")

    # Stage 3: data validation; FAIL stops here.
    report = validate(raw, cfg)
    registry.write_json(run_dir, "data_report.json", report)
    for c in report["checks"]:
        if c["status"] != "PASS":
            log(f"      {c['status']}: {c['check']}: {c['detail']}")
    if report["status"] == "FAIL":
        raise RuntimeError(f"data validation failed, see {run_dir / 'data_report.json'}")
    log(f"[3/11] data validation {report['status']}")

    # Stage 4: features + labels, cached by data fingerprint so a re-run with new hyperparameters is fast.
    frame = features_cached(raw, start, end, fp, use_cache)
    log(f"[4/11] {len(frame):,} decision rows, {int(frame.y_2.sum())} with an onset within 2 h")

    # Stage 5: leak gate.
    splits = make_splits(frame.t, cfg)
    flat = {"train": [str(splits["dev"][0]), str(splits["dev"][1])], "test": [str(splits["test"][0]), str(splits["test"][1])]}
    leak = run_all(raw, frame, C.BASE_FEATURES + C.CAUSE_FEATURES, flat)
    registry.write_json(run_dir, "leakcheck.json", leak)
    if not leak["passed"]:
        raise RuntimeError(f"leak checks failed; refusing to train. See {run_dir / 'leakcheck.json'}")
    log("[5/11] leak checks passed")

    # Stages 6-8: search, calibrate, fit (base features).
    version = f"fc-{run_id}"
    base_feats = model_features(cfg)
    bundle, sres, oof = build_model(frame, splits, base_feats, cfg, version)
    log(f"[6/11] best params {sres['best_params']} | CV PR-AUC h2 {sres['trials'][0]['mean_pr_auc_h2']:.3f} "
        f"(roadmap default scored {next(t['mean_pr_auc_h2'] for t in sres['trials'] if t['params'] == cfg['search']['roadmap_default']):.3f})")
    log(f"[7/11] calibration {[oof[h]['calibrated'] for h in C.HORIZONS]} | alert threshold {bundle.threshold:.2f}")
    log(f"[8/11] final fit: {sres['n_estimators']} trees on {len(seg(frame, splits['dev'])):,} rows")

    # Stage 10 (before evaluation so the chosen model is the one evaluated): Gemini cause features on/off.
    test = seg(frame, splits["test"])
    ablation = None
    has_causes = frame[C.CAUSE_FEATURES[:-1]].to_numpy().sum() > 0
    if has_causes and cfg["causes"] != "off":
        b2, s2, oof2 = build_model(frame, splits, base_feats + C.CAUSE_FEATURES, cfg, version + "-causes")
        cv_off, cv_on = sres["trials"][0]["mean_pr_auc_h2"], s2["trials"][0]["mean_pr_auc_h2"]
        t_off = pr_auc(test.y_2, bundle.predict(test).p_2)
        t_on = pr_auc(test.y_2, b2.predict(test).p_2)
        ablation = {"cv_pr_auc_h2": {"off": cv_off, "on": cv_on}, "test_pr_auc_h2": {"off": t_off, "on": t_on},
                    "lift_test_h2": t_on - t_off, "chosen": "on" if (cfg["causes"] == "on" or cv_on > cv_off) else "off"}
        if ablation["chosen"] == "on":                       # chosen on CV, never on test
            bundle, sres, oof = b2, s2, oof2
        log(f"[10/11] Gemini ablation: test PR-AUC h2 off {t_off:.3f} / on {t_on:.3f} -> using {ablation['chosen']}")
    else:
        log("[10/11] Gemini ablation skipped (no CAUSE_LABELS rows)")

    # Stage 9: evaluate on the untouched test days.
    pred = bundle.predict(test)
    ons = _test_onsets(raw, test, splits)
    ev = evaluate(pred, ons, bundle.threshold)
    leads = ev.pop("_leads", [])
    checks = [onset_check(ev["onsets"]["onsets"], cfg)]
    metrics = {"run_id": run_id, "model_version": bundle.version, "window": [str(start), str(end)],
               "data_fingerprint": fp, "splits": {"folds": [{k: [str(a), str(b)] for k, (a, b) in f.items()} for f in splits["folds"]],
                                                  "dev": [str(x) for x in splits["dev"]], "test": [str(x) for x in splits["test"]]},
               "rows": int(len(frame)), "test": ev, "cv": {"best_params": sres["best_params"], "n_estimators": sres["n_estimators"],
                                                           "oof": oof}, "gemini_ablation": ablation,
               "data_validation": report["status"], "post_checks": checks}
    metrics = _jsonable(metrics)
    registry.write_json(run_dir, "cv.json", _jsonable(sres))
    registry.write_json(run_dir, "metrics.json", metrics)
    m2 = metrics["test"]["h2"]
    log(f"[9/11] test PR-AUC h2 {m2['pr_auc']:.3f} vs persistence {m2['pr_auc_persistence']:.3f} | "
        f"onset recall {ev['onsets']['recall']:.2f}, median lead {ev['onsets']['median_lead_h']} h")
    if checks[0]["status"] != "PASS":
        log(f"      WARN: {checks[0]['detail']}")

    # Save the model into the run folder.
    bundle.meta = {"run_id": run_id, "window": metrics["window"], "splits": metrics["splits"], "data_fingerprint": fp,
                   "contracts_version": C.CONTRACTS_VERSION, "leads": leads}
    with open(run_dir / "model.pkl", "wb") as fh:
        pickle.dump(bundle, fh)
    if charts:
        from .charts import training_charts
        training_charts(bundle, metrics, pred, raw, ons, run_dir / "charts")

    # Stage 11: gate. The champion is re-scored on THIS run's test days so the comparison is like for like.
    champ = registry.champion_path()
    champ_score = None
    if champ is not None:
        cb = Bundle.load(champ)
        if set(cb.features) <= set(test.columns):
            champ_score = pr_auc(test.y_2, cb.predict(test).p_2)
    decision = registry.gate(metrics, report["status"], leak["passed"], champ_score, cfg)
    promoted = registry.register(run_id, run_dir, {"window": metrics["window"], "model_version": bundle.version,
                                                   "pr_auc_h2": m2["pr_auc"], "persistence_h2": m2["pr_auc_persistence"]},
                                 decision, force)
    log(f"[11/11] gate: {'PROMOTED to champion' if promoted else 'not promoted'}"
        + ("" if decision["promote"] else f" ({'; '.join(decision['reasons'])})") + (" [forced]" if force and not decision["promote"] else ""))
    return {"run_id": run_id, "run_dir": str(run_dir), "metrics": metrics, "promoted": promoted, "bundle": bundle,
            "pred": pred, "raw": raw, "onsets": ons}


def main(argv=None):
    ap = argparse.ArgumentParser(description="Systematic FLASHCROWD training run")
    ap.add_argument("--config", default=str(DEFAULT_CONFIG))
    ap.add_argument("--start")
    ap.add_argument("--end")
    ap.add_argument("--force", action="store_true", help="promote even if the gate says no")
    ap.add_argument("--no-cache", action="store_true")
    ap.add_argument("--no-charts", action="store_true")
    a = ap.parse_args(argv)
    run(open_store(), load_config(a.config), a.start, a.end, a.force, not a.no_cache, not a.no_charts)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/charts.py
"""Charts for the slides and PC2's "How it works" screen (CONTRACT §7 P3): dark background, PNG at 2x."""
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config as C

TEAL, CORAL, GREY, GOLD = "#3fd0c9", "#ff6b6b", "#9aa4b2", "#f4c95d"
COLORS = {"autopilot": TEAL, "persistence": GOLD, "reactive": CORAL, "scheduled": GREY, "oracle": "#ffffff"}


# One place for the house style so every PNG matches.
def _fig(w=8, h=4.6):
    plt.style.use("dark_background")
    fig, ax = plt.subplots(figsize=(w, h))
    ax.grid(alpha=0.18)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    return fig, ax


def _save(fig, out_dir: Path, name: str) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    p = out_dir / name
    fig.tight_layout()
    fig.savefig(p, dpi=200)
    plt.close(fig)
    return p


# Reliability: are the calibrated probabilities honest? Points on the diagonal = yes.
def reliability_chart(metrics: dict, out_dir: Path) -> Path:
    rel = metrics["test"]["reliability_h2"]
    fig, ax = _fig(6, 5)
    ax.plot([0, 1], [0, 1], "--", color=GREY, lw=1)
    if rel:
        ax.plot([r["mean_p"] for r in rel], [r["frac_pos"] for r in rel], "o-", color=TEAL)
        for r in rel:
            ax.annotate(str(r["n"]), (r["mean_p"], r["frac_pos"]), fontsize=7, color=GREY, xytext=(4, 4),
                        textcoords="offset points")
    ax.set(xlabel="Predicted chance of a surge within 2 h", ylabel="Share that surged", xlim=(0, 1), ylim=(0, 1),
           title="Reliability on held-out days (labels = rows per bin)")
    return _save(fig, out_dir, "reliability.png")


# Lead time: how many hours of warning each detected onset got.
def lead_time_chart(leads: list, metrics: dict, out_dir: Path) -> Path:
    fig, ax = _fig(6, 4)
    if leads:
        ax.hist(leads, bins=np.arange(-0.5, 3.5, 1), color=TEAL, rwidth=0.8)
    o = metrics["test"]["onsets"]
    ax.set(xlabel="Hours of warning before the surge hour", ylabel="Surges",
           title=f"Lead time: median {o['median_lead_h']} h, {o['recall']:.0%} of {o['onsets']} surges caught")
    return _save(fig, out_dir, "lead_time.png")


# Feature importance (gain) of the 2-hour classifier: what the model actually leans on.
def importance_chart(bundle, out_dir: Path) -> Path:
    b = bundle.classifiers[2].booster_
    imp = pd.Series(b.feature_importance("gain"), index=b.feature_name()).sort_values().tail(15)
    fig, ax = _fig(7, 5)
    ax.barh(imp.index, imp.values, color=TEAL)
    ax.set(xlabel="Total gain", title="What drives the 2-hour surge forecast")
    return _save(fig, out_dir, "feature_importance.png")


# One annotated surge: human edits (line) running ahead of page views (bars).
def surge_example_chart(raw, onsets: pd.DataFrame, pred: pd.DataFrame, out_dir: Path) -> Path | None:
    if onsets.empty:
        return None
    o = onsets.sort_values("views", ascending=False).iloc[0]
    w0, w1 = o.hour_utc - pd.Timedelta(hours=8), o.hour_utc + pd.Timedelta(hours=6)
    pv = raw.pageviews[(raw.pageviews.wiki == o.wiki) & (raw.pageviews.title == o.title)]
    pv = pv[(pv.hour_utc >= w0) & (pv.hour_utc <= w1)]
    ed = raw.edits[(raw.edits.wiki == o.wiki) & (raw.edits.title == o.title) & ~raw.edits.is_bot]
    ed = ed[(ed.ts >= w0) & (ed.ts <= w1)].set_index("ts").resample("15min").size()
    fig, ax = _fig(9, 4.6)
    ax.bar(pv.hour_utc + pd.Timedelta(minutes=30), pv.views, width=0.035, color=GREY, label="page views / hour")
    ax.set_ylabel("Views per hour")
    ax2 = ax.twinx()
    ax2.plot(ed.index, ed.values, color=TEAL, lw=2, label="human edits / 15 min")
    ax2.set_ylabel("Edits per 15 min", color=TEAL)
    p = pred[(pred.wiki == o.wiki) & (pred.title == o.title) & (pred.p_2 >= pred.attrs.get("thr", 0.5))]
    first = p.t.min() if len(p) else None
    if first is not None and not pd.isna(first):
        ax.axvline(first, color=GOLD, ls="--", lw=1.5)
        ax.annotate(f"forecast fires {first:%H:%M}", (first, ax.get_ylim()[1] * 0.9), color=GOLD, fontsize=9,
                    xytext=(5, 0), textcoords="offset points")
    ax.axvline(o.hour_utc, color=CORAL, lw=1.5)
    ax.annotate("surge hour", (o.hour_utc, ax.get_ylim()[1] * 0.75), color=CORAL, fontsize=9, xytext=(5, 0),
                textcoords="offset points")
    ax.set_title(f"{o.title.replace('_', ' ')} ({o.wiki}): editors move before readers")
    return _save(fig, out_dir, "surge_example.png")


def training_charts(bundle, metrics, pred, raw, onsets, out_dir: Path | None = None) -> list[Path]:
    out_dir = Path(out_dir or C.CHART_DIR)
    pred = pred.copy()
    pred.attrs["thr"] = bundle.threshold
    paths = [reliability_chart(metrics, out_dir), lead_time_chart(bundle.meta.get("leads", []) if bundle.meta else
                                                                  [], metrics, out_dir),
             importance_chart(bundle, out_dir), surge_example_chart(raw, onsets, pred, out_dir)]
    return [p for p in paths if p]


# Pareto: server-hours (cost) vs % requests dropped (failures); lower-left wins.
def pareto_chart(points: list[dict], headline: str, out_dir: Path) -> Path:
    fig, ax = _fig(8, 5)
    df = pd.DataFrame(points)
    for fam, g in df.groupby("family"):
        g = g.sort_values("server_hours")
        style = "o-" if len(g) > 1 else "D"
        ax.plot(g.server_hours, 100 - g.served_pct, style, color=COLORS.get(fam, GREY), label=fam, ms=6)
        for r in g.itertuples():
            if getattr(r, "kappa", None) is not None and not pd.isna(r.kappa):
                ax.annotate(f"k={r.kappa:g}", (r.server_hours, 100 - r.served_pct), fontsize=7, color=GREY,
                            xytext=(4, 3), textcoords="offset points")
    # Linear axis with a little room below zero so the "dropped nothing" points sit visibly on the floor.
    top = max(0.05, float((100 - df.served_pct).max()) * 1.15)
    ax.set_ylim(-top * 0.05, top)
    ax.set(xlabel="Server-hours (cost)", ylabel="% requests dropped", title=headline)
    ax.legend(frameon=False)
    return _save(fig, out_dir, "pareto.png")

## 8. Refinement

Diagnose the champion (by wiki, by time of day, worst errors, feature-group ablation, thresholds, drift) and retrain with the fixes; the gate still decides.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/refine.py
"""Model refinement: diagnose the champion, then retrain with what the diagnosis found.

    flashcrowd refine            diagnosis only -> artifacts/runs/<champion>/refine.md + refine.json
    flashcrowd refine --apply    diagnosis, then a new training run with the recommended config (gate still decides)

What it looks at:
  1 where the model is weak     PR-AUC by wiki and by time of day; calibration error
  2 what it gets wrong          the worst false alarms and misses with the evidence the model saw
  3 which inputs earn their keep  CV PR-AUC with each feature group removed (same settings, same folds)
  4 alert threshold trade-off   precision / recall at each threshold on the test days
  5 drift                       train vs test feature shift (PSI); live hit rate vs test precision (scorecard)
Recommendations it can apply: drop feature groups whose removal raises CV PR-AUC, widen the hyperparameter
search, lower the surge multiple if there are too few onsets. Nothing is applied without --apply, and a refined
model replaces the champion only if it passes the same promotion gate as any other run.
"""
from __future__ import annotations

import argparse
import copy
import json

import numpy as np
import pandas as pd

from . import config as C
from . import registry
from .data import load_window
from .evaluate import pr_auc, reliability
from .model import Bundle
from .scorecard import build as build_scorecard
from .store import Store, open_store
from .train import _fold_fit, features_cached, fingerprint, load_config, make_splits, model_features, run, seg

GROUPS = {"edits": C.EDIT_FEATURES, "page lifecycle": C.PAGE_FEATURES, "page views": C.VIEW_FEATURES,
          "main page": C.MAIN_PAGE_FEATURES, "time": [f for f in C.TIME_FEATURES if f != "wiki_id"],
          "cross-language": C.XLANG_FEATURES}
SHOW = ["edits_60m", "humans_60m", "n_other_wikis_3h", "ratio_same_hour_med7", "is_tfa_today", "in_itn_6h",
        "revert_share_60m", "bot_share_60m"]


# Population stability index between two samples of one feature (> 0.25 = a real shift).
def psi(a: np.ndarray, b: np.ndarray, bins: int = 10) -> float:
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 50 or len(b) < 50:
        return float("nan")
    edges = np.unique(np.quantile(a, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    pa = np.histogram(np.clip(a, edges[0], edges[-1]), edges)[0] / len(a) + 1e-4
    pb = np.histogram(np.clip(b, edges[0], edges[-1]), edges)[0] / len(b) + 1e-4
    return float(np.sum((pa - pb) * np.log(pa / pb)))


def _cv_score(frame, splits, feats, params, cfg) -> float:
    scores = []
    for f in splits["folds"]:
        tr, va = seg(frame, f["train"]), seg(frame, f["val"])
        p, _ = _fold_fit(tr, va, "y_2", feats, params, cfg)
        scores.append(pr_auc(va.y_2, p))
    return float(np.nanmean(scores))


def diagnose(store: Store, log=print) -> dict:
    # Load the champion and rebuild exactly the frame it was trained on (same window -> same cache).
    reg = registry.load()
    if not reg.get("champion"):
        raise RuntimeError("no champion yet: run `flashcrowd train` first")
    run_dir = C.ARTIFACTS / "runs" / reg["champion"]
    cfg = json.loads((run_dir / "config.json").read_text())
    metrics = json.loads((run_dir / "metrics.json").read_text())
    cv = json.loads((run_dir / "cv.json").read_text())
    bundle = Bundle.load(run_dir / "model.pkl")
    start, end = (pd.Timestamp(x) for x in metrics["window"])
    raw = load_window(store, start, end)
    frame = features_cached(raw, start, end, fingerprint(raw, start, end))
    splits = make_splits(frame.t, cfg)
    test = seg(frame, splits["test"])
    pred = bundle.predict(test)
    log(f"[1/5] champion {reg['champion']}: {len(test):,} test rows, {int(test.y_2.sum())} onsets within 2 h")

    # 1. Where it is weak: per wiki and per 6-hour block of the day, plus expected calibration error.
    by_wiki = []
    for w, g in pred.groupby("wiki"):
        by_wiki.append({"wiki": w, "rows": int(len(g)), "onsets": int(g.y_2.sum()), "pr_auc": pr_auc(g.y_2, g.p_2)})
    by_hour = []
    for blk, g in pred.groupby((pred.hour_utc // 6).astype(int)):
        by_hour.append({"hours_utc": f"{blk * 6:02d}-{blk * 6 + 5:02d}", "onsets": int(g.y_2.sum()), "pr_auc": pr_auc(g.y_2, g.p_2)})
    rel = reliability(pred.y_2, pred.p_2)
    ece = float(sum(abs(b["mean_p"] - b["frac_pos"]) * b["n"] for b in rel) / max(1, sum(b["n"] for b in rel)))

    # 2. What it gets wrong, with the evidence it saw (top false alarms and misses).
    cols = ["wiki", "title", "t", "p_2"] + SHOW
    fa = pred[(pred.y_2 == 0)].nlargest(10, "p_2")[cols]
    miss = pred[(pred.y_2 == 1)].nsmallest(10, "p_2")[cols]
    patterns = []
    silent = float((miss.edits_60m == 0).mean()) if len(miss) else 0.0
    if silent >= 0.3:
        patterns.append(f"{silent:.0%} of the worst misses had no human edits in the hour before: silent surges, which "
                        "only Main Page or page-view features can catch")
    wars = float((fa.revert_share_60m >= 0.3).mean()) if len(fa) else 0.0
    if wars >= 0.3:
        patterns.append(f"{wars:.0%} of the worst false alarms had >= 30% reverts: edit wars that look like news")
    bots = float((fa.bot_share_60m >= 0.5).mean()) if len(fa) else 0.0
    if bots >= 0.3:
        patterns.append(f"{bots:.0%} of the worst false alarms were mostly bot edits")
    if not patterns:
        patterns.append("No single pattern explains most of the worst errors")
    log(f"[2/5] ECE {ece:.3f}; weakest wiki: "
        f"{min((w for w in by_wiki if w['onsets'] >= 3), key=lambda w: w['pr_auc'], default={'wiki': '-'})['wiki']}")

    # 3. Feature-group ablation with the champion's own settings and folds.
    params = cv["best_params"]
    feats = list(bundle.features)
    base = _cv_score(frame, splits, feats, params, cfg)
    ablation = []
    for name, group in GROUPS.items():
        keep = [f for f in feats if f not in group]
        if len(keep) == len(feats):
            continue
        sc = _cv_score(frame, splits, keep, params, cfg)
        ablation.append({"group": name, "features": [f for f in group if f in feats], "cv_pr_auc_without": sc,
                         "change": sc - base})
    ablation.sort(key=lambda a: a["change"])
    log(f"[3/5] CV PR-AUC {base:.3f}; removing each group: "
        + ", ".join(f"{a['group']} {a['change']:+.3f}" for a in ablation))

    # 4. Threshold trade-off on the test days (information for the alert setting; the model keeps its own).
    thr = []
    for t in (0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6):
        pos = pred.p_2 >= t
        tp = int((pos & (pred.y_2 == 1)).sum())
        thr.append({"threshold": t, "alerts": int(pos.sum()), "precision": tp / max(1, int(pos.sum())),
                    "recall": tp / max(1, int(pred.y_2.sum()))})

    # 5. Drift: did the test days look like the training days? (Time features and page age grow by definition.) And are live forecasts as good as the test?
    dev = seg(frame, splits["dev"])
    shift = sorted(({"feature": f, "psi": psi(dev[f].to_numpy(float), test[f].to_numpy(float))} for f in feats
                    if f not in ("hour_utc", "dow", "local_hour", "wiki_id", "page_age_h")), key=lambda d: -np.nan_to_num(d["psi"]))[:8]
    live = None
    if pd.Timestamp(end) < pd.Timestamp.utcnow().tz_localize(None):
        sc = build_scorecard(store, end, bundle.threshold)
        if sc["checked"]:
            prec_live = sc["hits"] / max(1, sc["hits"] + sc["false_alarms"])
            prec_test = next(x["precision"] for x in thr if x["threshold"] >= bundle.threshold)
            live = {"checked": sc["checked"], "hits": sc["hits"], "false_alarms": sc["false_alarms"],
                    "misses": sc["misses"], "live_precision": prec_live, "test_precision_at_threshold": prec_test}
    log(f"[4/5] biggest train->test shift: {shift[0]['feature']} PSI {shift[0]['psi']:.2f}" if shift else "[4/5] no shift data")

    # Recommendations, each with the evidence behind it.
    recs, refined = [], copy.deepcopy(cfg)
    harmful = [a for a in ablation if a["change"] >= 0.005]
    if harmful:
        drop = sorted({f for a in harmful for f in a["features"]})
        refined["drop_features"] = sorted(set(refined.get("drop_features", [])) | set(drop))
        recs.append(f"Drop {', '.join(a['group'] for a in harmful)}: CV PR-AUC rises without them "
                    f"({', '.join(f'{a['change']:+.3f}' for a in harmful)}).")
    gap = cv["trials"][0]["mean_pr_auc_h2"] - cv["trials"][min(2, len(cv["trials"]) - 1)]["mean_pr_auc_h2"]
    if gap > 0.02 or len(cv["trials"]) <= 11:
        refined["search"]["n_trials"] = max(cfg["search"]["n_trials"] * 2, 20)
        why = (f"the best setting beats the third-best by {gap:.3f} CV PR-AUC, so better ones may exist" if gap > 0.02
               else f"only {len(cv['trials'])} settings were tried")
        recs.append(f"Widen the search to {refined['search']['n_trials']} trials: {why}.")
    if int(test.y_2.sum()) < cfg["validation"]["min_test_onsets"]:
        recs.append(f"Only {int(test.y_2.sum())} onsets in the test days: add a day of data, or lower SURGE_MULT from "
                    f"{C.SURGE_MULT:g} to 4 (config.py, and re-run `flashcrowd setup` so PC1's view matches).")
    if ece > 0.05:
        recs.append(f"Calibration error {ece:.3f} is high: more validation days (cv_folds) give isotonic more positives.")
    if live and live["live_precision"] < 0.5 * live["test_precision_at_threshold"]:
        recs.append(f"Live precision {live['live_precision']:.0%} is under half the test precision: retrain on the "
                    "latest days (`flashcrowd train`) and check PC1's loaders for a change.")
    if any(d["psi"] > 0.25 for d in shift if not np.isnan(d["psi"])):
        recs.append("Some inputs shifted a lot between training and test days (PSI > 0.25): retrain nightly.")
    if not recs:
        recs.append("No change needed: keep the champion and retrain nightly as new days arrive.")
    log(f"[5/5] {len(recs)} recommendation(s)")

    out = {"champion": reg["champion"], "model_version": bundle.version, "cv_pr_auc": base,
           "test": metrics["test"]["h2"], "by_wiki": by_wiki, "by_hour": by_hour, "ece": ece, "patterns": patterns,
           "worst_false_alarms": fa.assign(t=fa.t.astype(str)).to_dict("records"),
           "worst_misses": miss.assign(t=miss.t.astype(str)).to_dict("records"), "ablation": ablation,
           "thresholds": thr, "shift": shift, "live": live, "recommendations": recs, "refined_config": refined}
    (run_dir / "refine.json").write_text(json.dumps(out, indent=1, default=str))
    (run_dir / "refine.md").write_text(_report(out, bundle))
    return out


def _f(x, d=3):
    return "-" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{d}f}"


# Human-readable report for the team (and the slides' "what we learned" bullet).
def _report(o: dict, bundle) -> str:
    L = [f"# Refinement report: {o['model_version']}", "", "## Recommendations", *[f"- {r}" for r in o["recommendations"]],
         "", f"Test PR-AUC (2 h) {_f(o['test']['pr_auc'])} vs persistence {_f(o['test']['pr_auc_persistence'])}; "
             f"CV PR-AUC {_f(o['cv_pr_auc'])}; calibration error {_f(o['ece'])}.", "",
         "## Feature groups (CV PR-AUC change when removed; negative = the group helps)", "",
         "| Group | Change |", "|---|---|", *[f"| {a['group']} | {a['change']:+.3f} |" for a in o["ablation"]], "",
         "## By wiki", "", "| Wiki | Onsets | PR-AUC |", "|---|---|---|",
         *[f"| {w['wiki']} | {w['onsets']} | {_f(w['pr_auc'])} |" for w in o["by_wiki"]], "",
         "## By time of day (UTC)", "", "| Hours | Onsets | PR-AUC |", "|---|---|---|",
         *[f"| {h['hours_utc']} | {h['onsets']} | {_f(h['pr_auc'])} |" for h in o["by_hour"]], "",
         f"## Alert threshold (model uses {bundle.threshold:.2f})", "", "| Threshold | Alerts | Precision | Recall |",
         "|---|---|---|---|", *[f"| {t['threshold']:.2f} | {t['alerts']} | {t['precision']:.0%} | {t['recall']:.0%} |" for t in o["thresholds"]],
         "", "## Error patterns", *[f"- {p}" for p in o["patterns"]], "", "## Biggest input shifts (PSI)", "",
         "| Feature | PSI |", "|---|---|", *[f"| {d['feature']} | {_f(d['psi'], 2)} |" for d in o["shift"]], ""]
    if o["live"]:
        lv = o["live"]
        L += ["## Live", "", f"{lv['checked']} forecasts checked: {lv['hits']} hits, {lv['false_alarms']} false alarms, "
                             f"{lv['misses']} misses; live precision {lv['live_precision']:.0%} vs test "
                             f"{lv['test_precision_at_threshold']:.0%}.", ""]
    return "\n".join(L)


def main(argv=None):
    ap = argparse.ArgumentParser(description="Diagnose the champion model and optionally retrain with the fixes")
    ap.add_argument("--apply", action="store_true", help="retrain with the recommended config (the gate still decides)")
    a = ap.parse_args(argv)
    store = open_store()
    out = diagnose(store)
    print("\n".join(f"  - {r}" for r in out["recommendations"]))
    print(f"report: {C.ARTIFACTS / 'runs' / out['champion'] / 'refine.md'}")
    if a.apply:
        cfg = out["refined_config"]
        cfg["_comment"] = f"refined from {out['champion']}"
        res = run(store, cfg)
        print(f"refined run {res['run_id']}: {'PROMOTED' if res['promoted'] else 'not promoted (champion kept)'}")


if __name__ == "__main__":
    main()

## 9. Post-processing and the forecast job

Region weights and regional forecasts, the exact rows written to `FORECASTS` / `REGION_FORECASTS`, the live and batch jobs, and the Gemini cause labeller.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/regions.py
"""Region weights, regional demand and regional forecasts (CONTRACT §3.1, §3.3, §7 "Region forecasts").

Weights (stated assumption, on a slide):
  de/fr/it/nl -> LDN, es/pt -> NJ, ja/ko/id -> TYO (weight 1.0)
  enwiki      -> COUNTRY_SHARE for that article/day when available, else LDN 0.35 / NJ 0.45 / TYO 0.20
                 x each region's hour-of-day curve, renormalised
Look-ahead note: COUNTRY_SHARE for day D is only complete after D ends, so FORECASTS use the share of day D-1;
the simulator's TRUE demand may use day D (it is ground truth, never an input to a decision).
"""
from __future__ import annotations

from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

from . import config as C
from .data import HOUR, RawData


# Default enwiki split at a given UTC hour: the static shares scaled by how awake each region is.
def en_default_weights(hour_utc: pd.Timestamp) -> dict[str, float]:
    ts = pd.Timestamp(hour_utc).tz_localize("UTC") if pd.Timestamp(hour_utc).tzinfo is None else pd.Timestamp(hour_utc)
    raw = {r: C.EN_DEFAULT_WEIGHTS[r] * C.hour_curve(ts.tz_convert(ZoneInfo(C.REGION_TZ[r])).hour) for r in C.REGIONS}
    s = sum(raw.values())
    return {r: v / s for r, v in raw.items()}


# Weights for one article at one hour. `share_day_lag=1` = use yesterday's COUNTRY_SHARE (forecast-safe).
def article_weights(raw: RawData, wiki: str, title: str, hour_utc, share_day_lag: int = 1) -> dict[str, float]:
    if wiki != "enwiki":
        return {r: 1.0 if C.WIKI_REGION[wiki] == r else 0.0 for r in C.REGIONS}
    day = pd.Timestamp(hour_utc).normalize() - pd.Timedelta(days=share_day_lag)
    cs = raw.country_share
    s = cs[(cs.wiki == wiki) & (cs.title == title) & (cs.day == day)] if len(cs) else cs
    if len(s):
        w = s.groupby("region").share.sum().reindex(C.REGIONS, fill_value=0.0)
        if w.sum() > 0:
            return (w / w.sum()).to_dict()
    return en_default_weights(hour_utc)


# Regional demand V_r(h) = sum over tracked articles of views x weight (CONTRACT §3.3), using the warehouse
# aggregates in RawData (all titles, not just candidates).
def regional_views(raw: RawData) -> pd.DataFrame:
    w = raw.wiki_hour_views.copy()
    w["hour_utc"] = pd.to_datetime(w.hour_utc)
    non_en = w[w.wiki != "enwiki"].assign(region=lambda d: d.wiki.map(C.WIKI_REGION))
    out = non_en.groupby(["hour_utc", "region"]).views.sum().unstack().reindex(columns=C.REGIONS, fill_value=0).fillna(0)
    en = w[w.wiki == "enwiki"].set_index("hour_utc").views.astype(float)
    sh = raw.en_shared.copy()
    shared_w = sh.pivot_table(index="hour_utc", columns="region", values="views_weighted", aggfunc="sum") \
        .reindex(columns=C.REGIONS, fill_value=0).fillna(0) if len(sh) else pd.DataFrame(columns=C.REGIONS)
    shared_raw = sh.groupby("hour_utc").views_raw.max() if len(sh) else pd.Series(dtype=float)
    idx = out.index.union(en.index)
    out = out.reindex(idx, fill_value=0.0)
    rest = (en.reindex(idx, fill_value=0) - shared_raw.reindex(idx, fill_value=0)).clip(lower=0)
    dw = pd.DataFrame([en_default_weights(h) for h in idx], index=idx)
    for r in C.REGIONS:
        out[r] = out[r] + shared_w[r].reindex(idx, fill_value=0) + rest * dw[r]
    return out.sort_index()


# CONTRACT §7 region forecast for target hour T (issued at t):
#   q50 = baseline_r(T) + sum_a p_a x excess50_a x w_r(a)
#   q90 = baseline_r(T) + sum_{a: p_a >= 0.3} excess90_a x w_r(a)
# baseline_r(T) = regional total at T - 24 h (same hour yesterday), only if that dump was published by t.
# excess = predicted peak minus the article's own same-hour-yesterday views (its normal traffic is already in
# the baseline; adding the full peak would count it twice).
def region_forecasts(raw: RawData, pred: pd.DataFrame, issued_at, reg_views: pd.DataFrame | None = None) -> list[dict]:
    issued_at = pd.Timestamp(issued_at)
    reg_views = regional_views(raw) if reg_views is None else reg_views
    hp = raw.hour_published
    t0 = issued_at.floor("h")
    rows = []
    w = {(a.wiki, a.title): article_weights(raw, a.wiki, a.title, t0) for a in pred[["wiki", "title"]].drop_duplicates().itertuples()}
    base_a = pred.views_same_hour_yesterday.fillna(pred.med7_next_hour).fillna(0).to_numpy()
    for h in C.HORIZONS:
        T = t0 + (h - 1) * HOUR
        y = T - pd.Timedelta(hours=24)
        if y in hp.index and hp[y] <= issued_at and y in reg_views.index:
            base = reg_views.loc[y]
        else:                                     # fallback: latest published regional hour
            pub = hp[hp <= issued_at].index
            pub = [x for x in pub if x in reg_views.index]
            base = reg_views.loc[max(pub)] if pub else pd.Series(0.0, index=C.REGIONS)
        p = pred[f"p_{h}"].to_numpy()
        ex50 = np.clip(pred.size_q50.to_numpy() - base_a, 0, None)
        ex90 = np.clip(pred.size_q90.to_numpy() - base_a, 0, None)
        for r in C.REGIONS:
            wr = np.array([w[(a, b)][r] for a, b in zip(pred.wiki, pred.title)])
            q50 = float(base[r] + (p * ex50 * wr).sum())
            q90 = float(base[r] + ((p >= C.REGION_Q90_P_MIN) * ex90 * wr).sum())
            rows.append({"issued_at": issued_at, "region": r, "target_hour": T, "q50_views": q50,
                         "q90_views": max(q90, q50 * 1.05)})        # keeps the autopilot's log-normal sigma > 0
    return rows

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/postprocess.py
"""Post-processing: model output -> the exact rows PC1 reads (FORECASTS, REGION_FORECASTS) and the JSON shapes
in CONTRACT §3.4, written idempotently (re-running a cron hour replaces that hour's rows, never duplicates)."""
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from . import config as C
from .data import RawData
from .regions import article_weights, region_forecasts
from .store import Store, ts_literal


def iso(t) -> str:
    return pd.Timestamp(t).strftime("%Y-%m-%dT%H:%M:%SZ")


# Keep the top N titles per issue time (all 3 horizons each). Writing every candidate every 15 minutes would
# put ~100k rows/hour into FORECASTS for no benefit: the radar shows 8 and the scorecard checks the confident ones.
def select_rows(pred: pd.DataFrame, top: int = C.FORECAST_WRITE_TOP) -> pd.DataFrame:
    pred = pred.assign(_pmax=pred[["p_1", "p_2", "p_3"]].max(axis=1))
    return pred.sort_values("_pmax", ascending=False).groupby("t", group_keys=False).head(top).drop(columns="_pmax")


# One FORECASTS row per (article, horizon). Region = the article's biggest region weight at issue time.
def article_rows(raw: RawData, pred: pd.DataFrame, issued_at) -> pd.DataFrame:
    rows = []
    for r in pred.itertuples(index=False):
        w = article_weights(raw, r.wiki, r.title, pd.Timestamp(issued_at).floor("h"))
        region = max(w, key=w.get)
        reasons = list(getattr(r, "top_reasons", []) or [])
        for h in C.HORIZONS:
            rows.append({"issued_at": pd.Timestamp(issued_at), "wiki": r.wiki, "title": r.title, "region": region,
                         "horizon_h": h, "p_surge": float(getattr(r, f"p_{h}")), "size_q50": float(r.size_q50),
                         "size_q90": float(r.size_q90), "cause": r.cause if isinstance(r.cause, str) else None,
                         "top_reasons": reasons})
    return pd.DataFrame(rows)


def region_rows(raw: RawData, pred: pd.DataFrame, issued_at, reg_views=None) -> pd.DataFrame:
    return pd.DataFrame(region_forecasts(raw, pred, issued_at, reg_views))


# CONTRACT §3.4 JSON shapes for /live/forecasts and the radar (PC1 can serve these files directly as a fallback).
def to_json(arts: pd.DataFrame, regs: pd.DataFrame) -> tuple[list, list]:
    a = [{"issued_at": iso(r.issued_at), "wiki": r.wiki, "title": r.title, "region": r.region, "horizon_h": int(r.horizon_h),
          "p_surge": round(float(r.p_surge), 4), "size_q50": round(float(r.size_q50)), "size_q90": round(float(r.size_q90)),
          "cause": r.cause, "top_reasons": list(r.top_reasons)} for r in arts.itertuples()]
    g = [{"issued_at": iso(r.issued_at), "region": r.region, "target_hour": iso(r.target_hour),
          "q50_views": round(float(r.q50_views)), "q90_views": round(float(r.q90_views))} for r in regs.itertuples()]
    return a, g


def export_json(arts: pd.DataFrame, regs: pd.DataFrame, out_dir: Path | None = None) -> list[Path]:
    out_dir = Path(out_dir or C.LIVE_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    a, g = to_json(arts, regs)
    pa, pg = out_dir / "forecasts.json", out_dir / "region_forecasts.json"
    pa.write_text(json.dumps(a, ensure_ascii=False, indent=1))
    pg.write_text(json.dumps(g, indent=1))
    return [pa, pg]


# Idempotent writes: delete this issue window's rows, then insert. top_reasons is a VARIANT column.
def write(store: Store, arts: pd.DataFrame, regs: pd.DataFrame, issued_from, issued_to) -> dict:
    where = f"issued_at >= {ts_literal(issued_from)} AND issued_at <= {ts_literal(issued_to)}"
    n_a = store.replace_rows(arts, "FORECASTS", where, variant_cols=("top_reasons",))
    n_r = store.replace_rows(regs, "REGION_FORECASTS", where)
    return {"forecasts": n_a, "region_forecasts": n_r}

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/gemini_labeler.py
"""Gemini cause labeller (CONTRACT §7): runs on bursts only, writes CAUSE_LABELS.

Burst = edits_60m >= 8 and humans_60m >= 3 at decision time t. The prompt sees ONLY evidence that existed at t
(edit summaries with ts <= t, counts as of t, Main Page status as of t). Labels are cached by (wiki, title, hour)
so re-runs and the live job never pay twice.

    flashcrowd-gemini --start 2026-10-17 --end 2026-10-24T09:00 --budget 3000     # history, for training
    (the live forecast job calls label_bursts() itself every run when GEMINI_API_KEY is set)
"""
from __future__ import annotations

import argparse
import json
import os
import time

import pandas as pd

from . import config as C
from .data import HOUR, RawData, load_window
from .features import build, make_grid
from .store import Store, open_store, ts_literal

# JSON schema forces Gemini to answer with one of the contract's causes (CONTRACT §3.4 enum).
SCHEMA = {"type": "object", "properties": {
    "cause": {"type": "string", "enum": list(C.CAUSES)},
    "confidence": {"type": "number"}, "reason": {"type": "string"}}, "required": ["cause", "confidence", "reason"]}

PROMPT = """You classify why a Wikipedia article is being edited right now.
Article: {title} ({wiki}). Edits in the last hour: {n}, unique human editors: {humans}, reverts: {reverts}.
On the Main Page: {main_page}. Recent edit summaries (newest first):
{comments}
Return the most likely cause. Use only this evidence."""

CACHE = C.CACHE_DIR / "gemini_cache.jsonl"


class GeminiLabeler:
    # `client` can be injected (tests use a fake); otherwise the real google-genai client is built from env vars.
    def __init__(self, client=None, model: str | None = None, budget: int = 3000):
        if client is None:
            from google import genai
            client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        self.client = client
        self.model = model or os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
        self.budget = budget
        self.calls = 0
        self.cache: dict[str, dict] = {}
        if CACHE.exists():
            for line in CACHE.read_text().splitlines():
                d = json.loads(line)
                self.cache[d["key"]] = d["label"]

    # One call with temperature 0 and the schema; exponential back-off on rate limits (429 / RESOURCE_EXHAUSTED).
    def _call(self, prompt: str) -> dict:
        delay = 2.0
        for _ in range(6):
            try:
                resp = self.client.models.generate_content(
                    model=self.model, contents=prompt,
                    config={"response_mime_type": "application/json", "response_schema": SCHEMA, "temperature": 0})
                label = json.loads(resp.text)
                if label.get("cause") not in C.CAUSES:
                    label["cause"] = "other"
                label["confidence"] = float(min(max(label.get("confidence", 0), 0), 1))
                return label
            except Exception as e:                                   # noqa: BLE001 - SDK raises several types
                if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
                    time.sleep(delay)
                    delay *= 2
                    continue
                raise
        raise RuntimeError("Gemini rate-limited after 6 retries")

    def label(self, wiki, title, t, evidence: dict) -> dict | None:
        key = f"{wiki}|{title}|{pd.Timestamp(t):%Y-%m-%dT%H}"
        if key in self.cache:
            return self.cache[key]
        if self.calls >= self.budget:
            return None
        self.calls += 1
        lab = self._call(PROMPT.format(wiki=wiki, title=title.replace("_", " "), **evidence))
        self.cache[key] = lab
        CACHE.parent.mkdir(parents=True, exist_ok=True)
        with CACHE.open("a") as fh:
            fh.write(json.dumps({"key": key, "label": lab}, ensure_ascii=False) + "\n")
        return lab


# Pick the bursts from a feature frame and build each prompt's evidence from edits with ts <= t only.
def label_bursts(raw: RawData, frame: pd.DataFrame, labeler: GeminiLabeler) -> pd.DataFrame:
    bursts = frame[(frame.edits_60m >= C.GEMINI_BURST_EDITS_60M) & (frame.humans_60m >= C.GEMINI_BURST_HUMANS_60M)]
    rows = []
    e = raw.edits
    for b in bursts.itertuples():
        w = e[(e.wiki == b.wiki) & (e.title == b.title) & (e.ts <= b.t) & (e.ts > b.t - HOUR)]
        human = w[~w.is_bot].sort_values("ts", ascending=False)
        on_mp = "yes, Today's Featured Article" if b.is_tfa_today else ("yes, In the news" if b.in_itn_6h else "no")
        ev = {"n": int(len(human)), "humans": int(human.user_name.nunique()), "reverts": int(human.is_revert.sum()),
              "main_page": on_mp, "comments": "\n".join(c for c in human.comment.fillna("").head(20) if c) or "(no summaries)"}
        lab = labeler.label(b.wiki, b.title, b.t, ev)
        if lab:
            rows.append({"ts": b.t, "wiki": b.wiki, "title": b.title, "cause": lab["cause"],
                         "confidence": lab["confidence"], "reason": str(lab.get("reason", ""))[:500]})
    return pd.DataFrame(rows, columns=["ts", "wiki", "title", "cause", "confidence", "reason"])


# Idempotent write of one window of labels.
def write_labels(store: Store, labels: pd.DataFrame, t_from, t_to) -> int:
    return store.replace_rows(labels, "CAUSE_LABELS", f"ts >= {ts_literal(t_from)} AND ts <= {ts_literal(t_to)}")


def main(argv=None):
    ap = argparse.ArgumentParser(description="Label edit bursts with Gemini and write CAUSE_LABELS")
    ap.add_argument("--start", required=True)
    ap.add_argument("--end", required=True)
    ap.add_argument("--budget", type=int, default=3000)
    a = ap.parse_args(argv)
    store = open_store()
    raw = load_window(store, a.start, a.end)
    frame = build(raw, make_grid(raw, a.start, a.end), with_labels=False, with_causes=False)
    lab = GeminiLabeler(budget=a.budget)
    out = label_bursts(raw, frame, lab)
    n = write_labels(store, out, a.start, a.end)
    print(f"{n} cause labels written ({lab.calls} Gemini calls, rest from cache)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/forecast_job.py
"""Forecast job: writes FORECASTS + REGION_FORECASTS for PC1 (CONTRACT §7 P3).

    flashcrowd-forecast live                      # cron: */15 * * * *   (issued_at = now floored to 15 min)
    flashcrowd-forecast live --at 2026-10-24T15:15
    flashcrowd-forecast batch --start 2026-10-22 --end 2026-10-23T23:00     # history, for boss levels + backtest

Live order per run:
  1 pull the last 8 days from the store          4 features as of issued_at (+ provenance assert)
  2 cut to what existed at issued_at              5 predict p_1..p_3, sizes, top_reasons
  3 Gemini on bursts -> CAUSE_LABELS (if key)     6 post-process -> FORECASTS / REGION_FORECASTS + JSON files
"""
from __future__ import annotations

import argparse
import os
from datetime import datetime, timezone

import pandas as pd

from . import config as C
from . import postprocess as P
from .data import HOUR, load_window
from .features import PROVENANCE, build, make_grid
from .model import Bundle
from .regions import regional_views
from .store import Store, open_store


# Fail loudly if any input used was newer than the issue time (the live twin of the training leak check).
def assert_no_lookahead(frame: pd.DataFrame) -> None:
    for c in PROVENANCE:
        bad = frame[c].notna() & (frame[c] > frame.t)
        if bad.any():
            raise RuntimeError(f"look-ahead in live features: {c} newer than issued_at for {int(bad.sum())} rows")


def live(store: Store, bundle: Bundle, issued_at=None, gemini: bool | None = None, write: bool = True) -> dict:
    # 1-2: pull, then cut to exactly what had been published/recorded at issued_at.
    issued_at = pd.Timestamp(issued_at) if issued_at else pd.Timestamp(datetime.now(timezone.utc)).tz_localize(None).floor("15min")
    raw = load_window(store, issued_at.floor("h"), issued_at.floor("h"), label_hours=0).available(issued_at)
    raw.end = issued_at
    grid = make_grid(raw, issued_at, issued_at, every="15min" if issued_at.minute else "h")

    # 3: Gemini labels for current bursts go into CAUSE_LABELS first, then become features of this run.
    gemini = bool(os.environ.get("GEMINI_API_KEY")) if gemini is None else gemini
    n_labels = 0
    if gemini and bundle.uses_causes:
        from .gemini_labeler import GeminiLabeler, label_bursts, write_labels
        pre = build(raw, grid, with_labels=False, with_causes=False)
        labels = label_bursts(raw, pre, GeminiLabeler(budget=200))
        if write:
            n_labels = write_labels(store, labels, issued_at, issued_at)
        raw.causes = pd.concat([raw.causes, labels], ignore_index=True)

    # Nothing to forecast (no edits, no published hour, nothing on the Main Page): say so instead of failing.
    if grid.empty:
        return {"issued_at": str(issued_at), "candidates": 0, "forecast_rows": 0, "region_rows": 0,
                "cause_labels": n_labels, "written": {}, "files": [], "model_version": bundle.version,
                "note": "no candidate articles at this time (is the edit recorder running and are page views loaded?)"}

    # 4-5: features as of issued_at, predictions, reasons for the rows we will publish.
    frame = build(raw, grid, with_labels=False)
    assert_no_lookahead(frame)
    pred = bundle.predict(frame)
    pred = P.select_rows(pred)
    pred = bundle.predict(pred, reasons=True)

    # 6: post-process into PC1's tables + JSON files.
    arts = P.article_rows(raw, pred, issued_at)
    regs = P.region_rows(raw, pred, issued_at)
    files = P.export_json(arts, regs)
    written = P.write(store, arts, regs, issued_at, issued_at) if write else {}
    return {"issued_at": str(issued_at), "candidates": int(len(grid)), "forecast_rows": int(len(arts)),
            "region_rows": int(len(regs)), "cause_labels": n_labels, "written": written,
            "files": [str(f) for f in files], "model_version": bundle.version}


def batch(store: Store, bundle: Bundle, start, end, write: bool = True, every: str = "h") -> tuple[pd.DataFrame, pd.DataFrame]:
    """Forecasts for every issue hour in [start, end] (used for boss levels, the backtest and the slides).
    Features are built once for the window; each row still only sees data available at its own t."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    raw = load_window(store, start, end)
    frame = build(raw, make_grid(raw, start, end, every=every), with_labels=False)
    assert_no_lookahead(frame)
    pred = P.select_rows(bundle.predict(frame))
    pred = bundle.predict(pred, reasons=True)
    rv = regional_views(raw)
    arts, regs = [], []
    for t, g in pred.groupby("t"):
        arts.append(P.article_rows(raw, g, t))
        regs.append(P.region_rows(raw.available(t), g, t, rv))
    arts = pd.concat(arts, ignore_index=True) if arts else pd.DataFrame()
    regs = pd.concat(regs, ignore_index=True) if regs else pd.DataFrame()
    if write:
        P.write(store, arts, regs, start, end)
    return arts, regs


def main(argv=None):
    ap = argparse.ArgumentParser(description="Issue FLASHCROWD forecasts")
    sub = ap.add_subparsers(dest="mode", required=True)
    lv = sub.add_parser("live")
    lv.add_argument("--at", help="issue time (UTC); default now floored to 15 min")
    lv.add_argument("--no-write", action="store_true")
    lv.add_argument("--no-gemini", action="store_true")
    bt = sub.add_parser("batch")
    bt.add_argument("--start", required=True)
    bt.add_argument("--end", required=True)
    bt.add_argument("--no-write", action="store_true")
    a = ap.parse_args(argv)
    store, bundle = open_store(), Bundle.load()
    if a.mode == "live":
        print(live(store, bundle, a.at, gemini=False if a.no_gemini else None, write=not a.no_write))
    else:
        arts, regs = batch(store, bundle, a.start, a.end, write=not a.no_write)
        print(f"{len(arts)} FORECASTS rows and {len(regs)} REGION_FORECASTS rows for {a.start} -> {a.end}")


if __name__ == "__main__":
    main()

## 10. Shared engine (`flashcrowd_engine`, what PC1 installs)

Constants, demand, forecasts with no look-ahead, the minute simulator, the baseline policies, the newsvendor autopilot, grading and the `GameSession` PC1's server drives.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/__init__.py
"""FLASHCROWD shared engine (CONTRACTS §3.3). Pure Python + numpy; no Snowflake, no ML libraries.

PC1 usage:
    from flashcrowd_engine import Constants, Simulator, ForecastBook, region_minute_demand
    from flashcrowd_engine.decision import Autopilot, Persistence
    from flashcrowd_engine.policies import Reactive, Scheduled, Oracle, scheduled_table
    from flashcrowd_engine.grading import review
"""
from .constants import REGIONS, Constants, calibrate_scale, from_measurements
from .demand import minute_demand, region_minute_demand
from .forecasts import ForecastBook, LookAheadError
from .sim import Simulator
from .session import GameSession

__version__ = "0.3.0"
CONTRACTS_VERSION = "1.0"
__all__ = ["REGIONS", "Constants", "calibrate_scale", "from_measurements", "minute_demand", "region_minute_demand",
           "ForecastBook", "LookAheadError", "Simulator", "GameSession", "CONTRACTS_VERSION"]

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/constants.py
"""Simulator constants (CONTRACTS §3.3). Placeholders until PC1's measurements land in INFRA_MEASUREMENTS."""
from __future__ import annotations

import math
import statistics
from dataclasses import asdict, dataclass, field

# The three serving regions and the Vultr data centre each one maps to (CONTRACTS §3.1).
REGIONS = ("LDN", "NJ", "TYO")
VULTR_REGION = {"LDN": "lhr", "NJ": "ewr", "TYO": "nrt"}


# Frozen so a session's constants cannot drift mid-game; every number here is one PC1 measures.
@dataclass(frozen=True)
class Constants:
    C: float = 250.0                       # req/s one server sustains at p95 < 200 ms (PC1 load test)
    B: dict = field(default_factory=lambda: {"LDN": 3, "NJ": 3, "TYO": 4})  # median boot-to-serving minutes
    SCALE: float = 1.0                     # requests per real Wikipedia view
    MIN_SERVERS: int = 1
    MAX_SERVERS: int = 40
    TARGET_UTIL: float = 0.8               # autopilot / scheduled size servers to 80% of C

    # Turn an hourly view rate into a server count at the target utilisation (used by autopilot + scheduled).
    def servers_for_views(self, views_per_hour: float, util: float | None = None) -> int:
        """Servers needed for an hourly view rate: ceil(SCALE x V / 3600 / (C x util))."""
        u = self.TARGET_UTIL if util is None else util
        return math.ceil(self.SCALE * views_per_hour / 3600.0 / (self.C * u) - 1e-9)

    def to_json(self) -> dict:
        return asdict(self)


# Read PC1's INFRA_MEASUREMENTS rows: median boot time per region (ceil to minutes) and the best capacity C.
def from_measurements(rows, base: Constants | None = None) -> Constants:
    """Build constants from INFRA_MEASUREMENTS rows (dicts or a DataFrame) written by PC1.

    kind='boot'     value in seconds or minutes (unit column), one row per boot -> median, ceil to minutes
    kind='capacity' value in req/s -> the highest reported value is C
    """
    base = base or Constants()
    if hasattr(rows, "to_dict"):
        rows = rows.to_dict("records")
    # Collect boot samples per region and capacity samples, accepting either upper- or lower-case column names.
    boots: dict[str, list[float]] = {}
    caps: list[float] = []
    for r in rows:
        kind = str(r.get("kind") or r.get("KIND") or "").lower()
        val = r.get("value", r.get("VALUE"))
        unit = str(r.get("unit") or r.get("UNIT") or "").lower()
        region = r.get("region") or r.get("REGION")
        if val is None:
            continue
        if kind == "boot" and region in REGIONS:
            minutes = float(val) / 60.0 if unit.startswith("s") else float(val)
            boots.setdefault(region, []).append(minutes)
        elif kind == "capacity":
            caps.append(float(val))
    # Regions without measurements keep the default boot time, so a partial bench still works.
    B = dict(base.B)
    for region, vals in boots.items():
        B[region] = max(1, math.ceil(statistics.median(vals)))
    C = max(caps) if caps else base.C
    return Constants(C=C, B=B, SCALE=base.SCALE, MIN_SERVERS=base.MIN_SERVERS, MAX_SERVERS=base.MAX_SERVERS,
                     TARGET_UTIL=base.TARGET_UTIL)


# SCALE converts views into requests; pick it so an ordinary hour needs about 4 servers per region.
def calibrate_scale(hourly_views: dict[str, list[float]], constants: Constants, target_servers: float = 4.0) -> Constants:
    """Pick SCALE so a median hour needs ``target_servers`` per region (the contract says 2-6)."""
    allv = [v for vals in hourly_views.values() for v in vals if v > 0]
    if not allv:
        return constants
    med = statistics.median(allv)
    scale = target_servers * constants.C * constants.TARGET_UTIL * 3600.0 / med
    return Constants(C=constants.C, B=dict(constants.B), SCALE=scale, MIN_SERVERS=constants.MIN_SERVERS,
                     MAX_SERVERS=constants.MAX_SERVERS, TARGET_UTIL=constants.TARGET_UTIL)

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/demand.py
"""Hourly regional views -> per-minute request rate (CONTRACTS §3.3 Demand)."""
from __future__ import annotations

import numpy as np


def minute_demand(hourly_views, scale: float) -> np.ndarray:
    """D(m) in requests/minute for every minute of the hours given.

    The rate at each hour's midpoint is SCALE x V(h) / 60; minutes in between are linearly interpolated
    and the first/last half hours hold the edge value.
    """
    # Validate input: demand must be a 1-D list of non-negative, finite hourly view counts.
    v = np.asarray(hourly_views, dtype=float)
    if v.ndim != 1 or len(v) == 0:
        raise ValueError("hourly_views must be a non-empty 1-D sequence")
    if np.any(v < 0) or not np.all(np.isfinite(v)):
        raise ValueError("hourly views must be finite and non-negative")
    # Requests per minute at each hour's midpoint, then straight lines between midpoints (edges held flat).
    rate = scale * v / 60.0
    mids = np.arange(len(v)) * 60 + 30
    return np.interp(np.arange(len(v) * 60), mids, rate)


def region_minute_demand(hourly: dict[str, list[float]], scale: float) -> dict[str, np.ndarray]:
    return {r: minute_demand(v, scale) for r, v in hourly.items()}

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/forecasts.py
"""Forecast rows for a session, exposed to policies only up to ``now``.

Rows are sorted by issue minute once. At minute m a policy gets a ``Visible`` sequence that ends at the last
row with issued_at <= now; indexing past that end raises ``LookAheadError``. Nothing is copied per minute.
"""
from __future__ import annotations

import bisect
from collections.abc import Sequence
from datetime import datetime, timezone


# Raised (and tested) whenever anything asks for a forecast issued after the current minute.
class LookAheadError(RuntimeError):
    """Raised when code asks for a forecast that had not been issued yet."""


# Accept ISO strings, pandas Timestamps or datetimes, and always return a timezone-aware UTC datetime.
def parse_utc(x) -> datetime:
    if isinstance(x, datetime):
        return x if x.tzinfo else x.replace(tzinfo=timezone.utc)
    if hasattr(x, "to_pydatetime"):
        return parse_utc(x.to_pydatetime())
    s = str(x).replace("Z", "+00:00").replace(" ", "T", 1)
    d = datetime.fromisoformat(s)
    return d if d.tzinfo else d.replace(tzinfo=timezone.utc)


def iso(d: datetime) -> str:
    return d.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# Whole minutes from the session start to a timestamp; negative if it was issued before the level began.
def minutes_between(start: datetime, t: datetime) -> int:
    return int((parse_utc(t) - start).total_seconds() // 60)


# A read-only, zero-copy window onto the sorted forecast list: it simply stops at the last row issued <= now.
class Visible(Sequence):
    __slots__ = ("_rows", "_n")

    def __init__(self, rows: list, n: int):
        self._rows, self._n = rows, n

    def __len__(self):
        return self._n

    def __iter__(self):
        for j in range(self._n):
            yield self._rows[j]

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self._rows[j] for j in range(*i.indices(self._n))]
        if i < 0:
            i += self._n
        if not 0 <= i < self._n:
            raise LookAheadError(f"forecast index {i} is not visible yet (only {self._n} issued)")
        return self._rows[i]


# Holds every forecast for a session, sorted once by issue minute, and hands out Visible windows per minute.
class ForecastBook:
    """Region + article forecasts for one session. Each row is a dict in CONTRACTS §3.4 shape."""

    def __init__(self, start_utc, region_rows=(), article_rows=()):
        self.start = parse_utc(start_utc)
        self._region = self._prep(region_rows, needs_target=True)
        self._article = self._prep(article_rows, needs_target=False)
        # Issue minutes kept in plain lists so bisect finds the visible cut-off in O(log n).
        self._rm = [r["_m"] for r in self._region]
        self._am = [r["_m"] for r in self._article]

    def _prep(self, rows, needs_target):
        out = []
        for r in rows:
            r = dict(r)
            r["_m"] = minutes_between(self.start, r["issued_at"])
            if needs_target:
                r["_target_m"] = minutes_between(self.start, r["target_hour"])
                if r["_target_m"] < r["_m"] - 60:
                    raise ValueError(f"region forecast targets an hour that ended before it was issued: {r}")
            out.append(r)
        out.sort(key=lambda r: r["_m"])
        return out

    # Everything issued at or before minute m: this is the only forecast access a policy ever gets.
    def visible(self, m: int) -> tuple[Visible, Visible]:
        return (Visible(self._region, bisect.bisect_right(self._rm, m)),
                Visible(self._article, bisect.bisect_right(self._am, m)))

    # Walk backwards from the newest visible row to find the latest forecast covering a target minute.
    @staticmethod
    def latest_region(visible: Visible, region: str, target_minute: int):
        """Newest visible region forecast whose target hour contains ``target_minute``."""
        for i in range(len(visible) - 1, -1, -1):
            r = visible[i]
            if r["region"] == region and r["_target_m"] <= target_minute < r["_target_m"] + 60:
                return r
        return None

    # Radar helper: the top article forecasts from the most recent issue time (for reasons and the review).
    @staticmethod
    def latest_articles(visible: Visible, region: str | None = None, k: int = 8):
        """Top-k article forecasts by p_surge from the most recent issue time."""
        if not len(visible):
            return []
        last = visible[len(visible) - 1]["_m"]
        rows = []
        for i in range(len(visible) - 1, -1, -1):
            r = visible[i]
            if r["_m"] != last:
                break
            if region is None or r.get("region") == region:
                rows.append(r)
        rows.sort(key=lambda r: -float(r.get("p_surge", 0)))
        return rows[:k]

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/sim.py
"""Minute-step, multi-region server simulator (CONTRACTS §3.3). Deterministic: no randomness anywhere.

Each minute m, per region:
  (1) servers whose ready_at <= m join ``running``
  (2) cap = running x C x 60; served = min(D, cap); dropped = D - served
  (3) server_minutes += running + len(booting)          (booting servers cost too)
  (4) breach_minutes += 1 if dropped > 0.01 x D
  (5) the policy (if any) acts on the observation; its actions start booting with ready_at = m + B[r]

A human player's actions are applied with ``apply`` before ``step`` (PC1: "apply queued actions -> step").
"""
from __future__ import annotations

import math
from collections import deque
from datetime import timedelta

import numpy as np

from .constants import REGIONS, Constants
from .forecasts import ForecastBook, iso, parse_utc

# Policies see the last 15 minutes of utilisation and demand (CONTRACTS §3.3 observation).
HIST = 15


# Per-region counters named exactly as in the contract, plus the short histories policies observe.
class RegionState:
    __slots__ = ("running", "booting", "served", "demand", "dropped", "server_minutes", "breach_minutes",
                 "util_hist", "demand_hist")

    def __init__(self, running: int):
        self.running = running
        self.booting: list[int] = []
        self.served = self.demand = self.dropped = 0.0
        self.server_minutes = 0
        self.breach_minutes = 0
        self.util_hist: deque = deque(maxlen=HIST)
        self.demand_hist: deque = deque(maxlen=HIST)


# One simulator holds all regions for one player (you, autopilot, reactive, ...).
class Simulator:
    def __init__(self, demand: dict[str, np.ndarray], constants: Constants, start_utc, policy=None,
                 forecasts: ForecastBook | None = None, initial: dict[str, int] | None = None, name: str = "sim"):
        self.k = constants
        self.start = parse_utc(start_utc)
        # Keep LDN/NJ/TYO order, check every region has the same number of minutes and sane demand.
        self.regions = [r for r in REGIONS if r in demand] + [r for r in demand if r not in REGIONS]
        self.D = {r: np.asarray(demand[r], dtype=float) for r in self.regions}
        lens = {len(v) for v in self.D.values()}
        if len(lens) != 1:
            raise ValueError("all regions need the same number of minutes")
        self.minutes = lens.pop()
        for r in self.regions:
            if np.any(self.D[r] < 0) or not np.all(np.isfinite(self.D[r])):
                raise ValueError("demand must be finite and non-negative")
        self.policy = policy
        self.forecasts = forecasts or ForecastBook(self.start)
        self.name = name if policy is None else getattr(policy, "name", name)
        self.m = 0
        # Starting fleet: enough servers for minute 0 at the target utilisation, unless the caller sets it.
        self.state: dict[str, RegionState] = {}
        for r in self.regions:
            n0 = (initial or {}).get(r)
            if n0 is None:
                n0 = math.ceil(self.D[r][0] / (self.k.C * 60 * self.k.TARGET_UTIL) - 1e-9)
            self.state[r] = RegionState(int(min(max(n0, self.k.MIN_SERVERS), self.k.MAX_SERVERS)))
        # per-minute logs for grading/review
        self.log_running = {r: np.zeros(self.minutes, dtype=int) for r in self.regions}
        self.log_booting = {r: np.zeros(self.minutes, dtype=int) for r in self.regions}
        self.log_served = {r: np.zeros(self.minutes) for r in self.regions}
        self.launches: list[tuple[int, str, int]] = []   # (minute, region, servers started)
        if policy is not None and hasattr(policy, "reset"):
            policy.reset(self)

    # ---------------- actions ----------------
    def apply(self, action: dict) -> dict:
        """Apply {"type":"scale","region":"TYO","delta":+3}. Returns {"ok": True} or {"ok": False, "error": ...}."""
        if action.get("type") != "scale":
            return {"ok": False, "error": "unknown_action"}
        r = action.get("region")
        if r not in self.state:
            return {"ok": False, "error": "unknown_region"}
        try:
            delta = int(action.get("delta", 0))
        except (TypeError, ValueError):
            return {"ok": False, "error": "bad_delta"}
        st, k = self.state[r], self.k
        # Scale up: never above MAX_SERVERS; new servers boot and serve from minute m + B[region].
        if delta > 0:
            room = k.MAX_SERVERS - st.running - len(st.booting)
            n = min(delta, room)
            if n <= 0:
                return {"ok": False, "error": "max_servers"}
            st.booting.extend([self.m + k.B[r]] * n)
            if self.m < self.minutes:
                self.launches.append((self.m, r, n))
            return {"ok": True, "applied": n}
        # Scale down: cancel the newest boots first, then stop running servers, never below MIN_SERVERS.
        if delta < 0:
            remove = -delta
            cancel = min(remove, len(st.booting))
            if cancel:
                st.booting.sort()
                del st.booting[len(st.booting) - cancel:]          # cancel the newest boots first
            remove -= cancel
            can = max(0, st.running - k.MIN_SERVERS)
            take = min(remove, can)
            st.running -= take
            if cancel == 0 and take == 0:
                return {"ok": False, "error": "min_servers"}
            return {"ok": True, "applied": -(cancel + take)}
        return {"ok": True, "applied": 0}

    # ---------------- stepping ----------------
    @property
    def now(self):
        return self.start + timedelta(minutes=self.m)

    @property
    def done(self) -> bool:
        return self.m >= self.minutes

    # The exact observation shape in CONTRACTS §3.3; forecasts are Visible windows cut at the current minute.
    def observation(self) -> dict:
        reg_vis, art_vis = self.forecasts.visible(self.m)
        return {
            "m": self.m, "now_utc": iso(self.now),
            "regions": {r: {"running": s.running, "booting": sorted(s.booting), "util_hist": list(s.util_hist),
                            "demand_hist": list(s.demand_hist)} for r, s in self.state.items()},
            "region_forecasts": reg_vis, "article_forecasts": art_vis,
        }

    # Advance one minute in the contract's order and return the per-region numbers for the WebSocket tick.
    def step(self) -> dict:
        if self.done:
            raise RuntimeError("simulation finished")
        m, k, tick = self.m, self.k, {}
        for r, st in self.state.items():
            # (1) servers whose boot finished join running.
            ready = [x for x in st.booting if x <= m]
            if ready:
                st.running += len(ready)
                st.booting = [x for x in st.booting if x > m]
            # (2) serve what capacity allows, (3) bill running + booting, (4) count a breach if > 1% dropped.
            d = float(self.D[r][m])
            cap = st.running * k.C * 60.0
            served = min(d, cap)
            dropped = d - served
            st.served += served
            st.demand += d
            st.dropped += dropped
            st.server_minutes += st.running + len(st.booting)
            if dropped > 0.01 * d:
                st.breach_minutes += 1
            st.util_hist.append(d / cap if cap > 0 else float("inf"))
            st.demand_hist.append(d)
            self.log_running[r][m] = st.running
            self.log_booting[r][m] = len(st.booting)
            self.log_served[r][m] = served
            tick[r] = {"running": st.running, "booting": sorted(st.booting), "demand_rps": d / 60.0,
                       "cap_rps": cap / 60.0, "dropped_rps": dropped / 60.0}
        # (5) the policy acts last, on what it can see now; its new servers are ready at m + B.
        if self.policy is not None:
            for a in self.policy.act(self.observation()) or []:
                self.apply(a)
        self.m += 1
        return tick

    def run(self) -> "Simulator":
        while not self.done:
            self.step()
        return self

    # ---------------- scores ----------------
    def score(self) -> dict:
        served = sum(s.served for s in self.state.values())
        demand = sum(s.demand for s in self.state.values())
        return {
            "served_pct": 100.0 * served / demand if demand else 100.0,
            "server_hours": sum(s.server_minutes for s in self.state.values()) / 60.0,
            "breach_minutes": int(sum(s.breach_minutes for s in self.state.values())),
        }

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/policies.py
"""Baseline policies (CONTRACTS §7 table). Each returns a list of scale actions from one observation."""
from __future__ import annotations

import math
from datetime import timedelta

import numpy as np

from .constants import Constants
from .forecasts import ForecastBook, iso, parse_utc


# Build an action in the API's shape.
def scale(region: str, delta: int) -> dict:
    return {"type": "scale", "region": region, "delta": int(delta)}


# Base class: reset() is called when a simulator starts, act() once per minute.
class Policy:
    name = "policy"

    def reset(self, sim) -> None:
        self.sim = sim
        self.k: Constants = sim.k

    def act(self, obs: dict) -> list[dict]:  # pragma: no cover - interface
        raise NotImplementedError


# Baseline the pitch compares against: a textbook utilisation-threshold autoscaler.
class Reactive(Policy):
    """Common autoscaler defaults: utilisation > 70% for 2 consecutive minutes -> +1 (at most one scale-up per
    2 minutes); utilisation < 30% for 15 consecutive minutes -> -1."""
    name = "reactive"

    def __init__(self, up=0.70, down=0.30, up_minutes=2, down_minutes=15, cooldown=2):
        self.up, self.down, self.up_m, self.down_m, self.cool = up, down, up_minutes, down_minutes, cooldown

    def reset(self, sim):
        super().reset(sim)
        self.hi = {r: 0 for r in sim.regions}
        self.lo = {r: 0 for r in sim.regions}
        self.last_up = {r: -10**9 for r in sim.regions}

    def act(self, obs):
        out, m = [], obs["m"]
        for r, s in obs["regions"].items():
            u = s["util_hist"][-1] if s["util_hist"] else 0.0
            # Count consecutive hot and cold minutes from the latest utilisation sample.
            self.hi[r] = self.hi[r] + 1 if u > self.up else 0
            self.lo[r] = self.lo[r] + 1 if u < self.down else 0
            if self.hi[r] >= self.up_m and m - self.last_up[r] >= self.cool:
                out.append(scale(r, +1))
                self.last_up[r] = m
            elif self.lo[r] >= self.down_m:
                out.append(scale(r, -1))
                self.lo[r] = 0
        return out


# Baseline 2: a fixed daily plan built from training days, changed only on the hour.
class Scheduled(Policy):
    """Servers per region per hour-of-day = ceil(q90 of that hour's demand over training days / (C x 0.8)),
    set on the hour. ``table[region][hour_of_day]`` comes from ``scheduled_table``."""
    name = "scheduled"

    def __init__(self, table: dict[str, list[int]]):
        self.table = table

    def act(self, obs):
        now = parse_utc(obs["now_utc"])
        if now.minute != 0 and obs["m"] != 0:
            return []
        out = []
        for r, s in obs["regions"].items():
            want = self.table[r][now.hour]
            have = s["running"] + len(s["booting"])
            if want != have:
                out.append(scale(r, want - have))
        return out


# 90th percentile of each hour-of-day's demand over the training days -> servers at 80% utilisation.
def scheduled_table(hourly_views: dict[str, list[tuple[int, float]]], k: Constants) -> dict[str, list[int]]:
    """hourly_views[region] = [(hour_of_day_utc, views), ...] over the training days."""
    out = {}
    for r, rows in hourly_views.items():
        by_h = {h: [] for h in range(24)}
        for h, v in rows:
            by_h[int(h)].append(float(v))
        out[r] = [min(k.MAX_SERVERS, max(k.MIN_SERVERS, k.servers_for_views(float(np.quantile(by_h[h], 0.9)))
                                         if by_h[h] else k.MIN_SERVERS)) for h in range(24)]
    return out


# Ceiling for charts only: perfect knowledge of demand, so the gap to it is what forecasting could still win.
class Oracle(Policy):
    """Upper bound for charts only: knows the true per-minute demand and starts servers exactly B[r] early."""
    name = "oracle"

    def __init__(self, demand: dict[str, np.ndarray], lookahead_pad: int = 10):
        self.D = {r: np.asarray(v, dtype=float) for r, v in demand.items()}
        self.pad = lookahead_pad

    def act(self, obs):
        out, m, k = [], obs["m"], self.k
        for r, s in obs["regions"].items():
            D, B = self.D[r], k.B[r]
            # Servers needed at minute i with zero drops; start exactly B minutes early, release when the next window is quieter.
            need = lambda i: math.ceil(D[min(i, len(D) - 1)] / (k.C * 60) - 1e-9)
            have = s["running"] + len(s["booting"])
            target_soon = need(m + B)
            if target_soon > have:
                out.append(scale(r, target_soon - have))
                continue
            keep = max(need(i) for i in range(m + 1, min(len(D), m + B + 1 + self.pad) + 1)) if m + 1 < len(D) else 0
            keep = max(keep, k.MIN_SERVERS)
            if have > keep:
                out.append(scale(r, keep - have))
        return out


# Baseline 3 input: pretend the forecast for each hour is yesterday's same hour.
def persistence_forecasts(yesterday_views: dict[str, list[float]], start_utc, q90_mult: float = 1.3) -> list[dict]:
    """Region forecasts for the persistence baseline: each target hour = same hour yesterday.

    yesterday_views[r][i] = views in hour (start - 24h + i). Issued 3 h before the target (the value was
    published ~22 h earlier, so this is conservative). q90 = q50 x ``q90_mult`` (stated assumption).
    """
    start = parse_utc(start_utc)
    rows = []
    for r, vals in yesterday_views.items():
        for i, v in enumerate(vals):
            target = start + timedelta(hours=i)
            rows.append({"issued_at": iso(target - timedelta(hours=3)), "region": r, "target_hour": iso(target),
                         "q50_views": float(v), "q90_views": float(v) * q90_mult})
    return rows

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/decision.py
"""Newsvendor autopilot (CONTRACTS §7 decision.py).

c_o = 1 per idle server-hour, c_u = kappa per server-hour of shortfall -> critical ratio alpha = kappa / (kappa + 1).
Every 5 minutes, per region, look at the forecast for t + B[r] (the earliest a new server could help) through
t + B[r] + 5 (the next decision can only help from t + 5 + B[r] on, so this one must cover the gap), model demand
as log-normal from (q50, q90), provision for its alpha-quantile at 80% utilisation:
    mu = ln q50,  sigma = (ln q90 - ln q50) / 1.2816,  Q_alpha = exp(mu + sigma z_alpha)
    N* = ceil(SCALE x Q_alpha / 3600 / (C x 0.8)), clamped to [MIN, MAX]
Scale up immediately; scale down only if N* for both t+B and t+B+60 is below running - 1, one server per 10 minutes.
"""
from __future__ import annotations

import math
from statistics import NormalDist

from .forecasts import ForecastBook
from .policies import Policy, persistence_forecasts, scale

# z-score of the 90th percentile, used to turn (q50, q90) into a log-normal sigma.
Z90 = 1.2815515655446004


# Newsvendor critical ratio: kappa = 10 -> provision for the 91st percentile of demand.
def critical_ratio(kappa: float) -> float:
    return kappa / (kappa + 1.0)


# Fit a log-normal through q50 and q90 and read off the alpha-quantile.
def lognormal_quantile(q50: float, q90: float, alpha: float) -> float:
    q50 = max(float(q50), 1e-9)
    q90 = max(float(q90), q50)
    mu = math.log(q50)
    sigma = (math.log(q90) - mu) / Z90
    return math.exp(mu + sigma * NormalDist().inv_cdf(min(max(alpha, 1e-6), 1 - 1e-6)))


# FLASHCROWD's decision engine. `decisions` keeps the latest action + reasons per region for /agent/reasons.
class Autopilot(Policy):
    name = "autopilot"

    def __init__(self, kappa: float = 10.0, every: int = 5, down_every: int = 10, forecasts: ForecastBook | None = None,
                 name: str | None = None):
        self.kappa = kappa
        self.alpha = critical_ratio(kappa)
        self.every, self.down_every = every, down_every
        self._own_book = forecasts          # persistence passes its own forecasts; otherwise use the session's
        if name:
            self.name = name
        self.decisions: dict[str, dict] = {}   # latest decision per region, for /agent/reasons

    def reset(self, sim):
        super().reset(sim)
        self.last_down = {r: -10**9 for r in sim.regions}

    # Servers needed for the forecast covering `minute`, clamped to [MIN, MAX].
    def _target(self, vis, r, minute) -> tuple[int | None, dict | None]:
        f = ForecastBook.latest_region(vis, r, minute)
        if f is None:
            return None, None
        q = lognormal_quantile(f["q50_views"], f["q90_views"], self.alpha)
        n = self.k.servers_for_views(q)
        return min(max(n, self.k.MIN_SERVERS), self.k.MAX_SERVERS), f

    def act(self, obs):
        m = obs["m"]
        # Decide only every 5 minutes, like the roadmap's autopilot.
        if m % self.every:
            return []
        if self._own_book is not None:
            reg_vis, art_vis = self._own_book.visible(m)
        else:
            reg_vis, art_vis = obs["region_forecasts"], obs["article_forecasts"]
        out = []
        for r, s in obs["regions"].items():
            B = self.k.B[r]
            # Cover t+B and t+B+5: the next decision cannot add capacity before t+5+B.
            cands = [self._target(reg_vis, r, x) for x in (m + B, m + B + self.every)]
            cands = [c for c in cands if c[0] is not None]
            if not cands:
                continue
            n1, f = max(cands, key=lambda c: c[0])
            # Look an hour further for the scale-down hysteresis check.
            n2, _ = self._target(reg_vis, r, m + B + 60)
            have = s["running"] + len(s["booting"])
            action = "hold"
            if n1 > have:
                out.append(scale(r, n1 - have))
                action = f"start {n1 - have} server{'s' if n1 - have > 1 else ''}"
            elif n1 < s["running"] - 1 and (n2 is None or n2 < s["running"] - 1) and m - self.last_down[r] >= self.down_every:
                out.append(scale(r, -1))
                self.last_down[r] = m
                action = "stop 1 server"
            # Explain the decision with the region's top article forecasts (deduplicated across horizons).
            arts = ForecastBook.latest_articles(art_vis, r, k=12)
            seen, top = set(), []
            for a in arts:
                key = (a.get("wiki"), a.get("title"))
                if key not in seen:
                    seen.add(key)
                    top.append(a)
            reasons = []
            for a in top[:3]:
                rs = a.get("top_reasons") or []
                reasons.append(f"{str(a.get('title', '')).replace('_', ' ')} ({a.get('wiki')}): "
                               f"{float(a.get('p_surge', 0)):.0%} surge chance" + (f"; {rs[0]}" if rs else ""))
            self.decisions[r] = {"region": r, "m": m, "now_utc": obs["now_utc"], "action": action, "target": n1,
                                 "q50_views": f["q50_views"], "q90_views": f["q90_views"],
                                 "p_surge": float(top[0]["p_surge"]) if top else 0.0, "reasons": reasons}
        return out


# Same rule, naive forecast: isolates how much of the win comes from the model rather than the rule.
class Persistence(Autopilot):
    """Same decision rule, but the forecast for each hour is that hour's demand yesterday (CONTRACTS §7)."""

    def __init__(self, yesterday_views: dict[str, list[float]], start_utc, kappa: float = 10.0, q90_mult: float = 1.3):
        book = ForecastBook(start_utc, persistence_forecasts(yesterday_views, start_utc, q90_mult))
        super().__init__(kappa=kappa, forecasts=book, name="persistence")

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/grading.py
"""Scores, regional surge onsets and the shift review (CONTRACTS §3.5 review object)."""
from __future__ import annotations

import math
from datetime import timedelta

import numpy as np

from .constants import Constants
from .forecasts import ForecastBook, iso

# Verdict thresholds from CONTRACTS §3.5.
LATE_WINDOW = 30          # "late" = enough servers within 30 min after onset
SHORT_WINDOW = 60         # minutes_short counted over the first hour of the surge
BRILLIANT_P = 0.5
OVERSPEND_SERVERS = 3
OVERSPEND_WINDOW = 60
OVERSPEND_HORIZON = 180
WARMUP = 10               # launches in the first minutes only size the fleet for the level; never "overspent"


# Servers required each minute to drop nothing.
def needed_servers(D: np.ndarray, k: Constants) -> np.ndarray:
    """Servers needed each minute to drop nothing (full capacity C)."""
    return np.ceil(np.asarray(D) / (k.C * 60.0) - 1e-9).astype(int)


# Regional surge onset: needed servers jump to >= 1.5x the trailing 3-hour median and by >= 2 servers.
def surge_onsets(D: np.ndarray, k: Constants, ratio: float = 1.5, min_jump: int = 2, lookback: int = 180,
                 refractory: int = 120) -> list[int]:
    """Minutes where needed servers first rise to >= ratio x the trailing median and >= +min_jump servers."""
    need = needed_servers(D, k)
    onsets, last = [], -10**9
    for m in range(1, len(need)):
        base = np.median(need[max(0, m - lookback): m])
        if need[m] >= max(ratio * base, base + min_jump) and need[m - 1] < max(ratio * base, base + min_jump):
            if m - last >= refractory:
                onsets.append(m)
                last = m
    return onsets


# Grade letter for the review screen.
def letter(player: dict, ref: dict) -> str:
    """A-F on served %, one grade lower if it used > 30% more server-hours than the reference (autopilot)."""
    s = player["served_pct"]
    g = "A" if s >= 99 else "B" if s >= 97 else "C" if s >= 94 else "D" if s >= 90 else "F"
    if ref and ref.get("server_hours") and player["server_hours"] > 1.3 * ref["server_hours"] and g != "F":
        g = "ABCDF"["ABCDF".index(g) + 1]
    return g


# Highest article p_surge in a region visible at a minute (for the 'brilliant' test: scaled up while p < 0.5).
def _p_at(book: ForecastBook | None, region: str, minute: int) -> float:
    if book is None:
        return float("nan")
    _, arts = book.visible(minute)
    top = ForecastBook.latest_articles(arts, region, k=1)
    return float(top[0]["p_surge"]) if top else 0.0


# One player's verdict for one surge: brilliant / on_time / late / missed, with minutes short.
def _verdict(sim, r: str, onset: int, need: np.ndarray, book) -> dict:
    run, boot = sim.log_running[r], sim.log_booting[r]
    ok = run >= need
    end = min(len(need), onset + SHORT_WINDOW)
    short = int((~ok[onset:end]).sum())
    out = {"ready_at_onset": int(run[onset]), "minutes_short": short}
    if ok[onset]:
        # when did the fleet (running + booting) first reach what the onset needed, without dropping back?
        cover = (run + boot) >= need[onset]
        s = onset
        while s > 0 and cover[s - 1] and onset - s < 360:
            s -= 1
        # Brilliant needs a real scale-up: servers started in [s-1, onset) while the surge chance was still < 0.5.
        # A fleet that was already big enough from the start is simply on time.
        ups = [m for (m, rr, n) in sim.launches if rr == r and s - 1 <= m < onset]
        p = _p_at(book, r, ups[0]) if ups else float("nan")
        out["verdict"] = "brilliant" if (ups and not math.isnan(p) and p < BRILLIANT_P) else "on_time"
        if ups:
            out["scale_up_started_m"] = ups[0]
    elif ok[onset: min(len(need), onset + LATE_WINDOW + 1)].any():
        out["verdict"] = "late"
    else:
        out["verdict"] = "missed"
    return out


# The review object: scores + grade per player, then every regional surge with each player's verdict.
def review(sims: dict, k: Constants, onset_titles: dict | None = None, book: ForecastBook | None = None,
           reference: str = "autopilot", surge_ratio: float = 1.5, min_jump: int = 2) -> dict:
    """Review object: per-player scores + grade, and one entry per regional surge with each player's verdict.

    sims: {"you": Simulator, "autopilot": ..., "reactive": ..., "scheduled": ...} sharing the same demand.
    onset_titles: optional {(region, minute): "Typhoon"} for display.
    """
    any_sim = next(iter(sims.values()))
    scores = {name: s.score() for name, s in sims.items()}
    ref = scores.get(reference, {})
    out = {name: {**sc, "grade": letter(sc, ref)} for name, sc in scores.items()}
    surges = []
    for r in any_sim.regions:
        D = any_sim.D[r]
        need = needed_servers(D, k)
        onsets = surge_onsets(D, k, ratio=surge_ratio, min_jump=min_jump)
        for o in onsets:
            peak = int(need[o: min(len(need), o + SHORT_WINDOW)].max())
            item = {"title": (onset_titles or {}).get((r, o), "Regional surge"), "region": r,
                    "onset": iso(any_sim.start + timedelta(minutes=o)),
                    "onset_m": o, "needed_servers": peak}
            for name, s in sims.items():
                item[name] = _verdict(s, r, o, need, book)
            surges.append(item)
        # overspent: >= 3 servers started within an hour for a surge that never came
        for name, s in sims.items():
            launches = [(m, n) for (m, rr, n) in s.launches if rr == r and m >= WARMUP]
            i = 0
            while i < len(launches):
                m0 = launches[i][0]
                group = [n for (m, n) in launches if m0 <= m < m0 + OVERSPEND_WINDOW]
                if sum(group) >= OVERSPEND_SERVERS and not any(m0 <= o <= m0 + OVERSPEND_HORIZON for o in onsets) \
                        and need[m0: m0 + OVERSPEND_HORIZON].max(initial=0) <= need[max(0, m0 - 1)] + 1:
                    surges.append({"title": "No surge", "region": r, "onset": None, "onset_m": None,
                                   "needed_servers": int(need[m0]), name: {"verdict": "overspent",
                                                                           "started_m": m0, "servers": sum(group)}})
                    while i < len(launches) and launches[i][0] < m0 + OVERSPEND_WINDOW:
                        i += 1
                else:
                    i += 1
    surges.sort(key=lambda x: (x["onset_m"] is None, x["onset_m"] or 0))
    out["surges"] = surges
    return out


# Pareto points (x = server-hours, y = % dropped) with the non-dominated ones flagged.
def pareto_points(results: list[dict]) -> list[dict]:
    """Points for the Pareto chart: x = server_hours, y = 100 - served_pct; marks the non-dominated ones."""
    pts = sorted(({**r, "x": r["server_hours"], "y": 100.0 - r["served_pct"]} for r in results),
                 key=lambda p: (p["x"], p["y"]))
    best = float("inf")
    for p in pts:
        p["pareto"] = p["y"] < best
        best = min(best, p["y"])
    return pts

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_engine/session.py
"""One game session = four players on the same demand (CONTRACTS §3.5). Built for PC1's sessions.py / ws.py.

    s = GameSession.from_level_file("levels/oct-15-night-ldn.json")      # package written by `flashcrowd export`
    s.info()                       -> POST /sessions response body
    s.action({...})                -> POST /sessions/{id}/action response body (queued for the next tick)
    s.tick()                       -> one WebSocket message per sim minute ({"type":"tick"...} then {"type":"end"...})
    s.review()                     -> GET /sessions/{id}/review
    s.reasons("TYO")               -> GET /agent/reasons?region=TYO
All numbers come from the shared simulator; nothing here is random, so a replay gives the same scores.
"""
from __future__ import annotations

import json
import uuid
from datetime import timedelta

from .constants import REGIONS, Constants
from .decision import Autopilot
from .demand import region_minute_demand
from .forecasts import ForecastBook, iso, parse_utc
from .grading import review as grade_review
from .grading import surge_onsets
from .policies import Reactive, Scheduled, scheduled_table
from .sim import Simulator

MAX_DELTA = 5                   # CONTRACTS §6 P2: delta must be in [-5, +5]
ALERT_P = 0.6                   # an alert fires the first time a title's forecast crosses 0.6
RADAR_N = 8
CITY = {"LDN": "London", "NJ": "New Jersey", "TYO": "Tokyo"}


def _public(row: dict) -> dict:
    """Forecast row without the engine's private bookkeeping keys (those start with '_')."""
    return {k: v for k, v in row.items() if not k.startswith("_")}


class GameSession:
    # Build the four simulators (you, autopilot, reactive, scheduled) over the level's demand.
    def __init__(self, level: dict, hourly_views: dict[str, list[float]], constants: Constants,
                 region_forecasts=(), article_forecasts=(), scheduled: dict | None = None, kappa: float = 10.0,
                 session_id: str | None = None, surge_ratio: float | None = None):
        self.level = level
        self.id = session_id or uuid.uuid4().hex[:12]
        self.k = constants
        self.start = parse_utc(level["start"])
        self.minutes = int(level["hours"]) * 60
        D = region_minute_demand({r: list(hourly_views[r])[: int(level["hours"])] for r in REGIONS if r in hourly_views},
                                 constants.SCALE)
        self.D = D
        self.book = ForecastBook(self.start, region_forecasts, article_forecasts)
        # The scheduled baseline needs a per-hour plan; without one, it plans from this level's own hourly demand.
        table = scheduled or scheduled_table(
            {r: [((self.start + timedelta(hours=i)).hour, v) for i, v in enumerate(hourly_views[r])] for r in D}, constants)
        self.sims = {
            "you": Simulator(D, constants, self.start, None, self.book, name="you"),
            "autopilot": Simulator(D, constants, self.start, Autopilot(kappa=kappa), self.book),
            "reactive": Simulator(D, constants, self.start, Reactive(), self.book),
            "scheduled": Simulator(D, constants, self.start, Scheduled(table), self.book),
        }
        self.queue: list[dict] = []
        self.alerted: set = set()
        self.ended = False
        # Regional surge rule for the review (needed servers >= ratio x trailing 3 h median). 1.5 by default; a level
        # package may carry a lower value when whole-region demand is too smooth to show a 1.5x jump.
        self.surge_ratio = float(surge_ratio or level.get("surge_ratio") or 1.5)
        self.min_jump = int(level.get("surge_min_jump") or 2)

    # Load a level package written by `flashcrowd export` (plain JSON, no pandas needed on the server).
    @classmethod
    def from_level_file(cls, path, session_id: str | None = None, kappa: float = 10.0) -> "GameSession":
        pkg = json.loads(open(path, encoding="utf-8").read())
        k = Constants(**{**pkg["constants"], "B": dict(pkg["constants"]["B"])})
        return cls(pkg["level"], pkg["hourly_views"], k, pkg.get("region_forecasts", []), pkg.get("article_forecasts", []),
                   pkg.get("scheduled"), kappa, session_id)

    # POST /sessions response body.
    def info(self) -> dict:
        return {"session_id": self.id, "start": iso(self.start), "end": iso(self.start + timedelta(minutes=self.minutes)),
                "constants": {"C": self.k.C, "B": dict(self.k.B), "SCALE": self.k.SCALE}}

    # POST /sessions/{id}/action: validate now, apply at the start of the next tick.
    def action(self, a: dict) -> dict:
        if self.ended:
            return {"ok": False, "error": "session_over"}
        if a.get("type") != "scale":
            return {"ok": False, "error": "unknown_action"}
        r = a.get("region")
        if r not in self.sims["you"].state:
            return {"ok": False, "error": "unknown_region"}
        try:
            d = int(a.get("delta"))
        except (TypeError, ValueError):
            return {"ok": False, "error": "bad_delta"}
        if d == 0 or abs(d) > MAX_DELTA:
            return {"ok": False, "error": "bad_delta"}
        st = self.sims["you"].state[r]
        pending = sum(x["delta"] for x in self.queue if x["region"] == r)
        fleet = st.running + len(st.booting) + pending
        if d < 0 and fleet <= self.k.MIN_SERVERS:
            return {"ok": False, "error": "min_servers"}
        if d > 0 and fleet >= self.k.MAX_SERVERS:
            return {"ok": False, "error": "max_servers"}
        self.queue.append({"type": "scale", "region": r, "delta": d})
        return {"ok": True}

    # Radar: the latest issue's article forecasts, one row per title (its most likely horizon), top 8.
    def _radar(self, art_vis) -> list[dict]:
        best = {}
        for row in ForecastBook.latest_articles(art_vis, None, k=10_000):
            key = (row.get("wiki"), row.get("title"))
            if key not in best or row["p_surge"] > best[key]["p_surge"]:
                best[key] = row
        return [_public(r) for r in sorted(best.values(), key=lambda r: -float(r["p_surge"]))[:RADAR_N]]

    # Alerts: the first time a title's forecast reaches 0.6, once per title per session.
    def _alerts(self, radar: list[dict]) -> list[dict]:
        out = []
        for r in radar:
            key = (r.get("wiki"), r.get("title"))
            if float(r["p_surge"]) >= ALERT_P and key not in self.alerted:
                self.alerted.add(key)
                name = str(r.get("title", "")).replace("_", " ")
                out.append({"level": "warn", "text": f"Surge likely on '{name}' within {int(r.get('horizon_h', 2))} h, "
                                                     f"mostly {CITY.get(r.get('region'), r.get('region'))}"})
        return out

    # One WebSocket message: apply queued actions, step all four players, then build the tick.
    def tick(self) -> dict:
        if self.ended or self.sims["you"].done:
            self.ended = True
            return {"type": "end", "review_url": f"/api/sessions/{self.id}/review"}
        for a in self.queue:
            self.sims["you"].apply(a)
        self.queue.clear()
        m = self.sims["you"].m
        players = {}
        for name, sim in self.sims.items():
            t = sim.step()
            players[name] = {**({"regions": t} if name == "you" else {}), **sim.score()}
        _, art_vis = self.book.visible(m)
        radar = self._radar(art_vis)
        return {"type": "tick", "m": m, "now": iso(self.start + timedelta(minutes=m)), "players": players,
                "radar": radar, "alerts": self._alerts(radar)}

    def run_to_end(self) -> list[dict]:
        msgs = []
        while not self.ended:
            msgs.append(self.tick())
        return msgs

    # GET /sessions/{id}/review: grades + every regional surge with each player's verdict.
    def review(self) -> dict:
        titles = {}
        for r, D in self.D.items():
            for o in surge_onsets(D, self.k, ratio=self.surge_ratio, min_jump=self.min_jump):
                _, arts = self.book.visible(o)
                top = ForecastBook.latest_articles(arts, r, k=1)
                if top:
                    titles[(r, o)] = str(top[0]["title"]).replace("_", " ")
        out = grade_review(self.sims, self.k, onset_titles=titles, book=self.book, surge_ratio=self.surge_ratio,
                           min_jump=self.min_jump)
        for s in out["surges"]:
            s.pop("onset_m", None)
        return out

    # GET /agent/reasons?region=: the autopilot's latest decision for that region, for the voice agent.
    def reasons(self, region: str) -> dict:
        d = self.sims["autopilot"].policy.decisions.get(region)
        if not d:
            return {"region": region, "action": "hold", "reasons": [], "p_surge": 0.0}
        return {"region": region, "action": d["action"], "reasons": d["reasons"], "p_surge": round(d["p_surge"], 3)}

## 11. Backtest, scorecard and handoff to PC1 / PC2

Every policy through the engine on the held-out days, the live scorecard, and the files PC1 and PC2 receive.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/backtest.py
"""Decision backtest + Pareto chart + boss levels (CONTRACT §7 P2/P3).

    flashcrowd-backtest                 # test days of the champion model, writes their forecasts to the store too

1 forecasts for the test days via forecast_job.batch (same code as live; also gives PC1 the level forecasts)
2 true regional demand V_r(h) -> per-minute requests through flashcrowd_engine.region_minute_demand
3 constants from INFRA_MEASUREMENTS (PC1's measured boot times and capacity); SCALE calibrated on training days
4 every policy through the same simulator: autopilot and persistence at kappa in {2, 5, 10, 20, 50},
  reactive, scheduled, oracle
5 Pareto chart + headline ("at the same server-hours as reactive, __% fewer dropped requests")
6 boss levels: the 3 largest regional onsets in the test days, 6-hour windows -> artifacts/levels.json for /levels
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import pandas as pd

from flashcrowd_engine import Constants, ForecastBook, Simulator, calibrate_scale, from_measurements, region_minute_demand
from flashcrowd_engine.decision import Autopilot, Persistence
from flashcrowd_engine.grading import pareto_points, review
from flashcrowd_engine.policies import Oracle, Reactive, Scheduled, scheduled_table

from . import config as C
from .charts import pareto_chart
from .data import HOUR, load_window
from .forecast_job import batch
from .model import Bundle
from .postprocess import iso
from .regions import regional_views
from .store import Store, open_store

KAPPAS = (2, 5, 10, 20, 50)
CITY = {"LDN": "London", "NJ": "New Jersey", "TYO": "Tokyo"}


# Measured constants if PC1 has written them, else the engine defaults; SCALE so a median training hour needs 4
# servers per region (the contract asks for 2-6).
def constants(store: Store, train_views: pd.DataFrame) -> Constants:
    k = Constants()
    if store.has_table("INFRA_MEASUREMENTS"):
        m = store.query("SELECT kind, region, value, unit FROM INFRA_MEASUREMENTS")
        if len(m):
            k = from_measurements(m, k)
    return calibrate_scale({r: train_views[r].tolist() for r in C.REGIONS}, k, target_servers=4)


# Engine-shaped forecast rows (ISO strings) from the batch output.
def engine_rows(arts: pd.DataFrame, regs: pd.DataFrame) -> tuple[list, list]:
    reg = [{"issued_at": iso(r.issued_at), "region": r.region, "target_hour": iso(r.target_hour),
            "q50_views": float(r.q50_views), "q90_views": float(r.q90_views)} for r in regs.itertuples()]
    art = [{"issued_at": iso(r.issued_at), "wiki": r.wiki, "title": r.title, "region": r.region,
            "horizon_h": int(r.horizon_h), "p_surge": float(r.p_surge), "top_reasons": list(r.top_reasons)}
           for r in arts[arts.horizon_h == 2].itertuples()]
    return reg, art


# Hourly regional onsets: the hour regional views jump to >= 1.3x their same-hour 7-day median.
def regional_onsets(rv: pd.DataFrame, lo, hi, ratio: float = 1.3) -> list[tuple[pd.Timestamp, str, float]]:
    out = []
    for r in C.REGIONS:
        s = rv[r]
        med = pd.concat([s.shift(24 * k) for k in range(1, 8)], axis=1).median(axis=1)
        x = (s / med).loc[lo:hi]
        for h, val in x.items():
            prev = x.get(h - HOUR, 0)
            if val >= ratio and prev < ratio:
                out.append((h, r, float(val)))
    return sorted(out, key=lambda z: -z[2])


# Boss levels for PC1's /levels: 6-hour windows starting 2 h before the 3 biggest regional surges, found with the
# SAME minute-level onset rule the review uses (so every level has at least one graded surge). Falls back to the
# hourly ratio rule if the minute rule finds nothing (very smooth regional demand).
def boss_levels(raw, rv, lo, hi, k: Constants, D: dict, start, n=3) -> list[dict]:
    from flashcrowd_engine.grading import needed_servers, surge_onsets
    cands = []
    for r in C.REGIONS:
        need = needed_servers(D[r], k)
        for o in surge_onsets(D[r], k):
            base = max(1.0, float(np.median(need[max(0, o - 180): o])))
            cands.append((start + pd.Timedelta(minutes=o), r, float(need[o: o + 60].max()) / base))
    if not cands:
        cands = regional_onsets(rv, lo, hi)
    cands = sorted((c for c in cands if lo + 2 * HOUR <= c[0] <= hi - 4 * HOUR), key=lambda z: -z[2])
    levels, used = [], []
    pv = raw.pageviews.assign(region=raw.pageviews.wiki.map(C.WIKI_MAIN_REGION))
    for h, r, val in cands:
        if any(abs((h - u).total_seconds()) < 6 * 3600 for u in used):
            continue
        hh = h.floor("h")
        top = pv[(pv.hour_utc == hh) & (pv.region == r)].nlargest(1, "views")
        name = top.title.iloc[0].replace("_", " ") if len(top) else "Regional"
        start_l = hh - 2 * HOUR
        part = "night" if start_l.hour < 6 else "morning" if start_l.hour < 12 else "afternoon" if start_l.hour < 18 else "evening"
        levels.append({"id": f"{start_l:%b-%d}-{part}-{r}".lower(), "title": f"{name} surge in {CITY[r]}",
                       "start": iso(start_l), "hours": 6, "region": r,
                       "note": f"Servers needed jump {val:.1f}x at {h:%H:%M} UTC (held-out test day)"})
        used.append(h)
        if len(levels) == n:
            break
    return levels


def run(store: Store, bundle: Bundle, write: bool = True, log=print) -> dict:
    # 1: forecasts for the test days (also written to the store for PC1's sessions).
    t0, t1 = (pd.Timestamp(x) for x in bundle.meta["splits"]["test"])
    arts, regs = batch(store, bundle, t0, t1, write=write)
    log(f"[1/6] {len(arts):,} article and {len(regs):,} region forecasts for {t0} -> {t1}")

    # 2: true regional demand for the test window, plus training days for SCALE and the scheduled table.
    d0 = pd.Timestamp(bundle.meta["splits"]["dev"][0])
    raw = load_window(store, t0, t1, history_days=max(8, (t0 - d0).days + 2))
    rv = regional_views(raw)
    train_rv = rv.loc[d0: t0 - HOUR]
    k = constants(store, train_rv)
    start = t0
    hours = pd.date_range(t0, t1, freq="h")
    hourly = {r: rv[r].reindex(hours).fillna(0).to_numpy() for r in C.REGIONS}
    D = region_minute_demand(hourly, k.SCALE)
    yesterday = {r: rv[r].reindex(hours - pd.Timedelta(hours=24)).fillna(0).to_numpy() for r in C.REGIONS}
    table = scheduled_table({r: [(h.hour, v) for h, v in train_rv[r].items()] for r in C.REGIONS}, k)
    log(f"[2/6] constants C={k.C:g} req/s, B={k.B}, SCALE={k.SCALE:.4f} req/view")

    # 3-4: every policy through the same deterministic simulator.
    reg_rows, art_rows = engine_rows(arts, regs)
    book = ForecastBook(start, reg_rows, art_rows)
    results, sims = [], {}
    for kap in KAPPAS:
        for fam, pol in (("autopilot", Autopilot(kappa=kap)), ("persistence", Persistence(yesterday, start, kappa=kap))):
            s = Simulator(D, k, start, pol, forecasts=book).run()
            results.append({"family": fam, "kappa": kap, **s.score()})
            sims[(fam, kap)] = s
    for fam, pol in (("reactive", Reactive()), ("scheduled", Scheduled(table)), ("oracle", Oracle(D))):
        s = Simulator(D, k, start, pol, forecasts=book).run()
        results.append({"family": fam, "kappa": None, **s.score()})
        sims[(fam, None)] = s
    log("[3/6] " + " | ".join(f"{r['family']}{'' if r['kappa'] is None else ' k=' + str(r['kappa'])}: "
                              f"{r['served_pct']:.2f}% served, {r['server_hours']:.0f} sh" for r in results))

    # 5: headline. If reactive drops requests, compare dropped % at reactive's server-hours (read off the
    # autopilot curve by linear interpolation). If nobody drops anything (smooth hourly demand, fast boots), the
    # honest comparison is cost: the cheapest autopilot setting that serves at least as much as reactive.
    react = next(r for r in results if r["family"] == "reactive")
    ap = sorted([r for r in results if r["family"] == "autopilot"], key=lambda r: r["server_hours"])
    xs = [r["server_hours"] for r in ap]
    ys = [100 - r["served_pct"] for r in ap]
    re_drop = 100 - react["served_pct"]
    inside = xs[0] <= react["server_hours"] <= xs[-1]
    ap_drop = float(np.interp(react["server_hours"], xs, ys))
    fewer = (1 - ap_drop / re_drop) * 100 if re_drop > 1e-6 else None
    ok = [r for r in ap if r["served_pct"] >= react["served_pct"] - 1e-9]
    cheaper = min(ok, key=lambda r: r["server_hours"]) if ok else None
    saving = (1 - cheaper["server_hours"] / react["server_hours"]) * 100 if cheaper else None
    if fewer is not None and fewer > 0:
        headline = (f"At the same server-hours as the reactive autoscaler, FLASHCROWD drops {fewer:.0f}% fewer requests "
                    f"({ap_drop:.2f}% vs {re_drop:.2f}%)" + ("" if inside else " [extrapolated]"))
    elif saving is not None:
        headline = (f"Serving as much as the reactive autoscaler ({react['served_pct']:.2f}%), FLASHCROWD uses "
                    f"{saving:.0f}% fewer server-hours ({cheaper['server_hours']:.0f} vs {react['server_hours']:.0f}, "
                    f"kappa={cheaper['kappa']})")
    else:
        # No setting both matches reactive's service and costs less: state the trade-off of the cheapest-per-drop setting.
        best = min(ap, key=lambda r: r["server_hours"] * (1 + (100 - r["served_pct"])))
        save = (1 - best["server_hours"] / react["server_hours"]) * 100
        headline = (f"FLASHCROWD (kappa={best['kappa']}) serves {best['served_pct']:.2f}% vs reactive's "
                    f"{react['served_pct']:.2f}%, using {save:.0f}% {'fewer' if save >= 0 else 'more'} server-hours "
                    f"({best['server_hours']:.0f} vs {react['server_hours']:.0f})")
    pts = pareto_points(results)
    chart = pareto_chart(pts, "Lower-left wins: server-hours vs dropped requests (held-out days)", C.CHART_DIR)
    log(f"[4/6] {headline}")

    # 6: boss levels + a sample review (PC2 can use it as a realistic mock).
    levels = boss_levels(raw, rv, t0, t1, k, D, start)
    (C.ARTIFACTS / "levels.json").write_text(json.dumps(levels, indent=2, ensure_ascii=False))
    sample = None
    if levels:
        L = levels[0]
        ls = pd.Timestamp(L["start"].replace("Z", ""))
        hrs = pd.date_range(ls, ls + 5 * HOUR, freq="h")
        Dl = region_minute_demand({r: rv[r].reindex(hrs).fillna(0).to_numpy() for r in C.REGIONS}, k.SCALE)
        y_l = {r: rv[r].reindex(hrs - pd.Timedelta(hours=24)).fillna(0).to_numpy() for r in C.REGIONS}
        lsims = {"autopilot": Simulator(Dl, k, ls, Autopilot(kappa=10), forecasts=book).run(),
                 "reactive": Simulator(Dl, k, ls, Reactive(), forecasts=book).run(),
                 "scheduled": Simulator(Dl, k, ls, Scheduled(table), forecasts=book).run(),
                 "persistence": Simulator(Dl, k, ls, Persistence(y_l, ls), forecasts=book).run()}
        sample = review(lsims, k, book=book)
        (C.ARTIFACTS / "sample_review.json").write_text(json.dumps(sample, indent=2, default=str))
    log(f"[5/6] boss levels: {[l['id'] for l in levels]}")

    out = {"window": [str(t0), str(t1)], "constants": k.to_json(), "results": results, "headline": headline,
           "equal_server_hours": {"reactive_server_hours": react["server_hours"], "autopilot_dropped_pct": ap_drop,
                                  "reactive_dropped_pct": re_drop, "fewer_dropped_pct": fewer, "interpolated": inside},
           "equal_service": {"autopilot": cheaper, "reactive_server_hours": react["server_hours"],
                             "server_hours_saving_pct": saving},
           "levels": levels, "scheduled_table": table, "pareto_chart": str(chart), "model_version": bundle.version}
    (C.ARTIFACTS / "backtest.json").write_text(json.dumps(out, indent=2, default=str))
    log(f"[6/6] wrote {C.ARTIFACTS / 'backtest.json'}, levels.json, sample_review.json, charts/pareto.png")
    return out


def main(argv=None):
    ap = argparse.ArgumentParser(description="Backtest every policy on the held-out days")
    ap.add_argument("--no-write", action="store_true", help="do not write the test-day forecasts to the store")
    a = ap.parse_args(argv)
    run(open_store(), Bundle.load(), write=not a.no_write)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/scorecard.py
"""Live scorecard (CONTRACTS §3.5 scorecard object): were our forecasts right once Wikipedia published the hours?

    flashcrowd scorecard --since 2026-10-24T14:00      -> artifacts/live/scorecard.json (PC1 can serve it as-is)

For each (article, issue time) we take the radar's row (the horizon with the highest p_surge):
  hit          p >= alert threshold and an onset happened in (issued_at, issued_at + h]
  false_alarm  p >= threshold and no onset
  miss         p <  threshold but an onset happened
A forecast is only checked once every hour in its window has been PUBLISHED (published_at <= now), using the same
surge rule as the model (SURGE_HOURS view, or the Python rule when the view does not exist).
"""
from __future__ import annotations

import argparse
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from . import config as C
from .data import HOUR
from .labels import onset_flags, surge_flags
from .store import Store, open_store, ts_literal


def _iso(t) -> str:
    return pd.Timestamp(t).strftime("%Y-%m-%dT%H:%M:%SZ")


# Truth for a set of titles: views per hour (8 days back for the 7-day median) and onset hours.
def _truth(store: Store, titles: pd.DataFrame, h0, h1) -> tuple[pd.DataFrame, set]:
    store.register_temp(titles[["wiki", "title"]].drop_duplicates(), "FC_SC_TITLES")
    pv = store.query(f"""SELECT p.hour_utc, p.wiki, p.title, p.views FROM PAGEVIEWS_HOURLY p
                         JOIN FC_SC_TITLES t ON t.wiki = p.wiki AND t.title = p.title
                         WHERE p.hour_utc >= {ts_literal(h0 - pd.Timedelta(days=8))} AND p.hour_utc <= {ts_literal(h1)}""")
    pv["hour_utc"] = pd.to_datetime(pv.hour_utc)
    onsets = set()
    base = h0 - pd.Timedelta(days=8)
    n = int((h1 - base) / HOUR) + 1
    for (w, t), g in pv.groupby(["wiki", "title"]):
        v = np.zeros(n)
        v[((g.hour_utc - base) / HOUR).astype(int).to_numpy()] = g.views.to_numpy()
        for i in np.flatnonzero(onset_flags(surge_flags(v, w))):
            onsets.add((w, t, base + i * HOUR))
    return pv, onsets


def build(store: Store, since, threshold: float, now=None, limit: int = 100) -> dict:
    since = pd.Timestamp(since)
    now = pd.Timestamp(now) if now is not None else pd.Timestamp(datetime.now(timezone.utc)).tz_localize(None)
    f = store.query(f"""SELECT issued_at, wiki, title, region, horizon_h, p_surge FROM FORECASTS
                        WHERE issued_at >= {ts_literal(since)} AND issued_at <= {ts_literal(now)}""")
    empty = {"since": _iso(since), "checked": 0, "hits": 0, "misses": 0, "false_alarms": 0, "median_lead_h": None, "rows": []}
    if f.empty:
        return empty
    f["issued_at"] = pd.to_datetime(f.issued_at)
    # The radar's row per article and issue time: the horizon with the highest probability.
    f = f.sort_values("p_surge", ascending=False).drop_duplicates(["wiki", "title", "issued_at"])
    f["w0"] = f.issued_at.dt.floor("h")
    f["w1"] = f.w0 + (f.horizon_h - 1) * HOUR

    # Only windows whose hours have all been published by now can be verified.
    hp = store.query(f"""SELECT hour_utc, MAX(published_at) AS published_at FROM PAGEVIEWS_HOURLY
                         WHERE hour_utc >= {ts_literal(since - HOUR)} GROUP BY hour_utc""")
    hp = pd.Series(pd.to_datetime(hp.published_at).to_numpy(), index=pd.to_datetime(hp.hour_utc)).sort_index()
    pub = hp[hp <= now]
    if pub.empty:
        return empty
    last_pub = pub.index.max()
    f = f[f.w1 <= last_pub]
    if f.empty:
        return empty
    pv, onsets = _truth(store, f, f.w0.min(), f.w1.max())

    rows, leads = [], []
    for r in f.itertuples():
        hits = [o for (w, t, o) in onsets if w == r.wiki and t == r.title and r.w0 <= o <= r.w1]
        positive = r.p_surge >= threshold
        if positive and hits:
            outcome = "hit"
            leads.append((min(hits) - r.issued_at) / HOUR)
        elif positive:
            outcome = "false_alarm"
        elif hits:
            outcome = "miss"
        else:
            continue                                                  # confident "no surge" that was right: not listed
        g = pv[(pv.wiki == r.wiki) & (pv.title == r.title) & (pv.hour_utc >= r.w0) & (pv.hour_utc <= r.w1)]
        rows.append({"issued_at": _iso(r.issued_at), "wiki": r.wiki, "title": r.title, "p_surge": round(float(r.p_surge), 3),
                     "horizon_h": int(r.horizon_h), "outcome": outcome,
                     "actual_peak_views": int(g.views.max()) if len(g) else 0,
                     "verified_at": _iso(pub[pub.index <= r.w1].max())})
    c = pd.Series([x["outcome"] for x in rows]).value_counts() if rows else pd.Series(dtype=int)
    rows.sort(key=lambda x: x["issued_at"], reverse=True)
    return {"since": _iso(since), "checked": int(len(f)), "hits": int(c.get("hit", 0)), "misses": int(c.get("miss", 0)),
            "false_alarms": int(c.get("false_alarm", 0)),
            "median_lead_h": round(float(np.median(leads)), 2) if leads else None, "rows": rows[:limit]}


def main(argv=None):
    ap = argparse.ArgumentParser(description="Score published forecasts against what Wikipedia then published")
    ap.add_argument("--since", required=True)
    ap.add_argument("--now", help="pretend 'now' (for replays); default: the real time")
    a = ap.parse_args(argv)
    from .model import Bundle
    out = build(open_store(), a.since, Bundle.load().threshold, a.now)
    C.LIVE_DIR.mkdir(parents=True, exist_ok=True)
    p = C.LIVE_DIR / "scorecard.json"
    p.write_text(json.dumps(out, indent=1, ensure_ascii=False))
    print(f"{out['checked']} checked: {out['hits']} hits, {out['misses']} misses, {out['false_alarms']} false alarms, "
          f"median lead {out['median_lead_h']} h -> {p}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/handoff.py
"""Everything PC1 and PC2 need from PC3, generated from the trained model and the real (or local) data.

    flashcrowd export            (after `flashcrowd backtest`)

artifacts/handoff/
  pc1/levels.json               GET /levels body (CONTRACTS §3.5)
  pc1/levels/<id>.json          one package per boss level: GameSession.from_level_file(...) needs nothing else
  pc2/mocks/                    the seven mock files PC2's ?mock=1 mode reads (CONTRACTS §3.6), produced by
                                actually playing the first level through the engine, so every number is consistent
  pc2/charts/                   PNGs for the "How it works" screen
  README.md                     what is where
"""
from __future__ import annotations

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from flashcrowd_engine import Constants, GameSession

from . import config as C
from .data import HOUR, load_window
from .model import Bundle
from .regions import regional_views
from .scorecard import build as build_scorecard
from .store import Store, open_store, ts_literal


def _iso(t) -> str:
    return pd.Timestamp(t).strftime("%Y-%m-%dT%H:%M:%SZ")


def _dump(path: Path, obj) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=str))
    return path


# Forecast rows for a level window, straight from the store, in the engine's (CONTRACTS §3.4) shape.
# Only the top `top` titles per issue time are kept: the radar shows 8 and the reasons use the top 3 per region.
def level_forecasts(store: Store, start: pd.Timestamp, end: pd.Timestamp, top: int = 40) -> tuple[list, list]:
    lo, hi = ts_literal(start - 3 * HOUR), ts_literal(end)
    regs = store.query(f"""SELECT issued_at, region, target_hour, q50_views, q90_views FROM REGION_FORECASTS
                           WHERE issued_at >= {lo} AND issued_at <= {hi}""")
    arts = store.query(f"""SELECT issued_at, wiki, title, region, horizon_h, p_surge, size_q50, size_q90, cause, top_reasons
                           FROM FORECASTS WHERE issued_at >= {lo} AND issued_at <= {hi}""")
    if len(arts):
        # Rank TITLES (not probability values: isotonic calibration produces many ties) within each issue time.
        t = arts.groupby(["issued_at", "wiki", "title"], as_index=False).p_surge.max()
        t["rank"] = t.groupby("issued_at").p_surge.rank(method="first", ascending=False)
        arts = arts.merge(t.loc[t["rank"] <= top, ["issued_at", "wiki", "title"]], on=["issued_at", "wiki", "title"])
    reg_rows = [{"issued_at": _iso(r.issued_at), "region": r.region, "target_hour": _iso(r.target_hour),
                 "q50_views": float(r.q50_views), "q90_views": float(r.q90_views)} for r in regs.itertuples()]
    art_rows = []
    for r in arts.itertuples():
        reasons = r.top_reasons
        if isinstance(reasons, str):
            try:
                reasons = json.loads(reasons)
            except ValueError:
                reasons = [reasons]
        art_rows.append({"issued_at": _iso(r.issued_at), "wiki": r.wiki, "title": r.title, "region": r.region,
                         "horizon_h": int(r.horizon_h), "p_surge": round(float(r.p_surge), 4),
                         "size_q50": round(float(r.size_q50 or 0)), "size_q90": round(float(r.size_q90 or 0)),
                         "cause": r.cause if isinstance(r.cause, str) else None, "top_reasons": list(reasons or [])})
    return reg_rows, art_rows


# One self-contained level package: demand, constants, forecasts and the scheduled plan.
def level_package(store: Store, level: dict, constants: dict, scheduled: dict | None) -> dict:
    """The level carries `surge_ratio`: 1.5 (the engine default) unless the level shows no regional surge at 1.5,
    then the largest of 1.3 / 1.2 that does. The README lists which levels needed the lower ratio."""
    start = pd.Timestamp(level["start"].replace("Z", ""))
    end = start + (int(level["hours"]) - 1) * HOUR
    raw = load_window(store, start, end, history_days=2, label_hours=0)
    rv = regional_views(raw)
    hours = pd.date_range(start, end, freq="h")
    hourly = {r: [float(x) for x in rv[r].reindex(hours).fillna(0)] for r in C.REGIONS}
    reg, art = level_forecasts(store, start, end + HOUR)
    from flashcrowd_engine import region_minute_demand
    from flashcrowd_engine.grading import surge_onsets
    k = Constants(**{**constants, "B": dict(constants["B"])})
    D = region_minute_demand(hourly, k.SCALE)
    level = dict(level)
    # Keep the engine default (1.5x and +2 servers) when it finds a surge in this level. Otherwise use the biggest
    # jump the level really contains (needed servers vs the trailing 3 h median), floored at 1.1x and +1 server.
    from flashcrowd_engine.grading import needed_servers
    if any(surge_onsets(D[r], k) for r in D):
        level["surge_ratio"], level["surge_min_jump"] = 1.5, 2
    else:
        best = 1.0
        for r in D:
            need = needed_servers(D[r], k)
            for m in range(30, len(need)):
                base = float(np.median(need[max(0, m - 180): m]))
                if need[m] - base >= 1:
                    best = max(best, need[m] / max(base, 1.0))
        level["surge_ratio"] = round(max(1.1, min(1.5, np.floor(best * 20) / 20)), 2)
        level["surge_min_jump"] = 1
    return {"level": level, "constants": constants, "hourly_views": hourly, "scheduled": scheduled,
            "region_forecasts": reg, "article_forecasts": art}


# A believable human for the mock ticks: reacts 8 minutes after the first alert, then tops up once.
def scripted_player(session: GameSession, ticks: list) -> None:
    plan = {}
    for t in ticks:
        if t.get("alerts") and not plan:
            region = next((r["region"] for r in t["radar"] if r["p_surge"] >= 0.6), "LDN")
            plan = {t["m"] + 8: (region, 3), t["m"] + 15: (region, 2)}
    session_plan = plan or {30: ("LDN", 2)}
    return session_plan


def export(store: Store, bundle: Bundle, out: Path | None = None) -> dict:
    out = Path(out or C.ARTIFACTS / "handoff")
    bt_path = C.ARTIFACTS / "backtest.json"
    if not bt_path.exists():
        raise RuntimeError("run `flashcrowd backtest` first (it picks the boss levels and the constants)")
    bt = json.loads(bt_path.read_text())
    levels = [{k: v for k, v in l.items() if k in ("id", "title", "start", "hours", "note")} for l in bt["levels"]]
    k = bt["constants"]
    sched = bt.get("scheduled_table")

    # PC1: /levels body + one package per level.
    _dump(out / "pc1" / "levels.json", levels)
    pkgs = []
    for lv in levels:
        pkg = level_package(store, lv, k, sched)
        pkgs.append(pkg)
        _dump(out / "pc1" / "levels" / f"{lv['id']}.json", pkg)

    # PC2: play the first level through the engine twice: once to find the first alert, then with the scripted player.
    files = {}
    if pkgs:
        p0 = out / "pc1" / "levels" / f"{levels[0]['id']}.json"
        probe = GameSession.from_level_file(p0, session_id="mock")
        plan = scripted_player(probe, probe.run_to_end())
        s = GameSession.from_level_file(p0, session_id="abc123")
        ticks = []
        while not s.ended:
            for (r, d) in [plan[m] for m in plan if m == s.sims["you"].m]:
                s.action({"type": "scale", "region": r, "delta": d})
            ticks.append(s.tick())
        mocks = out / "pc2" / "mocks"
        mocks.mkdir(parents=True, exist_ok=True)
        files["levels.json"] = _dump(mocks / "levels.json", levels)
        files["session.json"] = _dump(mocks / "session.json", s.info())
        (mocks / "ticks.jsonl").write_text("\n".join(json.dumps(t, ensure_ascii=False) for t in ticks) + "\n")
        files["ticks.jsonl"] = mocks / "ticks.jsonl"
        files["review.json"] = _dump(mocks / "review.json", s.review())
        region = pkgs[0]["level"].get("region") or bt["levels"][0].get("region", "LDN")
        files["reasons.json"] = _dump(mocks / "reasons.json", s.reasons(region))
        last = max((a["issued_at"] for a in pkgs[0]["article_forecasts"]), default=None)
        latest = sorted([a for a in pkgs[0]["article_forecasts"] if a["issued_at"] == last], key=lambda a: -a["p_surge"])[:20]
        files["forecasts.json"] = _dump(mocks / "forecasts.json", latest)
        t0, t1 = bt["window"]
        files["scorecard.json"] = _dump(mocks / "scorecard.json",
                                        build_scorecard(store, t0, bundle.threshold, now=pd.Timestamp(t1) + 6 * HOUR))

    # PC2: charts for "How it works".
    charts = out / "pc2" / "charts"
    charts.mkdir(parents=True, exist_ok=True)
    for png in C.CHART_DIR.glob("*.png"):
        shutil.copy(png, charts / png.name)

    (out / "README.md").write_text(f"""# FLASHCROWD handoff (model {bundle.version})

## PC1 (backend)
- `pc1/levels.json` is the body of `GET /levels`.
- `pc1/levels/<id>.json` is one file per level. In `sessions.py`:
  `GameSession.from_level_file(path, session_id)`, then `.info()`, `.action()`, `.tick()`, `.review()`, `.reasons(region)`
  (`pip install "git+https://github.com/<ORG>/flashcrowd-model.git@engine-v0.3"`).
- Live tables: `FORECASTS`, `REGION_FORECASTS` (every 15 min), the `SURGE_HOURS` view (scorecard truth).

## PC2 (frontend)
- `pc2/mocks/` holds the seven files for `?mock=1`. They come from one real playthrough of level `{levels[0]['id'] if levels else '-'}`:
  {len(files.get('ticks.jsonl', '').read_text().splitlines()) if files.get('ticks.jsonl') else 0} tick messages, then the end message.
  The "you" player reacts 8 minutes after the first alert, so the review shows a realistic verdict.
- `pc2/charts/` holds the PNGs for "How it works".

## Surge rule per level
The review's regional surge rule is "servers needed jump to >= ratio x the trailing 3 h median".
{chr(10).join(f"- `{p['level']['id']}`: ratio {p['level']['surge_ratio']}, at least +{p['level']['surge_min_jump']} server(s)" for p in pkgs)}
A ratio below 1.5 means whole-region demand was smooth that day (see INTEGRATION.md §6).
""")
    return {"out": str(out), "levels": [l["id"] for l in levels], "mocks": sorted(files), "charts": sorted(p.name for p in charts.glob("*.png"))}


def main(argv=None):
    res = export(open_store(), Bundle.load())
    print(f"handoff -> {res['out']}\n  levels: {res['levels']}\n  mocks: {res['mocks']}\n  charts: {res['charts']}")


if __name__ == "__main__":
    main()

## 12. Offline data and the terminal

Synthetic data in PC1's exact table shapes (so everything runs without Snowflake) and the `flashcrowd` terminal menu.

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/synth.py
"""Synthetic data in PC1's exact table shapes (CONTRACTS §3.2), so the whole real-data pipeline runs offline.

Writes EDITS, PAGEVIEWS_HOURLY (with published_at ~ hour + 2 h, some files late), MAIN_PAGE_EVENTS (daily TFA,
ITN items), COUNTRY_SHARE, WIKIDATA_ITEMS and INFRA_MEASUREMENTS into a local DuckDB file.

Behaviour it plants (each one tests a real failure mode):
  news stories across wikis with edits leading views 0-3 h (some silent), deaths/sports/releases in comments,
  ITN items (views follow 30-90 min after the template edit), TFA (scheduled all-day surge),
  edit wars and vandalism (reverts, no traffic), bot bursts (flagged, no traffic), new pages.
"""
from __future__ import annotations

import argparse
from dataclasses import dataclass

import numpy as np
import pandas as pd

from . import config as C
from .store import DuckDBStore

# Edit summaries per story type, so the Gemini labeller has realistic text to classify.
STORY_WORDS = {"breaking_news": ["breaking: {x}", "update casualties", "add latest reports", "per Reuters"],
               "death": ["died today", "add death date", "obituary refs", "death confirmed"],
               "sports": ["final score", "match result", "add goal", "won the title"],
               "entertainment": ["release date", "trailer", "box office", "award win"]}
ROUTINE = ["copyedit", "fix typo", "ref", "format", "update infobox", "link", "cleanup", ""]
TZ_OFF = {"LDN": 0, "NJ": -5, "TYO": 9}


@dataclass
class SynthConfig:
    titles_per_wiki: int = 40
    days_views: int = 21          # page views loaded (7 extra days before the edits start)
    days_edits: int = 14
    stories: int = 70
    start: str = "2026-09-26 00:00:00"
    seed: int = C.SEED


def generate(cfg: SynthConfig = SynthConfig()) -> dict[str, pd.DataFrame]:
    rng = np.random.default_rng(cfg.seed)
    t0 = pd.Timestamp(cfg.start)
    H = cfg.days_views * 24
    edit_start_h = (cfg.days_views - cfg.days_edits) * 24

    # ---- articles: concepts (Wikidata items) with a page on several wikis
    n_concepts = int(cfg.titles_per_wiki * len(C.WIKIS) / 2.5)
    pages = []
    for q in range(n_concepts):
        wikis = [w for w in C.WIKIS if rng.random() < (0.8 if w == "enwiki" else 0.18)] or ["enwiki"]
        for w in wikis:
            pages.append((w, f"Item_{q}_{w[:2]}", f"Q{5000 + q}"))
    pages = pd.DataFrame(pages, columns=["wiki", "title", "qid"]).drop_duplicates(["wiki", "title"])
    P = len(pages)
    pidx = {(w, t): i for i, (w, t) in enumerate(zip(pages.wiki, pages.title))}
    by_qid = pages.groupby("qid").apply(lambda g: [pidx[(w, t)] for w, t in zip(g.wiki, g.title)],
                                        include_groups=False).to_dict()

    # Baseline traffic: heavy-tailed level x reading-hour curve of the wiki's main region.
    level = rng.lognormal(5.6, 1.0, P)
    level[pages.wiki.to_numpy() == "enwiki"] *= 2.5
    hours = np.arange(H)
    reg = np.array([C.WIKI_MAIN_REGION[w] for w in pages.wiki])
    local = (hours[None, :] + np.array([TZ_OFF[r] for r in reg])[:, None]) % 24
    diurnal = np.vectorize(C.hour_curve)(local)
    mult = np.ones((P, H))
    edits: list[tuple] = []
    main_page: list[tuple] = []
    users = [f"User{i}" for i in range(8000)]
    new_user = iter(range(10**6))

    # Generate edits for one page over a span of hours (routine, story, war, vandalism or bot).
    def add_edits(p, h0, h1, rate, kind="routine", humans=True, revert_p=0.02, bot=False, fixed_users=None):
        w, t = pages.wiki.iat[p], pages.title.iat[p]
        for h in range(max(h0, edit_start_h), min(h1, H)):
            for _ in range(rng.poisson(rate)):
                if bot:
                    u = f"Bot{rng.integers(0, 9)}"
                elif fixed_users:
                    u = fixed_users[rng.integers(0, len(fixed_users))]
                else:
                    u = f"Editor{next(new_user)}" if humans and rng.random() < 0.7 else users[rng.integers(0, 8000)]
                words = STORY_WORDS.get(kind, ROUTINE)
                comment = rng.choice(words) if kind in STORY_WORDS else rng.choice(ROUTINE)
                rev = rng.random() < revert_p
                if rev:
                    comment = rng.choice(["Reverted edits by " + u, "rv vandalism", "Undid revision 12345"])
                edits.append((t0 + pd.Timedelta(hours=float(h + rng.random())), w, t, 0, "edit", u, bot, rev,
                              int(rng.integers(-300, 800)) if not rev else int(rng.integers(-2000, -10)),
                              comment.replace("{x}", t)))

    # A surge: views multiply then decay; edits start `lead` hours before (0 = a silent surge).
    def surge(p, onset, peak, tau, lead, rate, kind, new_page=False):
        j = np.arange(H - onset)
        mult[p, onset:] *= 1 + (peak - 1) * np.exp(-j / tau)
        if new_page:
            w, t = pages.wiki.iat[p], pages.title.iat[p]
            edits.append((t0 + pd.Timedelta(hours=float(onset - lead - 0.5)), w, t, 0, "new", f"Editor{next(new_user)}",
                          False, False, 4000, "Created page"))
        add_edits(p, onset - lead, onset + 5, rate, kind)

    lo, hi = edit_start_h + 6, H - 8
    for _ in range(cfg.stories):                                     # cross-wiki news stories
        q = f"Q{5000 + int(rng.integers(0, n_concepts))}"
        onset = int(rng.integers(lo, hi))
        kind = rng.choice(list(STORY_WORDS))
        silent = rng.random() < 0.2
        new_page = rng.random() < 0.15
        for p in by_qid[q]:
            surge(p, onset + int(rng.integers(0, 2)), float(np.exp(rng.uniform(np.log(8), np.log(60)))),
                  float(rng.uniform(2, 8)), 0 if silent else int(rng.integers(1, 4)), float(rng.uniform(3, 10)),
                  kind, new_page)

    # Today's Featured Article: a scheduled all-day surge announced at 00:00.
    for d in range(cfg.days_views):                                  # Today's Featured Article (scheduled)
        en = np.flatnonzero(pages.wiki.to_numpy() == "enwiki")
        p = int(rng.choice(en))
        h0 = d * 24
        mult[p, h0: h0 + 24] *= 6.0
        main_page.append((t0 + pd.Timedelta(hours=float(h0)), "enwiki", pages.title.iat[p], "tfa"))
    for _ in range(cfg.days_views * 2):                              # In the news (views follow the template edit)
        en = np.flatnonzero(pages.wiki.to_numpy() == "enwiki")
        p = int(rng.choice(en))
        h = int(rng.integers(lo, hi))
        ts = t0 + pd.Timedelta(hours=float(h + rng.uniform(0, 0.5)))
        main_page.append((ts, "enwiki", pages.title.iat[p], "itn"))
        surge(p, h + 1, float(rng.uniform(6, 15)), 8.0, 0, 2.0, "breaking_news")

    # Noise that must NOT predict surges: background edits, edit wars, vandalism, bot bursts.
    for p in range(P):                                               # noise that must NOT predict surges
        add_edits(p, 0, H, 0.04)
        for _ in range(rng.poisson(0.8)):                            # edit war
            h = int(rng.integers(edit_start_h, H - 3))
            pair = [users[i] for i in rng.integers(0, 8000, 2)]
            add_edits(p, h, h + 3, 8, revert_p=0.45, fixed_users=pair)
        for _ in range(rng.poisson(0.6)):                            # vandalism
            h = int(rng.integers(edit_start_h, H - 2))
            add_edits(p, h, h + 1, 6, revert_p=0.5)
        for _ in range(rng.poisson(1.2)):                            # bot burst
            h = int(rng.integers(edit_start_h, H - 2))
            add_edits(p, h, h + 2, 25, bot=True)

    lam = level[:, None] * diurnal * mult
    views = rng.poisson(lam)
    # Dumps publish ~2 h after the hour; 5% are 2 h late, to exercise the published_at logic.
    pub_delay = 120 + rng.uniform(0, 25, H)
    pub_delay[rng.random(H) < 0.05] += 120                           # some dump files publish late
    rows = []
    for p in range(P):
        keep = views[p] >= 20                                        # loader drops rows with < 20 views
        hh = hours[keep]
        rows.append(pd.DataFrame({"hour_utc": t0 + pd.to_timedelta(hh, unit="h"), "wiki": pages.wiki.iat[p],
                                  "title": pages.title.iat[p], "views": views[p][keep]}))
    pv = pd.concat(rows, ignore_index=True)
    hidx = ((pv.hour_utc - t0) / pd.Timedelta(hours=1)).astype(int)
    pv["published_at"] = pv.hour_utc + pd.to_timedelta(pub_delay[hidx], unit="min")

    ed = pd.DataFrame(edits, columns=["ts", "wiki", "title", "namespace", "type", "user_name", "is_bot", "is_revert",
                                      "delta_bytes", "comment"])
    ed = ed[(ed.ts >= t0 + pd.Timedelta(hours=float(edit_start_h))) & (ed.ts < t0 + pd.Timedelta(hours=float(H)))]
    ed = ed.sort_values("ts").reset_index(drop=True)
    mp = pd.DataFrame(main_page, columns=["ts", "wiki", "title", "slot"])

    # country shares for busy enwiki titles (top-per-country only lists popular articles)
    en_pv = pv[pv.wiki == "enwiki"].assign(day=pv.hour_utc.dt.date)
    daily = en_pv.groupby(["day", "title"]).views.sum().reset_index()
    busy = daily[daily.views > 30000]
    cs = []
    for r in busy.itertuples():
        w = rng.dirichlet([3.5, 4.5, 2.0])
        for reg_, s in zip(C.REGIONS, w):
            cs.append((pd.Timestamp(r.day), "enwiki", r.title, reg_, float(s)))
    cs = pd.DataFrame(cs, columns=["day", "wiki", "title", "region", "share"])

    # A few boot and capacity rows so the engine's from_measurements path runs offline too.
    infra = []
    for reg_, med in (("LDN", 170), ("NJ", 150), ("TYO", 230)):
        for _ in range(5):
            infra.append((t0, "boot", reg_, None, float(rng.normal(med, 15)), "s", "synthetic"))
    infra.append((t0, "capacity", "LDN", None, 240.0, "req/s", "synthetic"))
    infra = pd.DataFrame(infra, columns=["ts", "kind", "region", "peer", "value", "unit", "note"])
    wd = pages[["wiki", "title", "qid"]]
    return {"EDITS": ed, "PAGEVIEWS_HOURLY": pv, "MAIN_PAGE_EVENTS": mp, "COUNTRY_SHARE": cs,
            "INFRA_MEASUREMENTS": infra, "WIKIDATA_ITEMS": wd}


def write(tables: dict[str, pd.DataFrame], path=C.LOCAL_DB) -> DuckDBStore:
    if path.exists():
        path.unlink()
    st = DuckDBStore(path)
    for name, df in tables.items():
        st.insert(df, name)
    return st


def main(argv=None):
    ap = argparse.ArgumentParser(description="Write synthetic FLASHCROWD tables to a local DuckDB file")
    ap.add_argument("--titles-per-wiki", type=int, default=SynthConfig.titles_per_wiki)
    ap.add_argument("--seed", type=int, default=C.SEED)
    a = ap.parse_args(argv)
    t = generate(SynthConfig(titles_per_wiki=a.titles_per_wiki, seed=a.seed))
    write(t)
    print({k: len(v) for k, v in t.items()}, "->", C.LOCAL_DB)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile flashcrowd-model/src/flashcrowd_model/cli.py
"""`flashcrowd` — the FLASHCROWD terminal. Run `flashcrowd` for the menu, or `flashcrowd <command>` directly.
(Not `fc`: that name is a built-in shell command.)

    flashcrowd             interactive menu
    flashcrowd doctor          check Python, packages, credentials and the data store
    flashcrowd check           what PC1 has loaded, enough days to train?, can PC3 write?
    flashcrowd setup           create the SURGE_HOURS view (once)
    flashcrowd bootstrap       check, setup, train, backtest, export, first live forecast
    flashcrowd scorecard SINCE     hits / misses / false alarms -> artifacts/live/scorecard.json
    flashcrowd refine [--apply]    diagnose the champion; --apply retrains with the fixes (gate decides)
    flashcrowd export          PC1 level packages + PC2 mocks and charts -> artifacts/handoff/
    flashcrowd status          champion model, recent runs, latest forecast file
    flashcrowd test | synth | train | backtest | live [--at T] | batch START END | gemini START END | sql START END
    flashcrowd store local|snowflake     switch where data is read from and written to (saved in .env)
"""
from __future__ import annotations

import importlib
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

from . import config as C

ENV_FILE = C.ROOT / ".env"
BOLD, DIM, GREEN, YELLOW, RED, RESET = ("\033[1m", "\033[2m", "\033[32m", "\033[33m", "\033[31m", "\033[0m") \
    if sys.stdout.isatty() and os.name != "nt" or os.environ.get("WT_SESSION") else ("",) * 6


# Read .env into the process environment so every command sees the same store and keys.
def load_env() -> None:
    if ENV_FILE.exists():
        for line in ENV_FILE.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    os.environ.setdefault("FLASHCROWD_STORE", f"duckdb:{C.LOCAL_DB}")


# Run one pipeline step as a child process so its output streams live and Ctrl+C stops only that step.
def run(module: str, *args: str) -> int:
    cmd = [sys.executable, "-W", "ignore", "-m", module, *args]
    print(f"{DIM}$ {' '.join(shlex.quote(c) for c in cmd[3:])}{RESET}")
    env = {**os.environ, "PYTHONPATH": str(C.ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")}
    try:
        return subprocess.call(cmd, cwd=C.ROOT, env=env)
    except KeyboardInterrupt:
        print(f"\n{YELLOW}stopped{RESET}")
        return 130


# ------------------------------------------------------------------ doctor / status
def _ok(flag: bool, label: str, detail: str = "") -> bool:
    mark = f"{GREEN}ok  {RESET}" if flag else f"{RED}fail{RESET}"
    print(f"  {mark} {label}{DIM}  {detail}{RESET}")
    return flag


def doctor() -> int:
    print(f"{BOLD}Python and packages{RESET}")
    good = _ok(sys.version_info >= (3, 10), "Python >= 3.10", sys.version.split()[0])
    for mod in ("numpy", "pandas", "duckdb", "sklearn", "lightgbm", "matplotlib", "pyarrow", "pytest"):
        try:
            m = importlib.import_module(mod)
            _ok(True, mod, getattr(m, "__version__", ""))
        except Exception as e:                                       # noqa: BLE001
            good = _ok(False, mod, f"pip install -e '.[model,dev]'  ({e.__class__.__name__})") and good
    for mod, why in (("snowflake.connector", "needed only for FLASHCROWD_STORE=snowflake"),
                     ("google.genai", "needed only for Gemini labels")):
        try:
            importlib.import_module(mod)
            _ok(True, mod)
        except Exception:                                             # noqa: BLE001
            print(f"  {YELLOW}skip{RESET} {mod}{DIM}  {why}{RESET}")

    store = os.environ["FLASHCROWD_STORE"]
    print(f"\n{BOLD}Data store{RESET}  {store}")
    if store == "snowflake":
        missing = [k for k in ("SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER") if not os.environ.get(k)]
        key = os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH")
        if not key and not os.environ.get("SNOWFLAKE_PASSWORD"):
            missing.append("SNOWFLAKE_PRIVATE_KEY_PATH")
        if not _ok(not missing, "Snowflake settings", f"missing {missing} (add them to .env)" if missing else
                   f"{os.environ['SNOWFLAKE_USER']} @ {os.environ['SNOWFLAKE_ACCOUNT']}"):
            return 1
        if key:
            kp = Path(key) if Path(key).is_absolute() else C.ROOT / key
            if not _ok(kp.exists(), "private key file", str(kp) if kp.exists() else f"not found: {kp} (copy PC1's .p8 there)"):
                return 1
    try:
        from .store import open_store
        st = open_store()
        for t in ("EDITS", "PAGEVIEWS_HOURLY", "MAIN_PAGE_EVENTS", "FORECASTS", "REGION_FORECASTS"):
            if st.has_table(t):
                n = int(st.query(f"SELECT COUNT(*) AS n FROM {t}").n.iloc[0])
                _ok(True, t, f"{n:,} rows" + ("  (empty: run `flashcrowd synth` for local data)" if n == 0 and store != "snowflake" else ""))
            else:
                good = _ok(False, t, "table missing") and good
    except Exception as e:                                           # noqa: BLE001
        good = _ok(False, "connect", str(e)[:120]) and good

    print(f"\n{BOLD}Keys{RESET}")
    if os.environ.get("GEMINI_API_KEY"):
        _ok(True, "GEMINI_API_KEY", "set")
    else:
        print(f"  {YELLOW}skip{RESET} GEMINI_API_KEY{DIM}  cause labels off until set{RESET}")
    return 0 if good else 1


def status() -> int:
    reg_path = C.ARTIFACTS / "registry.json"
    print(f"{BOLD}Store{RESET}  {os.environ['FLASHCROWD_STORE']}")
    if not reg_path.exists():
        print(f"{DIM}No training runs yet. Run `flashcrowd train`.{RESET}")
        return 0
    reg = json.loads(reg_path.read_text())
    print(f"{BOLD}Champion{RESET}  {reg.get('champion') or '-'}\n")
    print(f"{'run':<26}{'PR-AUC 2h':>10}{'persist.':>10}  promoted  window")
    for r in reg["runs"][-8:]:
        star = "*" if r["run_id"] == reg.get("champion") else " "
        pa = r.get("pr_auc_h2")
        pp = r.get("persistence_h2")
        print(f"{star}{r['run_id']:<25}{pa if pa is None else f'{pa:.3f}':>10}{pp if pp is None else f'{pp:.3f}':>10}"
              f"  {'yes' if r['promoted'] else 'no ':<8}  {r['window'][0][:16]} -> {r['window'][1][:16]}")
        if not r["promoted"] and r["gate"]["reasons"]:
            print(f"{DIM}   gate: {'; '.join(r['gate']['reasons'])}{RESET}")
    live = C.LIVE_DIR / "forecasts.json"
    if live.exists():
        rows = json.loads(live.read_text())
        print(f"\n{BOLD}Latest live forecast{RESET}  {rows[0]['issued_at'] if rows else '-'}  ({len(rows)} rows)")
        for row in sorted(rows, key=lambda x: -x["p_surge"])[:5]:
            print(f"  {row['p_surge']:.0%}  {row['wiki']:<7} {row['title'][:40]:<40} {row['region']}  +{row['horizon_h']}h")
    bt = C.ARTIFACTS / "backtest.json"
    if bt.exists():
        print(f"\n{BOLD}Backtest{RESET}  {json.loads(bt.read_text())['headline']}")
    return 0


# `check`: readiness of the real warehouse; `setup`: create the shared SURGE_HOURS view.
def warehouse_cmd(cmd: str) -> int:
    from .store import open_store
    from .train import load_config
    from . import warehouse as W
    print(f"{BOLD}{'Readiness' if cmd == 'check' else 'Setup'}{RESET}  {os.environ['FLASHCROWD_STORE']}")
    try:
        st = open_store()
    except Exception as e:                                           # noqa: BLE001
        print(f"  {RED}fail{RESET} connect  {str(e).splitlines()[0][:160]}")
        return 1
    lines = (W.readiness(st, load_config()) + W.write_probe(st)) if cmd == "check" else W.setup(st)
    col = {"ok": GREEN, "warn": YELLOW, "fail": RED}
    for l in lines:
        print(f"  {col[l['status']]}{l['status']:<4}{RESET} {l['check']:<20} {DIM}{l['detail']}{RESET}")
    return 1 if any(l["status"] == "fail" for l in lines) else 0


# The whole PC3 chain on the real data, stopping at the first failure:
# check -> setup -> train (short config if the backfill is short) -> backtest -> export -> one live forecast.
def bootstrap(rest: list[str]) -> int:
    from .store import open_store
    from .train import data_window, load_config
    print(f"{BOLD}Bootstrap{RESET}  {os.environ['FLASHCROWD_STORE']}")
    if warehouse_cmd("check"):
        print(f"{RED}Fix the failures above (usually a grant or a loader) and run bootstrap again.{RESET}")
        return 1
    warehouse_cmd("setup")
    a, b = data_window(open_store())
    days = max(0, (b.normalize() - a.normalize()).days + 1)
    full = load_config()["splits"]
    cfg = "configs/train.json" if days >= full["test_days"] + full["cv_folds"] + full["min_train_days"] else "configs/train_short.json"
    print(f"{DIM}{days} trainable days -> {cfg}{RESET}")
    steps = [("flashcrowd_model.train", ["--config", cfg, *rest]), ("flashcrowd_model.backtest", []),
             ("flashcrowd_model.handoff", []), ("flashcrowd_model.forecast_job", ["live"])]
    for mod, args in steps:
        if run(mod, *args):
            print(f"{RED}stopped at {mod}{RESET}")
            return 1
    print(f"{GREEN}Done.{RESET} Send artifacts/handoff/pc1 to PC1 and artifacts/handoff/pc2 to PC2; "
          "then schedule `flashcrowd live` every 15 minutes.")
    return 0


def set_store(which: str) -> int:
    value = "snowflake" if which == "snowflake" else f"duckdb:{C.LOCAL_DB}"
    lines = ENV_FILE.read_text().splitlines() if ENV_FILE.exists() else []
    lines = [l for l in lines if not l.startswith("FLASHCROWD_STORE=")] + [f"FLASHCROWD_STORE={value}"]
    ENV_FILE.write_text("\n".join(lines) + "\n")
    os.environ["FLASHCROWD_STORE"] = value
    print(f"store -> {value}  (saved in .env)")
    if which == "snowflake" and not os.environ.get("SNOWFLAKE_ACCOUNT"):
        print(f"{YELLOW}Add SNOWFLAKE_ACCOUNT / USER / PASSWORD to .env (copy .env.example).{RESET}")
    return 0


# ------------------------------------------------------------------ commands
def dispatch(argv: list[str]) -> int:
    cmd, rest = (argv[0], argv[1:]) if argv else ("menu", [])
    if cmd == "doctor":
        return doctor()
    if cmd == "status":
        return status()
    if cmd in ("check", "setup"):
        return warehouse_cmd(cmd)
    if cmd == "bootstrap":
        return bootstrap(rest)
    if cmd == "scorecard":
        return run("flashcrowd_model.scorecard", *(["--since", rest[0]] if rest else ["--since", "2000-01-01"]), *rest[1:])
    if cmd == "refine":
        return run("flashcrowd_model.refine", *rest)
    if cmd == "export":
        return run("flashcrowd_model.handoff", *rest)
    if cmd == "store" and rest:
        return set_store(rest[0])
    if cmd == "test":
        return run("pytest", "-q", *rest)
    if cmd == "synth":
        return run("flashcrowd_model.synth", *rest)
    if cmd == "train":
        return run("flashcrowd_model.train", *rest)
    if cmd == "backtest":
        return run("flashcrowd_model.backtest", *rest)
    if cmd == "live":
        return run("flashcrowd_model.forecast_job", "live", *rest)
    if cmd == "batch" and len(rest) >= 2:
        return run("flashcrowd_model.forecast_job", "batch", "--start", rest[0], "--end", rest[1], *rest[2:])
    if cmd == "gemini" and len(rest) >= 2:
        return run("flashcrowd_model.gemini_labeler", "--start", rest[0], "--end", rest[1], *rest[2:])
    if cmd == "sql" and len(rest) >= 2:
        return run("flashcrowd_model.sql_features", "--dialect", "snowflake", "--start", rest[0], "--end", rest[1])
    if cmd == "menu":
        return menu()
    print(__doc__)
    return 2


MENU = [
    ("doctor", "Check setup (packages, credentials, tables)", []),
    ("bootstrap", "Everything on the real data: check, setup, train, backtest, export, first live run", []),
    ("check", "Check the warehouse: data loaded, days to train, write permission", []),
    ("setup", "One-time: create the SURGE_HOURS view for PC1's scorecard", []),
    ("status", "Champion model, recent runs, latest forecast", []),
    ("test", "Run all tests", []),
    ("synth", "Make local synthetic data (offline stand-in for Snowflake)", []),
    ("train", "Systematic training run", []),
    ("refine", "Diagnose the champion model (add --apply at the prompt to retrain with the fixes)", []),
    ("backtest", "Backtest every policy, Pareto chart, boss levels", []),
    ("live", "Live forecast now (or at a given time)", ["Issue time UTC, e.g. 2026-10-24T15:15 (blank = now)"]),
    ("batch", "Forecasts for a past window", ["Start (UTC)", "End (UTC)"]),
    ("gemini", "Gemini cause labels for a window", ["Start (UTC)", "End (UTC)"]),
    ("sql", "Print the Snowflake feature SQL", ["Start (UTC)", "End (UTC)"]),
    ("scorecard", "Score published forecasts (hits, misses, false alarms)", ["Since (UTC), e.g. 2026-10-24T14:00"]),
    ("export", "Write the PC1 level packages and PC2 mocks + charts", []),
    ("store", "Switch data store", ["local or snowflake"]),
]


def menu() -> int:
    while True:
        print(f"\n{BOLD}FLASHCROWD{RESET}  {DIM}store: {os.environ['FLASHCROWD_STORE']}{RESET}")
        for i, (_, label, _) in enumerate(MENU, 1):
            print(f"  {i:>2}  {label}")
        print("   q  Quit")
        try:
            choice = input("\nflashcrowd> ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            return 0
        if choice.lower() in ("q", "quit", "exit"):
            return 0
        if not choice:
            continue
        if not choice.isdigit():                                  # allow typing a raw command, e.g. "train --force"
            dispatch(shlex.split(choice))
            continue
        i = int(choice) - 1
        if not 0 <= i < len(MENU):
            continue
        cmd, _, prompts = MENU[i]
        args = []
        for p in prompts:
            v = input(f"  {p}: ").strip()
            if v:
                args.append(v)
        if cmd == "live" and args:
            args = ["--at", args[0]]
        dispatch([cmd, *args])


def main(argv=None) -> int:
    load_env()
    return dispatch(list(sys.argv[1:] if argv is None else argv))


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile flashcrowd-model/pyproject.toml
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "flashcrowd-model"
version = "0.2.0"
description = "FLASHCROWD (PC3): shared simulator engine + surge forecasting pipeline"
requires-python = ">=3.10"
# The engine (flashcrowd_engine) needs only numpy, so PC1's server stays light:
#   pip install "git+https://github.com/<ORG>/flashcrowd-model.git@engine-v0.3"
dependencies = ["numpy>=1.26"]

[project.optional-dependencies]
# Everything the model pipeline needs (PC3's machine, the forecast cron box).
model = ["pandas>=2.1", "pyarrow>=15", "duckdb>=1.0", "scikit-learn>=1.5", "lightgbm>=4.3", "matplotlib>=3.8",
         "snowflake-connector-python[pandas]>=3.10", "python-dotenv>=1.0"]
gemini = ["google-genai>=1.0"]
dev = ["pytest>=8"]

[project.scripts]
flashcrowd-synth = "flashcrowd_model.synth:main"
flashcrowd-train = "flashcrowd_model.train:main"
flashcrowd-forecast = "flashcrowd_model.forecast_job:main"
flashcrowd-backtest = "flashcrowd_model.backtest:main"
flashcrowd-gemini = "flashcrowd_model.gemini_labeler:main"
flashcrowd-sql = "flashcrowd_model.sql_features:main"
flashcrowd = "flashcrowd_model.cli:main"

[tool.setuptools.packages.find]
where = ["src"]

[tool.setuptools.package-data]
flashcrowd_model = ["sql/*.sql"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
filterwarnings = ["ignore::DeprecationWarning", "ignore::FutureWarning", "ignore::UserWarning"]

# Part B: run the pipeline

### B.1 Install and choose the data source
`snowflake` uses PC1's warehouse through `sf.py`; `local` builds synthetic data in the same table shapes.

In [ ]:
MODE = "snowflake"          # "snowflake" (PC1's real data) or "local" (synthetic, offline)

import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "flashcrowd-model[model,gemini]"], check=False)
ROOT = os.path.abspath("flashcrowd-model")
sys.path.insert(0, os.path.join(ROOT, "src"))
if not os.path.exists(os.path.join(ROOT, ".env")):
    import shutil; shutil.copy(os.path.join(ROOT, ".env.example"), os.path.join(ROOT, ".env"))
from dotenv import load_dotenv; load_dotenv(os.path.join(ROOT, ".env"))
os.environ["FLASHCROWD_STORE"] = "snowflake" if MODE == "snowflake" else f"duckdb:{ROOT}/data/flashcrowd_local.duckdb"
import warnings; warnings.filterwarnings("ignore")
print("store:", os.environ["FLASHCROWD_STORE"])

### B.2 Data: connect (or make the local stand-in)

In [ ]:
from flashcrowd_model import config as C
from flashcrowd_model.store import open_store
if MODE == "local":
    from flashcrowd_model.synth import generate, write
    store = write(generate())
else:
    store = open_store()
print("connected:", type(store).__name__)

### B.3 Readiness check and one-time setup
Any `fail` names what to ask PC1 for (a grant or a loader fix).

In [ ]:
from flashcrowd_model import warehouse as W
from flashcrowd_model.train import load_config
cfg = load_config(os.path.join(ROOT, "configs", "train.json"))
for line in W.readiness(store, cfg) + W.write_probe(store) + W.setup(store):
    print(f"{line['status']:<5} {line['check']:<22} {line['detail']}")

### B.4 Systematic training
Validate, features, leak gate, search, calibrate, fit, evaluate, gate. Uses the short config automatically when the backfill has fewer than 9 usable days.

In [ ]:
from flashcrowd_model.train import data_window, run as train_run
a, b = data_window(store)
days = (b.normalize() - a.normalize()).days + 1
s = cfg["splits"]
if days < s["test_days"] + s["cv_folds"] + s["min_train_days"]:
    cfg = load_config(os.path.join(ROOT, "configs", "train_short.json"))
result = train_run(store, cfg)
result["metrics"]["test"]["h2"]

### B.5 Refinement
Diagnosis report, then a retrain with the recommended config (the gate decides).

In [ ]:
from flashcrowd_model.refine import diagnose
report = diagnose(store)
print("\n".join(report["recommendations"]))
refined = train_run(store, report["refined_config"])
print("refined model promoted:", refined["promoted"])

### B.6 Backtest, handoff files and the first live forecast

In [ ]:
from flashcrowd_model.model import Bundle
from flashcrowd_model import backtest, handoff, forecast_job
bundle = Bundle.load()
bt = backtest.run(store, bundle, write=True)
files = handoff.export(store, bundle)
print(bt["headline"]); print(files)
print(forecast_job.live(store, bundle))

### B.7 Charts and reports

In [ ]:
from IPython.display import Image, Markdown, display
for name in ["reliability", "lead_time", "feature_importance", "surge_example", "pareto"]:
    p = C.CHART_DIR / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p), width=640))
reg = __import__("json").load(open(C.ARTIFACTS / "registry.json"))
ref = C.ARTIFACTS / "runs" / report["champion"] / "refine.md"
display(Markdown(ref.read_text()))

### Done
- Send `flashcrowd-model/artifacts/handoff/pc1/` to PC1 and `.../pc2/` to PC2.
- Schedule `flashcrowd live` every 15 minutes and `flashcrowd train` nightly (see `INTEGRATION.md`).